# GridHeat AI — Extreme-Heat Grid Risk Pipeline (Texas Pilot)

**What this notebook does:** combines public grid-infrastructure, outage-history,
demand, vulnerability, and live-temperature data into a single spatial grid over a
pilot area, scores every grid cell for extreme-heat risk to the electric grid, and
recommends how to spend a limited resilience budget across the highest-risk cells.
A LangGraph agentic workflow ties the steps together end-to-end, and a Gradio
dashboard exposes the result.

**How this notebook is organized (and why):** the original exploration notebook
accumulated ~100 cells across many retries, bug fixes, and a mid-project pivot in
which the pilot city changed from Fresno → Sacramento → back to Fresno. This version
keeps only the final, working path for each step, in the order the pipeline actually
runs, with the reasoning and key findings from the exploration written up as markdown
so the *why* isn't lost even though the dead ends are removed.

**Pipeline at a glance:**
1. Set up the FortyGuard temperature API client
2. Fetch + clean 5 public datasets (transmission lines, outage history, demand,
   battery storage, environmental vulnerability)
3. Pick a pilot Area of Interest (AOI) and build a canonical 1×1 km "heat zone" grid
4. Join every dataset onto that grid
5. Train a temperature → electricity-demand forecasting model
6. Pull live/current heat and score every zone's risk
7. Wrap the pipeline in a LangGraph agent workflow with conditional routing
8. Optimize which risk-reduction actions to fund under a budget
9. Serve it all through a Gradio dashboard

**Re-run safety (caching):** almost every cell that fetches from a network API,
cleans a dataset, joins layers onto the grid, or trains the demand model checks
first whether its output file already exists on disk — if it does, the cell
**loads it instead of redoing the work**. This means you can re-run this whole
notebook top-to-bottom (e.g. "Run All") repeatedly without re-hitting rate-limited
APIs, re-uploading the EIA-860 file, or retraining the model every time. Each of
these cells exposes a `force_refresh = False` flag right where the check happens —
flip it to `True` (or delete the corresponding output file) to force that one step
to redo its work. The one exception is Section 5's live-heat pull, which is cached **per
calendar day** rather than forever, since "live" heat is meant to refresh once a
new day's data becomes available.


## 0. Setup — FortyGuard Temperature API Client

FortyGuard is the source for both **historical** temperature (to train the demand
model) and **live/current** temperature (to score real-time risk). It ships as a
small Python client distributed as a zip (`temperature-api-quickstart`), not a pip
package, so it has to be unzipped into the Colab runtime and added to `sys.path`
before it can be imported.

The API key is read from Colab's **Secrets** manager (the key icon in the left
sidebar) rather than hardcoded, and the smoke test below runs against FortyGuard's
own bundled sample polygon (San Jose) — not the real Fresno AOI — purely to confirm
the client works and to inspect the exact shape of a response (one tile's geometry +
properties) before building any real logic on top of it.


In [ ]:
"""


Option A (no repo link available): upload the zip you already have via the
Colab file browser (left sidebar -> Files -> upload
temperature-api-quickstart-main.zip), then this cell unzips it.

"""

# ---- 1. Get the repo code into the Colab runtime ----
import zipfile
import pathlib

ZIP_PATH = "/content/temperature-api-quickstart-main.zip"  # upload it here first
REPO_PATH = pathlib.Path("/content/temperature-api-quickstart-main")

if not REPO_PATH.exists():
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall("/content")



In [ ]:
# ---- 2. Install dependencies ----
!pip install -q -r {REPO_PATH}/requirements.txt

# ---- 3. Read the API key from Colab Secrets (the key icon in the left sidebar) ----
# Add a secret named FORTYGUARD_API_KEY there first, then toggle "Notebook access" on.
from google.colab import userdata
FORTYGUARD_API_KEY = userdata.get("FORTYGUARD_API_KEY")

# ---- 4. Import the client and run the test ----
import sys
sys.path.insert(0, str(REPO_PATH))

from fortyguard import FortyGuardClient
from fortyguard.samples import SAN_JOSE_POLYGON  # bundled ~40 mi^2 California AOI, within Premium cap

client = FortyGuardClient(api_key=FORTYGUARD_API_KEY)

response = client.create_heatmap(
    polygon_aoi=SAN_JOSE_POLYGON,
    start_date="2025-07-15",   # pick a real past date - a future date will likely return no data
    filter_type=3,             # single full day -> min/max/average per tile
    granularity=100,           # 100m tiles = coarsest/cheapest option, fine for a first test
)

result = response["result"]
stats = result.get("stats_data", {})
tiles = result.get("map_data", {}).get("features", [])

print(f"activity_id : {response['activity_id']}")
print(f"tile count  : {len(tiles)}")
print(f"stats keys  : {list(stats.keys())}")

if tiles:
    first_tile = tiles[0]
    print("\n--- shape of a single tile (this is what we'll grid-join everything else onto) ---")
    print("geometry type:", first_tile["geometry"]["type"])
    print("geometry coords (first ring):", first_tile["geometry"]["coordinates"][0][:2], "...")
    print("properties:", first_tile["properties"])

Submitted -> activity_id=b5609b62-195f-4461-9a16-3aed6fe6abc3
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: completed
Done.
activity_id : b5609b62-195f-4461-9a16-3aed6fe6abc3
tile count  : 10378
stats keys  : ['temperature_stats', 'overall_temperature_distribution', 'normal_temperature_distribution', 'temperature_frequency']

--- shape of a single tile (this is what we'll grid-join everything else onto) ---
geometry type: Polygon
geometry coords (first ring): [[-121.94353678503016, 37.29260397809613], [-121.9424050821698, 37.292593870357514]] ...
properties: {'tile_id': 0, 'average_temperature': 22.0369, 'min_temperature': 14.9232, 'max_temperature': 32.3286}


## 1. Data Acquisition & Cleaning

Five public datasets feed the pipeline. Each is pulled from its source API, then
cleaned down to just the columns the project needs. Two patterns repeat across all
of them and are worth calling out once instead of five times:

- **Pagination via `objectIds`.** Several of these ArcGIS FeatureServers cap a
  single request at ~2,000 rows (`maxRecordCount`). The fix used everywhere below is
  to first request just the object IDs (cheap), then batch-download features
  1,000 IDs at a time via POST. An earlier version of this notebook silently lost
  data by requesting everything in one GET — the giveaway was a suspiciously round
  2,000-row result.
- **CRS handling.** ArcGIS's `f=geojson` output is *always* WGS84 (EPSG:4326) per the
  GeoJSON spec, regardless of the layer's native projection — but `geopandas`'
  `GeoDataFrame.from_features()` doesn't tag a CRS automatically. Every fetch below
  sets it explicitly rather than leaving `gdf.crs` as `None`.


### 1.1 Grid Infrastructure — Texas Electric Substations & Transmission Lines (HIFLD)

In [ ]:
"""
[TX-1.1, CACHED] Fetch + clean Texas Electric Substations AND Transmission
Lines from HIFLD (Homeland Infrastructure Foundation-Level Data).

IMPORTANT DIFFERENCE BETWEEN THE TWO LAYERS:
- Substations have a STATE attribute field -> filter with a WHERE clause.
- Transmission lines do NOT have a STATE field (a single line can cross
  state boundaries), so they must be filtered spatially - by intersecting
  with Texas's geographic extent - not by an attribute equality check.

CACHING: skips both fetches entirely if the cleaned files already exist.
Delete them or set force_refresh=True to re-fetch.
"""
import os
import requests
import geopandas as gpd
import json  # add this to your imports at the top of the cell
SUBSTATIONS_OUTPUT = "tx_substations_clean.geojson"
TRANSMISSION_OUTPUT = "tx_transmission_lines_clean.geojson"
force_refresh = False

# --- VERIFIED: 59 distinct STATE values incl. all US states + territories,
# TX = 4939 rows. ---
SUBSTATIONS_URL = "https://services5.arcgis.com/HDRa0B57OVrv2E1q/arcgis/rest/services/Electric_Substations/FeatureServer/0"

# --- VERIFIED: nationwide extent (AK to HI), but NO state field - see note
# above. Filtered spatially below instead of by attribute. ---
#TRANSMISSION_URL = "https://maps.nccs.nasa.gov/mapping/rest/services/hifld_open/energy/FeatureServer/21"
TRANSMISSION_URL = "https://services2.arcgis.com/FiaPA4ga0iQKduv3/arcgis/rest/services/US_Electric_Power_Transmission_Lines/FeatureServer/0"
FIELD_STATE = "STATE"  # used for substations only
STATE_VALUE = "TX"

# Texas's approximate bounding box (WGS84 lon/lat) - a bbox is a coarse but
# reliable way to spatially filter transmission lines without needing a
# full Texas boundary polygon on hand. It will pull in a thin margin of
# lines just across the border, but that's an acceptable trade-off given
# the pipeline's 1x1 km grid will simply not have any AOI cells out there.
TEXAS_BBOX = {"xmin": -106.65, "ymin": 25.84, "xmax": -93.51, "ymax": 36.50, "spatialReference": {"wkid": 4326}}


def inspect_fields(layer_url: str) -> list:
    """Print the layer's field names so we can confirm FIELD_STATE is right
    (for substations) or confirm there's genuinely no STATE field (for
    transmission lines) before deciding how to filter."""
    meta = requests.get(layer_url, params={"f": "json"}, timeout=60).json()
    fields = [f["name"] for f in meta.get("fields", [])]
    if not fields:
        error = meta.get("error")
        if error:
            raise RuntimeError(
                f"Layer metadata request failed for {layer_url} - "
                f"the URL/layer index itself looks wrong, not just the field "
                f"name. Server said: {error}"
            )
    print(f"  Fields available: {fields}")
    return fields


def fetch_hifld_layer_by_state(layer_url: str, state_value: str, field_state: str) -> gpd.GeoDataFrame:
    """For layers with a STATE attribute (e.g. substations)."""
    fields = inspect_fields(layer_url)
    if field_state not in fields:
        raise RuntimeError(
            f"'{field_state}' not in layer fields above - update FIELD_STATE "
            f"to the correct field name and re-run this cell."
        )

    where_clause = f"{field_state} = '{state_value}'"
    id_resp = requests.get(f"{layer_url}/query",
                            params={"where": where_clause, "returnIdsOnly": "true", "f": "json"},
                            timeout=60)
    id_resp.raise_for_status()
    object_ids = id_resp.json().get("objectIds") or []
    print(f"  Texas object IDs on server: {len(object_ids)}")

    if not object_ids:
        raise RuntimeError(
            f"Query for {field_state}='{state_value}' returned 0 object IDs. "
            f"Either this layer has no Texas rows, or the state values use a "
            f"different format - worth spot-checking a few STATE values."
        )

    return _download_by_object_ids(layer_url, object_ids)




def fetch_hifld_layer_by_bbox(layer_url: str, bbox: dict) -> gpd.GeoDataFrame:
    """For layers with no STATE attribute (e.g. transmission lines) -
    filter spatially by intersecting with a bounding box instead."""
    inspect_fields(layer_url)  # just to confirm the URL/layer itself is valid

    id_resp = requests.get(
        f"{layer_url}/query",
        params={
            "where": "1=1",                     # <-- this server requires a where clause even with a geometry filter
            "geometry": json.dumps(bbox),        # <-- proper JSON, not str(dict)
            "geometryType": "esriGeometryEnvelope",
            "spatialRel": "esriSpatialRelIntersects",
            "inSR": "4326",
            "returnIdsOnly": "true",
            "f": "json",
        },
        timeout=60,
    )
    id_resp.raise_for_status()
    resp_json = id_resp.json()
    if "error" in resp_json:
        raise RuntimeError(f"Spatial query failed: {resp_json['error']}")
    object_ids = resp_json.get("objectIds") or []
    print(f"  Texas-area object IDs on server: {len(object_ids)}")

    if not object_ids:
        raise RuntimeError(
            "Spatial query returned 0 object IDs - double check the bbox "
            "coordinates and that inSR matches the bbox's spatial reference."
        )

    return _download_by_object_ids(layer_url, object_ids)

def _download_by_object_ids(layer_url: str, object_ids: list) -> gpd.GeoDataFrame:
    batch_size = 1000
    all_features = []
    for i in range(0, len(object_ids), batch_size):
        batch_ids = object_ids[i:i + batch_size]
        resp = requests.post(
            f"{layer_url}/query",
            data={"objectIds": ",".join(map(str, batch_ids)), "outFields": "*",
                  "returnGeometry": "true", "f": "geojson"},
            timeout=60,
        )
        resp.raise_for_status()
        all_features.extend(resp.json().get("features", []))
        print(f"  Downloaded {len(all_features)} / {len(object_ids)}")

    gdf = gpd.GeoDataFrame.from_features(all_features)
    gdf = gdf.set_crs("EPSG:4326")
    gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()
    n_invalid = (~gdf.geometry.is_valid).sum()
    if n_invalid > 0:
        gdf["geometry"] = gdf.geometry.buffer(0)
    return gdf


# ---- Substations ----
if os.path.exists(SUBSTATIONS_OUTPUT) and not force_refresh:
    print(f"{SUBSTATIONS_OUTPUT} already exists - loading cached copy.")
    gdf_substations = gpd.read_file(SUBSTATIONS_OUTPUT)
else:
    print("Fetching Texas substations...")
    gdf_substations = fetch_hifld_layer_by_state(SUBSTATIONS_URL, STATE_VALUE, FIELD_STATE)
    print(f"Final substations shape: {gdf_substations.shape}")
    gdf_substations.to_file(SUBSTATIONS_OUTPUT, driver="GeoJSON")
    print(f"Saved to {SUBSTATIONS_OUTPUT}")

# ---- Transmission lines ----
if os.path.exists(TRANSMISSION_OUTPUT) and not force_refresh:
    print(f"{TRANSMISSION_OUTPUT} already exists - loading cached copy.")
    gdf_transmission = gpd.read_file(TRANSMISSION_OUTPUT)
else:
    print("Fetching Texas-area transmission lines...")
    gdf_transmission = fetch_hifld_layer_by_bbox(TRANSMISSION_URL, TEXAS_BBOX)
    print(f"Final transmission lines shape: {gdf_transmission.shape}")
    gdf_transmission.to_file(TRANSMISSION_OUTPUT, driver="GeoJSON")
    print(f"Saved to {TRANSMISSION_OUTPUT}")

Fetching Texas substations...
  Fields available: ['OBJECTID_1', 'OBJECTID', 'ID', 'NAME', 'CITY', 'STATE', 'ZIP', 'TYPE', 'STATUS', 'COUNTY', 'COUNTYFIPS', 'COUNTRY', 'LATITUDE', 'LONGITUDE', 'NAICS_CODE', 'NAICS_DESC', 'SOURCE', 'SOURCEDATE', 'VAL_METHOD', 'VAL_DATE', 'LINES', 'MAX_VOLT', 'MIN_VOLT', 'MAX_INFER', 'MIN_INFER']
  Texas object IDs on server: 4939
  Downloaded 1000 / 4939
  Downloaded 2000 / 4939
  Downloaded 3000 / 4939
  Downloaded 4000 / 4939
  Downloaded 4939 / 4939
Final substations shape: (4939, 26)
Saved to tx_substations_clean.geojson
Fetching Texas-area transmission lines...
  Fields available: ['OBJECTID_1', 'OBJECTID', 'ID', 'TYPE', 'STATUS', 'NAICS_CODE', 'NAICS_DESC', 'SOURCE', 'SOURCEDATE', 'VAL_METHOD', 'VAL_DATE', 'OWNER', 'VOLTAGE', 'VOLT_CLASS', 'INFERRED', 'SUB_1', 'SUB_2', 'Shape__Len', 'Shape__Length']
  Texas-area object IDs on server: 10233
  Downloaded 1000 / 10233
  Downloaded 2000 / 10233
  Downloaded 3000 / 10233
  Downloaded 4000 / 10233
  Dow

### 1.2 Power Outages — DOE/ORNL EAGLE-I (county-level, 15-minute, 2014-2025)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
"""
[TX-1.2, CACHED] Load DOE/ORNL EAGLE-I county-level outage data for Texas.

The EAGLE-I yearly ZIP file is stored directly in Google Drive.
This cell:
1. Mounts Google Drive
2. Reads eaglei_outages_2024.zip
3. Extracts the CSV file(s)
4. Loads and combines them
5. Filters Texas
6. Filters summer months (June-September)
7. Saves a cleaned CSV locally for caching
"""

import os
import glob
import zipfile
import pandas as pd

# ============================================================
# Configuration
# ============================================================

OUTPUT_PATH = "eagle_i_tx_clean.csv"
ZIP_PATH = "/content/drive/MyDrive/eaglei_outages_2024.zip"

force_refresh = False


# ============================================================
# Mount Google Drive
# ============================================================

from google.colab import drive

drive.mount("/content/drive")


# ============================================================
# Use cached cleaned file if available
# ============================================================

if os.path.exists(OUTPUT_PATH) and not force_refresh:

    print(
        f"{OUTPUT_PATH} already exists - loading cached copy "
        f"(set force_refresh=True to re-clean from the ZIP)."
    )

    df_outages = pd.read_csv(
        OUTPUT_PATH,
        parse_dates=["run_start_time"]
    )

    print(f"Loaded shape: {df_outages.shape}")


# ============================================================
# Otherwise extract and clean the ZIP
# ============================================================

else:

    if not os.path.exists(ZIP_PATH):
        raise FileNotFoundError(
            f"ZIP file not found:\n{ZIP_PATH}\n"
            "Make sure eaglei_outages_2024.zip is directly inside My Drive."
        )

    print(f"Found ZIP: {ZIP_PATH}")

    # Temporary extraction directory
    EXTRACT_PATH = "/content/eaglei_outages_2024"

    os.makedirs(EXTRACT_PATH, exist_ok=True)

    # --------------------------------------------------------
    # Extract ZIP
    # --------------------------------------------------------

    print("\nExtracting ZIP...")

    with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(EXTRACT_PATH)

    print("Extraction completed.")


    # --------------------------------------------------------
    # Find CSV files
    # --------------------------------------------------------

    csv_files = glob.glob(
        os.path.join(EXTRACT_PATH, "**", "*.csv"),
        recursive=True
    )

    if not csv_files:
        raise FileNotFoundError(
            "No CSV files were found inside eaglei_outages_2024.zip."
        )

    print("\nCSV files found:")

    for path in csv_files:
        print(" -", path)


    # --------------------------------------------------------
    # Load CSV files
    # --------------------------------------------------------

    frames = []

    for path in csv_files:

        print(f"\nReading: {os.path.basename(path)}")

        df = pd.read_csv(path)

        print(f"Shape: {df.shape}")
        print(f"Columns: {list(df.columns)}")

        frames.append(df)


    # --------------------------------------------------------
    # Combine files
    # --------------------------------------------------------

    df_raw = pd.concat(
        frames,
        ignore_index=True
    )

    print(
        f"\nTotal raw rows across uploaded files: "
        f"{len(df_raw):,}"
    )


    # ========================================================
    # Detect state column
    # ========================================================

    if "state" in df_raw.columns:
        STATE_COL = "state"

    elif "State" in df_raw.columns:
        STATE_COL = "State"

    else:
        raise KeyError(
            "Could not find the state column. "
            f"Available columns: {list(df_raw.columns)}"
        )


    # ========================================================
    # Filter Texas
    # ========================================================

    df_tx = df_raw[
        df_raw[STATE_COL]
        .astype(str)
        .str.strip()
        .str.lower()
        == "texas"
    ].copy()

    print(
        f"Texas rows: {len(df_tx):,}"
    )


    # ========================================================
    # Detect time column
    # ========================================================

    if "run_start_time" in df_tx.columns:
        TIME_COL = "run_start_time"

    elif "run_start_time_utc" in df_tx.columns:
        TIME_COL = "run_start_time_utc"

    else:
        raise KeyError(
            "Could not find run_start_time column. "
            f"Available columns: {list(df_tx.columns)}"
        )


    # ========================================================
    # Convert timestamp
    # ========================================================

    df_tx["run_start_time"] = pd.to_datetime(
        df_tx[TIME_COL],
        errors="coerce"
    )

    # Remove invalid timestamps
    df_tx = df_tx[
        df_tx["run_start_time"].notna()
    ].copy()


    # ========================================================
    # Filter summer months
    # ========================================================

    df_tx["month"] = (
        df_tx["run_start_time"].dt.month
    )

    SUMMER_MONTHS = [6, 7, 8, 9]

    df_outages = df_tx[
        df_tx["month"].isin(SUMMER_MONTHS)
    ].copy()


    # ========================================================
    # Summary
    # ========================================================

    print(
        f"Texas summer-month rows: "
        f"{len(df_outages):,}"
    )

    if "county" in df_outages.columns:

        print(
            f"Counties represented: "
            f"{df_outages['county'].nunique()}"
        )

    else:

        print(
            "WARNING: 'county' column not found."
        )


    # ========================================================
    # Save cleaned dataset
    # ========================================================

    df_outages.to_csv(
        OUTPUT_PATH,
        index=False
    )

    print(
        f"\nSaved cleaned dataset to: "
        f"{OUTPUT_PATH}"
    )

    print(
        f"Final shape: {df_outages.shape}"
    )

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found ZIP: /content/drive/MyDrive/eaglei_outages_2024.zip

Extracting ZIP...
Extraction completed.

CSV files found:
 - /content/eaglei_outages_2024/eaglei_outages_2024.csv

Reading: eaglei_outages_2024.csv
Shape: (27701334, 6)
Columns: ['fips_code', 'county', 'state', 'customers_out', 'run_start_time', 'total_customers']

Total raw rows across uploaded files: 27,701,334
Texas rows: 2,921,200
Texas summer-month rows: 1,036,317
Counties represented: 253

Saved cleaned dataset to: eagle_i_tx_clean.csv
Final shape: (1036317, 7)


### 1.2.1 Historical Validation — Does EAGLE-I Outage Volume Correlate with Summer Heat?

In [ ]:
"""
[VALIDATION] Texas EAGLE-I summer seasonality validation.

Checks whether customers-out volume clusters around the hottest
part of summer, both across Texas and by county.
"""

import pandas as pd

# ============================================================
# Load cleaned EAGLE-I data
# ============================================================

df_outages = pd.read_csv(
    "eagle_i_tx_clean.csv",
    parse_dates=["run_start_time"]
)

print("Columns:")
print(list(df_outages.columns))

# ============================================================
# Detect customers-out column
# ============================================================

if "customers_out" in df_outages.columns:
    CUSTOMERS_COL = "customers_out"

elif "sum" in df_outages.columns:
    CUSTOMERS_COL = "sum"

else:
    raise KeyError(
        "Could not find customers-out column. "
        f"Available columns: {list(df_outages.columns)}"
    )

# ============================================================
# Basic validation
# ============================================================

print(f"\nTotal summer-month county-outage records: {len(df_outages):,}")

print(
    f"Date range: "
    f"{df_outages['run_start_time'].min()} "
    f"to "
    f"{df_outages['run_start_time'].max()}"
)

# ============================================================
# Monthly outage-customer volume
# ============================================================

monthly_impact = (
    df_outages
    .groupby(df_outages["run_start_time"].dt.month_name())[CUSTOMERS_COL]
    .sum()
    .reindex([
        "June",
        "July",
        "August",
        "September"
    ])
)

print(
    "\nCustomers-out (summed across all 15-min readings) "
    "by month, Texas:"
)

print(monthly_impact)

# ============================================================
# Per-county outage totals
# ============================================================

if "county" in df_outages.columns:
    county_col = "county"

elif "county_name" in df_outages.columns:
    county_col = "county_name"

else:
    raise KeyError(
        "Could not find county column. "
        f"Available columns: {list(df_outages.columns)}"
    )

county_outage_totals = (
    df_outages
    .groupby(county_col)[CUSTOMERS_COL]
    .sum()
    .sort_values(ascending=False)
)

print(
    "\nTop 15 Texas counties by cumulative "
    "summer customers-out:"
)

print(county_outage_totals.head(15))

Columns:
['fips_code', 'county', 'state', 'customers_out', 'run_start_time', 'total_customers', 'month']

Total summer-month county-outage records: 1,036,317
Date range: 2024-06-01 00:00:00 to 2024-09-30 23:45:00

Customers-out (summed across all 15-min readings) by month, Texas:
run_start_time
June          43846594
July         852255425
August        14538593
September     10836303
Name: customers_out, dtype: int64

Top 15 Texas counties by cumulative summer customers-out:
county
Harris         560269098
Montgomery      82160093
Fort Bend       55751901
Brazoria        50978624
Galveston       45086307
Liberty          9998937
Smith            9409274
Dallas           8653407
San Jacinto      7275108
Polk             7268745
Walker           5360896
Waller           4163504
Gregg            3931039
Wharton          3721228
Tarrant          3686613
Name: customers_out, dtype: int64


### 1.3 Electricity Demand — ERCOT Actual Load via EIA-930 (system-wide)

**Not `gridstatus`.** `gridstatus`'s live-report method only retains ~14 days of history - it can't reach back to the July-2025 pilot window this notebook needs, so EIA-930 is the sole demand source in this notebook (feeds Section 4's demand model and Section 2's Stage A descriptive context). `gridstatus` was tried as a way to get zone-level demand for pilot selection (Section 2) but was dropped from the plan - not working reliably in this environment - so Section 2 also ranks on EIA-930-only, system-wide terms now (no zone gate).


In [ ]:
"""
[TX-1.3, CACHED] Fetch ERCOT hourly demand via the EIA-930 Open Data API
(replaces the gridstatus/ERCOT MIS approach above, which only retains
~90 days of live report documents and can't reach back to summer 2025 -
see the notebook discussion for the "No documents found" root cause).

IMPORTANT GRANULARITY CHANGE FROM THE ORIGINAL VERSION:
EIA-930 reports demand at the BALANCING-AUTHORITY level only - i.e. ONE
number for ERCOT as a whole (respondent code "ERCO"), NOT broken out by
the 8 ERCOT Weather Zones (Coast, East, Far West, North, North Central,
South, South Central, West). This is a real resolution ceiling in this
data source, not a bug: Section 2's Stage-A "weather-zone gate" step
downstream needs to be adjusted to work off this single system-wide
series instead of ranking 8 zone columns - documented here so it isn't
mistaken for missing code later.

AUTH: needs a free EIA API key from https://www.eia.gov/opendata/register.php
Stored in Colab Secrets as EIA_API_KEY (same pattern as FORTYGUARD_API_KEY
in Section 0 - key icon in the left sidebar, toggle "Notebook access" on).

CACHING: skips the fetch if ercot_demand_clean.csv already exists.
"""
import os
import requests
import pandas as pd

OUTPUT_PATH = "ercot_demand_clean.csv"
force_refresh = False

if os.path.exists(OUTPUT_PATH) and not force_refresh:
    print(f"{OUTPUT_PATH} already exists - loading cached copy "
          f"(set force_refresh=True above to re-fetch).")
    df_demand = pd.read_csv(OUTPUT_PATH, parse_dates=["Time"])
    print(f"Loaded shape: {df_demand.shape}")
else:
    from google.colab import userdata
    EIA_API_KEY = userdata.get("EIA_API_KEY")

    START_DATE = "2025-06-01"   # widened to full summer (Jun-Sep) to match
    END_DATE = "2025-09-30"     # the EAGLE-I summer window used in 1.2.1

    BASE_URL = "https://api.eia.gov/v2/electricity/rto/region-data/data/"

    all_rows = []
    offset = 0
    PAGE_SIZE = 5000  # EIA API max rows per request

    while True:
        params = {
            "api_key": EIA_API_KEY,
            "frequency": "hourly",
            "data[0]": "value",
            "facets[respondent][]": "ERCO",   # ERCOT balancing authority
            "facets[type][]": "D",            # D = Demand
            "start": f"{START_DATE}T00",
            "end": f"{END_DATE}T23",
            "sort[0][column]": "period",
            "sort[0][direction]": "asc",
            "length": PAGE_SIZE,
            "offset": offset,
        }
        resp = requests.get(BASE_URL, params=params, timeout=30)
        resp.raise_for_status()
        page = resp.json()["response"]["data"]
        if not page:
            break
        all_rows.extend(page)
        print(f"  fetched {len(all_rows)} rows so far...")
        if len(page) < PAGE_SIZE:
            break
        offset += PAGE_SIZE

    if not all_rows:
        raise ValueError(
            "No rows returned from EIA API for ERCOT demand in the given "
            "date range - check EIA_API_KEY and the start/end dates."
        )

    df_demand = pd.DataFrame(all_rows)

    # ---- Normalize to match the shape the rest of the pipeline expects ----
    df_demand = df_demand.rename(columns={"period": "Time", "value": "ERCOT_Total"})
    df_demand["Time"] = pd.to_datetime(df_demand["Time"])
    df_demand["ERCOT_Total"] = pd.to_numeric(df_demand["ERCOT_Total"], errors="coerce")
    df_demand = (
        df_demand[["Time", "ERCOT_Total"]]
        .dropna(subset=["ERCOT_Total"])
        .drop_duplicates(subset="Time")
        .sort_values("Time")
        .reset_index(drop=True)
    )

    print(f"Fetched shape: {df_demand.shape}")
    print(f"Date range: {df_demand['Time'].min()} to {df_demand['Time'].max()}")
    print("Columns: ['Time', 'ERCOT_Total'] - single system-wide series, "
          "not per-weather-zone (see markdown note above).")

    df_demand.to_csv(OUTPUT_PATH, index=False)
    print(f"Saved to {OUTPUT_PATH}")

  fetched 2928 rows so far...
Fetched shape: (2928, 2)
Date range: 2025-06-01 00:00:00 to 2025-09-30 23:00:00
Columns: ['Time', 'ERCOT_Total'] - single system-wide series, not per-weather-zone (see markdown note above).
Saved to ercot_demand_clean.csv


### 1.4 Battery Storage (BESS) — EIA-860, Texas

In [ ]:
"""
[TX-1.4, CACHED] Same EIA-860 source and cleaning logic as the California
version - only change is the state filter (State == "TX" instead of
"CA"). Upload the same EIA-860 zip (or the latest annual release) here.

CACHING: skips the upload prompt if battery_storage_tx_clean.geojson
already exists.
"""
import os
import pandas as pd
import geopandas as gpd

OUTPUT_PATH = "battery_storage_tx_clean.geojson"
force_refresh = False

if os.path.exists(OUTPUT_PATH) and not force_refresh:
    print(f"{OUTPUT_PATH} already exists - loading cached copy "
          f"(set force_refresh=True above to re-upload + re-clean).")
    gdf_battery = gpd.read_file(OUTPUT_PATH)
    print(f"Loaded shape: {gdf_battery.shape}")
else:
    from google.colab import files
    import zipfile

    uploaded = files.upload()
    zip_file = list(uploaded.keys())[0]
    extract_path = "/content/eia8602024"
    with zipfile.ZipFile(zip_file, "r") as zip_ref:
        zip_ref.extractall(extract_path)
    print("Extracted files:", os.listdir(extract_path))

    STORAGE_PATH = "/content/eia8602024/3_4_Energy_Storage_Y2024.xlsx"
    PLANT_PATH = "/content/eia8602024/2___Plant_Y2024.xlsx"

    storage_df = pd.read_excel(STORAGE_PATH, sheet_name=0, header=1)
    plant_df = pd.read_excel(PLANT_PATH, sheet_name=0, header=1)

    storage_tx = storage_df[storage_df["State"] == "TX"].copy()
    storage_tx = storage_tx[storage_tx["Status"] == "OP"]  # Operating only

    plant_coords = plant_df[["Plant Code", "Latitude", "Longitude"]].drop_duplicates("Plant Code")
    storage_tx = storage_tx.merge(plant_coords, on="Plant Code", how="left")
    storage_tx = storage_tx.dropna(subset=["Latitude", "Longitude"])

    KEEP_COLUMNS = ["Utility Name", "Plant Code", "Plant Name", "County", "Generator ID",
                    "Technology", "Nameplate Capacity (MW)", "Nameplate Energy Capacity (MWh)",
                    "Maximum Charge Rate (MW)", "Maximum Discharge Rate (MW)",
                    "Operating Year", "Latitude", "Longitude"]
    storage_clean = storage_tx[KEEP_COLUMNS].copy()

    gdf_battery = gpd.GeoDataFrame(
        storage_clean,
        geometry=gpd.points_from_xy(storage_clean["Longitude"], storage_clean["Latitude"]),
        crs="EPSG:4326",
    )
    print(f"Final shape: {gdf_battery.shape}")
    print(f"Total TX energy capacity: {gdf_battery['Nameplate Energy Capacity (MWh)'].sum():.0f} MWh")
    gdf_battery.to_file(OUTPUT_PATH, driver="GeoJSON")
    print(f"Saved to {OUTPUT_PATH}")


Saving eia8602024.zip to eia8602024.zip
Extracted files: ['3_2_Wind_Y2024.xlsx', 'EIA-860 Form.xlsx', 'LayoutY2024.xlsx', '6_2_EnviroEquip_Y2024.xlsx', '3_3_Solar_Y2024.xlsx', 'EIA-860 instructions.pdf', '2___Plant_Y2024.xlsx', '3_1_Generator_Y2024.xlsx', '4___Owner_Y2024.xlsx', '6_1_EnviroAssoc_Y2024.xlsx', '1___Utility_Y2024.xlsx', '3_4_Energy_Storage_Y2024.xlsx', '3_5_Multifuel_Y2024.xlsx']
Final shape: (133, 14)
Total TX energy capacity: 11498 MWh
Saved to battery_storage_tx_clean.geojson


### 1.5 EPA EJScreen — Environmental & Social Vulnerability, Texas (block-group level)

In [ ]:
"""
[TX-1.5, CACHED] Load EPA EJScreen for Texas, cleaned down to the same
kind of heat-risk-relevant columns kept for CalEnviroScreen (physiological
vulnerability - elderly, children under 10, existing respiratory/
cardiovascular burden - not socioeconomic/access indicators).

IMPORTANT: EPA discontinued public access to the live EJScreen tool and
its ArcGIS FeatureServer on 2025-02-05. There is no live query API - the
only path is EPA's archived bulk files, uploaded here as a zip.

FIXES IN THIS VERSION (over the original):
1. The uploaded file (EJScreen_2024_Tract_with_AS_CNMI_GU_VI.csv.zip) is a
   ZIP containing a CSV, not a bare .csv or a shapefile/GDB - the original
   code's `raw_path.endswith(".csv")` check missed this, so it fell into
   the gpd.read_file() shapefile branch and silently returned 0 rows. This
   version extracts the zip and reads the CSV inside with pandas.
2. This particular release is TRACT-level (the filename says so), and like
   all EJScreen bulk CSVs it is ATTRIBUTE-ONLY - no geometry column. The
   original code's `gpd.GeoDataFrame(df_tx)` on a geometry-less DataFrame
   silently stays a plain DataFrame, which is why `.to_file()` failed with
   "'DataFrame' object has no attribute 'to_file'" downstream. This
   version explicitly downloads the matching Census TIGER cartographic
   boundary shapefile for Texas tracts and joins geometry onto the EJScreen
   attributes on the tract GEOID.
3. Column names are printed BEFORE any KEEP_COLUMNS assumption, since the
   2024 tract-level release may use different field names than older
   block-group releases (verify against the printed list on first run).

CACHING: skips the upload/clean step if ejscreen_tx_clean.geojson already
exists.
"""
import os
import glob
import zipfile

import pandas as pd
import geopandas as gpd
import requests

OUTPUT_PATH = "ejscreen_tx_clean.geojson"
force_refresh = False

if os.path.exists(OUTPUT_PATH) and not force_refresh:
    print(f"{OUTPUT_PATH} already exists - loading cached copy "
          f"(set force_refresh=True above to re-upload + re-clean).")
    gdf_clean = gpd.read_file(OUTPUT_PATH)
    print(f"Loaded shape: {gdf_clean.shape}")
else:
    from google.colab import files

    print("Upload the EJScreen archived release (zip containing a CSV, "
          "already downloaded from EPA's archive).")
    uploaded = files.upload()
    raw_path = list(uploaded.keys())[0]

    # ---- Step 1: unzip if needed, find the actual CSV inside ----
    if raw_path.lower().endswith(".zip"):
        extract_dir = "/content/ejscreen_extracted"
        os.makedirs(extract_dir, exist_ok=True)
        with zipfile.ZipFile(raw_path, "r") as zf:
            zf.extractall(extract_dir)
        csv_candidates = glob.glob(os.path.join(extract_dir, "**", "*.csv"), recursive=True)
        if not csv_candidates:
            raise FileNotFoundError(f"No CSV found inside {raw_path} - check the zip contents.")
        csv_path = csv_candidates[0]
        print(f"Extracted CSV: {csv_path}")
    else:
        csv_path = raw_path

    # ---- Step 2: read the CSV, show real columns before assuming any ----
    df = pd.read_csv(csv_path, low_memory=False)
    print(f"Raw shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")

    # EJScreen's tract/block-group ID field name has varied across
    # releases - "ID" and "FIPS" are the two most common; GEOID is an
    # 11-digit tract code (2-digit state + 3-digit county + 6-digit tract).
    ID_COL = next((c for c in ("ID", "FIPS", "GEOID") if c in df.columns), None)
    if ID_COL is None:
        raise KeyError(
            f"Couldn't find a tract/block-group ID column among {list(df.columns)} - "
            f"update ID_COL manually by inspecting the printed column list above."
        )

    df[ID_COL] = df[ID_COL].astype(str).str.zfill(11)  # tract GEOID is 11 digits, state=first 2
    df_tx = df[df[ID_COL].str.startswith("48")].copy()  # Texas FIPS = 48
    print(f"Texas rows (attribute-only, no geometry yet): {len(df_tx)}")

    if df_tx.empty:
        raise ValueError(
            f"0 Texas rows found filtering {ID_COL} by prefix '48' - inspect a few raw "
            f"{ID_COL} values (df[ID_COL].head()) to confirm the ID format matches "
            f"an 11-digit tract GEOID starting with the state FIPS code."
        )

    # ---- Step 3: this file has NO geometry - download the matching Census
    # TIGER cartographic boundary shapefile for Texas tracts and join it on
    # the GEOID, instead of assuming geometry already exists. ----
    tiger_dir = "/content/tiger_tx_tract"
    os.makedirs(tiger_dir, exist_ok=True)
    tiger_shp = None
    for year in (2024, 2023, 2022):  # fall back a year or two if the latest isn't published yet
        url = f"https://www2.census.gov/geo/tiger/GENZ{year}/shp/cb_{year}_48_tract_500k.zip"
        try:
            resp = requests.get(url, timeout=60)
            resp.raise_for_status()
        except requests.exceptions.RequestException as e:
            print(f"  {year} boundary file not available ({e}), trying an earlier year...")
            continue
        zip_path = f"{tiger_dir}/cb_{year}_48_tract_500k.zip"
        with open(zip_path, "wb") as f:
            f.write(resp.content)
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(tiger_dir)
        tiger_shp = glob.glob(f"{tiger_dir}/*.shp")[0]
        print(f"Using TIGER {year} Texas tract boundaries: {tiger_shp}")
        break

    if tiger_shp is None:
        raise RuntimeError(
            "Could not download any TIGER tract boundary file for Texas (tried "
            "2024/2023/2022) - check network access to www2.census.gov."
        )

    tracts_gdf = gpd.read_file(tiger_shp)[["GEOID", "geometry"]]
    tracts_gdf["GEOID"] = tracts_gdf["GEOID"].astype(str).str.zfill(11)

    gdf = tracts_gdf.merge(df_tx, left_on="GEOID", right_on=ID_COL, how="inner")
    gdf = gpd.GeoDataFrame(gdf, geometry="geometry", crs=tracts_gdf.crs)
    print(f"Matched to geometry: {len(gdf)} / {len(df_tx)} Texas rows")

    # ---- Step 4: keep only heat-risk-relevant columns that actually exist
    # in this release (verify against the printed column list above if any
    # of these are silently dropped). ----
    KEEP_COLUMNS_CANDIDATES = [
        ID_COL, "STATE_NAME", "CNTY_NAME", "ACSTOTPOP", "P_EJINDEX",
        "PEOPCOLORPCT", "OVER64PCT", "UNDER5PCT", "geometry",
    ]
    present = [c for c in KEEP_COLUMNS_CANDIDATES if c in gdf.columns]
    missing = [c for c in KEEP_COLUMNS_CANDIDATES if c not in gdf.columns and c != "geometry"]
    if missing:
        print(f"NOTE: these expected columns aren't in this release: {missing} - "
              f"check the raw column list above for renamed equivalents.")
    gdf_clean = gdf[present].copy()
    gdf_clean = gdf_clean[gdf_clean.geometry.notna() & ~gdf_clean.geometry.is_empty]

    print(f"Final shape: {gdf_clean.shape}")
    gdf_clean.to_file(OUTPUT_PATH, driver="GeoJSON")
    print(f"Saved to {OUTPUT_PATH}")

Upload the EJScreen archived release (zip containing a CSV, already downloaded from EPA's archive).


Saving EJScreen_2024_Tract_with_AS_CNMI_GU_VI.csv.zip to EJScreen_2024_Tract_with_AS_CNMI_GU_VI.csv (1).zip
Extracted CSV: /content/ejscreen_extracted/EJScreen_2024_Tract_with_AS_CNMI_GU_VI.csv
Raw shape: (86082, 230)
Columns: ['OID_', 'ID', 'STATE_NAME', 'ST_ABBREV', 'CNTY_NAME', 'REGION', 'ACSTOTPOP', 'ACSIPOVBAS', 'ACSEDUCBAS', 'ACSTOTHH', 'ACSTOTHU', 'ACSUNEMPBAS', 'ACSDISABBAS', 'DEMOGIDX_2', 'DEMOGIDX_5', 'PEOPCOLOR', 'PEOPCOLORPCT', 'LOWINCOME', 'LOWINCPCT', 'UNEMPLOYED', 'UNEMPPCT', 'DISABILITY', 'DISABILITYPCT', 'LINGISO', 'LINGISOPCT', 'LESSHS', 'LESSHSPCT', 'UNDER5', 'UNDER5PCT', 'OVER64', 'OVER64PCT', 'LIFEEXPPCT', 'PM25', 'OZONE', 'DSLPM', 'RSEI_AIR', 'PTRAF', 'PRE1960', 'PRE1960PCT', 'PNPL', 'PRMP', 'PTSDF', 'UST', 'PWDIS', 'NO2', 'DWATER', 'D2_PM25', 'D5_PM25', 'D2_OZONE', 'D5_OZONE', 'D2_DSLPM', 'D5_DSLPM', 'D2_RSEI_AIR', 'D5_RSEI_AIR', 'D2_PTRAF', 'D5_PTRAF', 'D2_LDPNT', 'D5_LDPNT', 'D2_PNPL', 'D5_PNPL', 'D2_PRMP', 'D5_PRMP', 'D2_PTSDF', 'D5_PTSDF', 'D2_UST', 'D5_UST',

In [ ]:
"""
[TX-1.5, CACHED] Load EPA EJScreen for Texas, cleaned down to the same
kind of heat-risk-relevant columns kept for CalEnviroScreen (physiological
vulnerability - elderly, children under 10, existing respiratory/
cardiovascular burden - not socioeconomic/access indicators).

IMPORTANT: EPA discontinued public access to the live EJScreen tool and
its ArcGIS FeatureServer on 2025-02-05. There is no live query API - the
only path is EPA's archived bulk files, uploaded here as a zip.

FIXES IN THIS VERSION (over the original):
1. The uploaded file (EJScreen_2024_Tract_with_AS_CNMI_GU_VI.csv.zip) is a
   ZIP containing a CSV, not a bare .csv or a shapefile/GDB - the original
   code's `raw_path.endswith(".csv")` check missed this, so it fell into
   the gpd.read_file() shapefile branch and silently returned 0 rows. This
   version extracts the zip and reads the CSV inside with pandas.
2. This particular release is TRACT-level (the filename says so), and like
   all EJScreen bulk CSVs it is ATTRIBUTE-ONLY - no geometry column. The
   original code's `gpd.GeoDataFrame(df_tx)` on a geometry-less DataFrame
   silently stays a plain DataFrame, which is why `.to_file()` failed with
   "'DataFrame' object has no attribute 'to_file'" downstream. This
   version explicitly downloads the matching Census TIGER cartographic
   boundary shapefile for Texas tracts and joins geometry onto the EJScreen
   attributes on the tract GEOID.
3. Column names are printed BEFORE any KEEP_COLUMNS assumption, since the
   2024 tract-level release may use different field names than older
   block-group releases (verify against the printed list on first run).
4. VULNERABILITY COLUMN FIX: P_EJINDEX doesn't exist in the EJScreen 2024
   release (confirmed via EJScreen_2024_Tract_Percentiles_Columns.xlsx -
   no EJINDEX column at all in this release). Using P_DEMOGIDX_5
   (Percentile for Supplemental Demographic Index) instead - closer fit
   for "power outages hit vulnerable people harder" since it folds in
   unemployment, less-than-HS education, and disability (disability
   matters directly here - medical equipment dependent on power).
5. RE-USE ALREADY-EXTRACTED CSV: if the zip was already uploaded and
   extracted in this runtime (/content/ejscreen_extracted has a CSV in
   it), skip the upload prompt entirely and read that CSV directly -
   avoids re-uploading + re-extracting an already-available file.

CACHING: skips the upload/clean step if ejscreen_tx_clean.geojson already
exists.
"""
import os
import glob
import zipfile

import pandas as pd
import geopandas as gpd
import requests

OUTPUT_PATH = "ejscreen_tx_clean.geojson"
force_refresh = True  # forced True this run - old cached file lacks the vulnerability column

EXTRACT_DIR = "/content/ejscreen_extracted"

if os.path.exists(OUTPUT_PATH) and not force_refresh:
    print(f"{OUTPUT_PATH} already exists - loading cached copy "
          f"(set force_refresh=True above to re-upload + re-clean).")
    gdf_clean = gpd.read_file(OUTPUT_PATH)
    print(f"Loaded shape: {gdf_clean.shape}")
else:
    # ---- Step 1: reuse an already-extracted CSV if one is sitting in
    # EXTRACT_DIR from a previous run in this runtime - skip the upload
    # prompt entirely in that case. ----
    existing_csvs = glob.glob(os.path.join(EXTRACT_DIR, "**", "*.csv"), recursive=True)

    if existing_csvs:
        csv_path = existing_csvs[0]
        print(f"Found already-extracted CSV, reusing it (no re-upload): {csv_path}")
    else:
        from google.colab import files

        print("Upload the EJScreen archived release (zip containing a CSV, "
              "already downloaded from EPA's archive).")
        uploaded = files.upload()
        raw_path = list(uploaded.keys())[0]

        if raw_path.lower().endswith(".zip"):
            os.makedirs(EXTRACT_DIR, exist_ok=True)
            with zipfile.ZipFile(raw_path, "r") as zf:
                zf.extractall(EXTRACT_DIR)
            csv_candidates = glob.glob(os.path.join(EXTRACT_DIR, "**", "*.csv"), recursive=True)
            if not csv_candidates:
                raise FileNotFoundError(f"No CSV found inside {raw_path} - check the zip contents.")
            csv_path = csv_candidates[0]
            print(f"Extracted CSV: {csv_path}")
        else:
            csv_path = raw_path

    # ---- Step 2: read the CSV, show real columns before assuming any ----
    df = pd.read_csv(csv_path, low_memory=False)
    print(f"Raw shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")

    # EJScreen's tract/block-group ID field name has varied across
    # releases - "ID" and "FIPS" are the two most common; GEOID is an
    # 11-digit tract code (2-digit state + 3-digit county + 6-digit tract).
    ID_COL = next((c for c in ("ID", "FIPS", "GEOID") if c in df.columns), None)
    if ID_COL is None:
        raise KeyError(
            f"Couldn't find a tract/block-group ID column among {list(df.columns)} - "
            f"update ID_COL manually by inspecting the printed column list above."
        )

    df[ID_COL] = df[ID_COL].astype(str).str.zfill(11)  # tract GEOID is 11 digits, state=first 2
    df_tx = df[df[ID_COL].str.startswith("48")].copy()  # Texas FIPS = 48
    print(f"Texas rows (attribute-only, no geometry yet): {len(df_tx)}")

    if df_tx.empty:
        raise ValueError(
            f"0 Texas rows found filtering {ID_COL} by prefix '48' - inspect a few raw "
            f"{ID_COL} values (df[ID_COL].head()) to confirm the ID format matches "
            f"an 11-digit tract GEOID starting with the state FIPS code."
        )

    # ---- Step 3: this file has NO geometry - download the matching Census
    # TIGER cartographic boundary shapefile for Texas tracts and join it on
    # the GEOID, instead of assuming geometry already exists. ----
    tiger_dir = "/content/tiger_tx_tract"
    os.makedirs(tiger_dir, exist_ok=True)
    tiger_shp = None
    for year in (2024, 2023, 2022):  # fall back a year or two if the latest isn't published yet
        url = f"https://www2.census.gov/geo/tiger/GENZ{year}/shp/cb_{year}_48_tract_500k.zip"
        try:
            resp = requests.get(url, timeout=60)
            resp.raise_for_status()
        except requests.exceptions.RequestException as e:
            print(f"  {year} boundary file not available ({e}), trying an earlier year...")
            continue
        zip_path = f"{tiger_dir}/cb_{year}_48_tract_500k.zip"
        with open(zip_path, "wb") as f:
            f.write(resp.content)
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(tiger_dir)
        tiger_shp = glob.glob(f"{tiger_dir}/*.shp")[0]
        print(f"Using TIGER {year} Texas tract boundaries: {tiger_shp}")
        break

    if tiger_shp is None:
        raise RuntimeError(
            "Could not download any TIGER tract boundary file for Texas (tried "
            "2024/2023/2022) - check network access to www2.census.gov."
        )

    tracts_gdf = gpd.read_file(tiger_shp)[["GEOID", "geometry"]]
    tracts_gdf["GEOID"] = tracts_gdf["GEOID"].astype(str).str.zfill(11)

    gdf = tracts_gdf.merge(df_tx, left_on="GEOID", right_on=ID_COL, how="inner")
    gdf = gpd.GeoDataFrame(gdf, geometry="geometry", crs=tracts_gdf.crs)
    print(f"Matched to geometry: {len(gdf)} / {len(df_tx)} Texas rows")

    # ---- Step 4: keep only heat-risk-relevant columns that actually exist
    # in this release (verify against the printed column list above if any
    # of these are silently dropped). ----
    KEEP_COLUMNS_CANDIDATES = [
        ID_COL, "STATE_NAME", "CNTY_NAME", "ACSTOTPOP", "P_DEMOGIDX_5",
        "PEOPCOLORPCT", "OVER64PCT", "UNDER5PCT", "geometry",
    ]
    present = [c for c in KEEP_COLUMNS_CANDIDATES if c in gdf.columns]
    missing = [c for c in KEEP_COLUMNS_CANDIDATES if c not in gdf.columns and c != "geometry"]
    if missing:
        print(f"NOTE: these expected columns aren't in this release: {missing} - "
              f"check the raw column list above for renamed equivalents.")
    gdf_clean = gdf[present].copy()
    gdf_clean = gdf_clean[gdf_clean.geometry.notna() & ~gdf_clean.geometry.is_empty]

    print(f"Final shape: {gdf_clean.shape}")
    gdf_clean.to_file(OUTPUT_PATH, driver="GeoJSON")
    print(f"Saved to {OUTPUT_PATH}")

Found already-extracted CSV, reusing it (no re-upload): /content/ejscreen_extracted/EJScreen_2024_Tract_with_AS_CNMI_GU_VI.csv
Raw shape: (86082, 230)
Columns: ['OID_', 'ID', 'STATE_NAME', 'ST_ABBREV', 'CNTY_NAME', 'REGION', 'ACSTOTPOP', 'ACSIPOVBAS', 'ACSEDUCBAS', 'ACSTOTHH', 'ACSTOTHU', 'ACSUNEMPBAS', 'ACSDISABBAS', 'DEMOGIDX_2', 'DEMOGIDX_5', 'PEOPCOLOR', 'PEOPCOLORPCT', 'LOWINCOME', 'LOWINCPCT', 'UNEMPLOYED', 'UNEMPPCT', 'DISABILITY', 'DISABILITYPCT', 'LINGISO', 'LINGISOPCT', 'LESSHS', 'LESSHSPCT', 'UNDER5', 'UNDER5PCT', 'OVER64', 'OVER64PCT', 'LIFEEXPPCT', 'PM25', 'OZONE', 'DSLPM', 'RSEI_AIR', 'PTRAF', 'PRE1960', 'PRE1960PCT', 'PNPL', 'PRMP', 'PTSDF', 'UST', 'PWDIS', 'NO2', 'DWATER', 'D2_PM25', 'D5_PM25', 'D2_OZONE', 'D5_OZONE', 'D2_DSLPM', 'D5_DSLPM', 'D2_RSEI_AIR', 'D5_RSEI_AIR', 'D2_PTRAF', 'D5_PTRAF', 'D2_LDPNT', 'D5_LDPNT', 'D2_PNPL', 'D5_PNPL', 'D2_PRMP', 'D5_PRMP', 'D2_PTSDF', 'D5_PTSDF', 'D2_UST', 'D5_UST', 'D2_PWDIS', 'D5_PWDIS', 'D2_NO2', 'D5_NO2', 'D2_DWATER', 'D5_DWATE

## 2. Pilot Area Selection — Texas County Screening

**Methodology (agreed after the California/Fresno experience, adapted for
the granularity mismatch between our four criteria — UPDATED after the
Texas EIA-930 switch):**

The four selection criteria live at different geographic resolutions -
heat can be near-point, vulnerability is block-group, outages are county.
Demand was originally meant to gate candidates at ERCOT's 8 Weather-Zone
level, but that data source (ERCOT MIS live reports) doesn't retain
documents far enough back to cover the summer 2025 pilot window (see
Section 1.3 discussion). The replacement source, EIA-930, only reports
demand at the **whole-ERCOT-system level** - one number, not 8 zones -
so a weather-zone *gate* is no longer possible with real data.

**Revised two-stage funnel (Stage A is now descriptive, not a filter):**

- **Stage A - System-wide demand context (descriptive only):** report
  ERCOT's overall summer 2025 peak/average demand so the pilot-selection
  writeup still has a real, documented demand figure behind it - but it
  no longer eliminates any candidate county, since there's no zone-level
  signal left to gate on.
- **Stage B - County ranking (all candidates now included):** rank ALL
  manually-curated candidate counties (the same major-metro list used
  before) using FortyGuard heat + EAGLE-I cumulative summer outage
  volume + EJScreen population-weighted vulnerability. Demand no longer
  contributes a per-county signal here either, for the same reason.
- **Stage C - Precise AOI:** unchanged - build the exact FortyGuard AOI
  for the winning county from Stage B.

**Documented consequence:** the pilot-county choice below is driven by
heat + outages + vulnerability only, not demand-informed. This is stated
explicitly rather than left implicit, since it's a real reduction in what
the original methodology intended.


In [ ]:
"""
[TX-2, STAGE A - REVISED, DESCRIPTIVE ONLY] ERCOT system-wide demand
context, using ercot_demand_clean.csv from Section 1.3 (EIA-930).

CHANGE FROM ORIGINAL: the original Stage A ranked 8 ERCOT weather zones
by peak load and used the top N zones as a GATE to filter candidate
counties. That's no longer possible - EIA-930 only gives a single
ERCOT-wide series (ERCOT_Total), not per-zone columns. This cell now
just reports the system-wide peak/average as documented context for the
writeup; it does NOT filter or eliminate any candidate county. All
candidates proceed to Stage B.
"""
import pandas as pd

df_demand = pd.read_csv("ercot_demand_clean.csv", parse_dates=["Time"])

demand_col = "ERCOT_Total" if "ERCOT_Total" in df_demand.columns else \
    [c for c in df_demand.columns if c not in ("Time", "Time_UTC")][0]

system_peak_mw = df_demand[demand_col].max()
system_mean_mw = df_demand[demand_col].mean()
peak_time = df_demand.loc[df_demand[demand_col].idxmax(), "Time"]

print("ERCOT system-wide demand (EIA-930, descriptive context only - not a gate):")
print(f"  Peak demand : {system_peak_mw:,.0f} MW at {peak_time}")
print(f"  Mean demand : {system_mean_mw:,.0f} MW")
print(f"  Sampled window: {df_demand['Time'].min()} to {df_demand['Time'].max()}")
print("\nNo weather-zone gate applied (EIA-930 has no zone breakdown) - "
      "all candidate counties proceed to Stage B unfiltered.")


ERCOT system-wide demand (EIA-930, descriptive context only - not a gate):
  Peak demand : 83,597 MW at 2025-08-18 23:00:00
  Mean demand : 63,659 MW
  Sampled window: 2025-06-01 00:00:00 to 2025-09-30 23:00:00

No weather-zone gate applied (EIA-930 has no zone breakdown) - all candidate counties proceed to Stage B unfiltered.


In [ ]:
"""
[TX-2, STAGE A cont. - REVISED v2] Candidate county list.

CHANGE FROM PREVIOUS VERSION: the original candidate list was hand-picked
by eye (major metros, one per weather zone) - it did NOT include any of
the actual EAGLE-I outage numbers from Section 1.2.1. Cross-checking that
list against the real top-15 counties by cumulative summer customers-out
found 9 counties missing entirely, including Montgomery (#2 statewide,
82M) and Brazoria (#4 statewide, 51M) - both bigger than several counties
that WERE on the hand-picked list. Since Stage B's ranking only ever sees
candidate_counties, those counties could never win the pilot regardless
of their real outage exposure.

FIX: the candidate list is now the UNION of the original hand-picked
metros (kept for weather-zone geographic spread / representativeness)
and the real top-15 outage counties from county_outage_totals (Section
1.2.1) - so no county with genuinely high documented grid stress is
excluded by construction.
"""
COUNTY_TO_WEATHER_ZONE_APPROX = {
    "Harris": "Coast",
    "Galveston": "Coast",
    "Fort Bend": "Coast",
    "Dallas": "North Central",
    "Tarrant": "North Central",
    "Collin": "North Central",
    "Travis": "South Central",
    "Bexar": "South Central",
    "Hidalgo": "South",
    "Cameron": "South",
    "El Paso": "Far West",
    "Midland": "Far West",
    "Ector": "Far West",
    "Lubbock": "North",
    "Potter": "North",
    "Nueces": "Coast",
    "Jefferson": "East",
    "Smith": "East",
}

# Real top-15 counties by cumulative summer customers-out (Section 1.2.1) -
# added regardless of weather-zone label, since that mapping is
# approximate/by-eye and shouldn't gate out documented outage exposure.
top_outage_counties = list(county_outage_totals.head(15).index)

candidate_counties = sorted(set(COUNTY_TO_WEATHER_ZONE_APPROX.keys()) | set(top_outage_counties))

added_from_outages = sorted(set(top_outage_counties) - set(COUNTY_TO_WEATHER_ZONE_APPROX.keys()))
print(f"Counties added from real outage data (not in the original hand-picked list): {added_from_outages}")
print(f"\nCandidate counties for Stage B ({len(candidate_counties)} total, hand-picked ∪ top-outage):")
print(candidate_counties)

Counties added from real outage data (not in the original hand-picked list): ['Brazoria', 'Gregg', 'Liberty', 'Montgomery', 'Polk', 'San Jacinto', 'Walker', 'Waller', 'Wharton']

Candidate counties for Stage B (27 total, hand-picked ∪ top-outage):
['Bexar', 'Brazoria', 'Cameron', 'Collin', 'Dallas', 'Ector', 'El Paso', 'Fort Bend', 'Galveston', 'Gregg', 'Harris', 'Hidalgo', 'Jefferson', 'Liberty', 'Lubbock', 'Midland', 'Montgomery', 'Nueces', 'Polk', 'Potter', 'San Jacinto', 'Smith', 'Tarrant', 'Travis', 'Walker', 'Waller', 'Wharton']


In [ ]:
"""
[TX-2, STAGE B] Rank candidate counties on heat + outages + vulnerability.

REVERTED: a zone-gate step (via gridstatus) briefly restricted this to
only counties inside the highest-demand weather zone. gridstatus was
dropped from the plan (not working reliably in this environment), so this
is back to ranking ALL candidate counties - no demand-based gate, same as
the "REVISED" EIA-930-only methodology in the Section 2 markdown above.

Heat: one cheap FortyGuard `environmental_parameters` call per county
centroid (point-based, Basic tier, NOT the polygon heatmap) - see section
0 setup for the client. Uses apparent_temperature_celsius at the hot
afternoon hour, not heat_index (which peaks at an unrealistic 2am per the
FortyGuard docs).
"""
import pandas as pd
import geopandas as gpd

# County centroids - county-seat coordinates (lat, lon) for each candidate
# in COUNTY_TO_WEATHER_ZONE_APPROX above. Approximate, not
# population-weighted centroids - good enough for a single representative
# environmental_parameters() point call per county, not for anything
# requiring sub-county precision.
COUNTY_CENTROIDS = {
    "Harris": (29.7604, -95.3698),        # Houston
    "Galveston": (29.3013, -94.7977),     # Galveston
    "Fort Bend": (29.5694, -95.7676),     # Richmond
    "Dallas": (32.7767, -96.7970),        # Dallas
    "Tarrant": (32.7555, -97.3308),       # Fort Worth
    "Collin": (33.1795, -96.4930),        # McKinney
    "Travis": (30.2672, -97.7431),        # Austin
    "Bexar": (29.4241, -98.4936),         # San Antonio
    "Hidalgo": (26.1004, -98.2630),       # Edinburg
    "Cameron": (25.9140, -97.4891),       # Brownsville
    "El Paso": (31.7619, -106.4850),      # El Paso
    "Midland": (31.9973, -102.0779),      # Midland
    "Ector": (31.8673, -102.3676),        # Odessa
    "Lubbock": (33.5779, -101.8552),      # Lubbock
    "Potter": (35.2220, -101.8313),       # Amarillo
    "Nueces": (27.8006, -97.3964),        # Corpus Christi
    "Jefferson": (30.0802, -94.1266),     # Beaumont
    "Smith": (32.3513, -95.3011),         # Tyler
    # --- Added: real top-15 outage counties not already covered above ---
    "Montgomery": (30.3116, -95.4560),    # Conroe
    "Brazoria": (29.1694, -95.4185),      # Angleton
    "Liberty": (30.0577, -94.7955),       # Liberty
    "San Jacinto": (30.5919, -95.1283),   # Coldspring
    "Polk": (30.7118, -94.9327),          # Livingston
    "Walker": (30.7235, -95.5508),        # Huntsville
    "Waller": (30.0977, -96.0778),        # Hempstead
    "Gregg": (32.5007, -94.7405),         # Longview
    "Wharton": (29.3116, -96.1027),       # Wharton
}

HOT_HOUR = 16  # 4 PM local
HOT_TIME = f"{HOT_HOUR:02d}:00"
QUERY_DATE = "2025-07-15"

def _point_aoi(lat, lon, buffer_deg=0.01):
    return {
        "type": "FeatureCollection",
        "features": [{
            "type": "Feature", "properties": {},
            "geometry": {"type": "Polygon", "coordinates": [[
                [lon - buffer_deg, lat - buffer_deg], [lon + buffer_deg, lat - buffer_deg],
                [lon + buffer_deg, lat + buffer_deg], [lon - buffer_deg, lat + buffer_deg],
                [lon - buffer_deg, lat - buffer_deg],
            ]]},
        }],
    }

heat_scores = {}
for county in candidate_counties:
    if county not in COUNTY_CENTROIDS:
        print(f"  [skip] no centroid coordinates yet for {county}")
        continue
    lat, lon = COUNTY_CENTROIDS[county]

    # Step 1: environmental_parameters() REQUIRES an ambient temperature
    # (Celsius) as input - it's the "thermal anchor" the API derives heat
    # index / apparent temp from (per notebooks/02_environmental_parameters.ipynb
    # in FortyGuard's quickstart repo). Source it from create_heatmap over a
    # tiny buffer polygon around the point, same pattern as their use-case
    # notebooks (peak_temp_c -> temperature=).
    heatmap_resp = client.create_heatmap(
        polygon_aoi=_point_aoi(lat, lon), start_date=QUERY_DATE, filter_type=3, granularity=100,
    )
    ambient_temp_c = heatmap_resp["result"]["stats_data"]["temperature_stats"]["mean"]

    # Step 2: filter_type=1 (single hour) at the hot hour -> each parameter
    # comes back as ONE value, not a 24-length array.
    resp = client.environmental_parameters(
        latitude=lat, longitude=lon, temperature=ambient_temp_c,
        start_date=QUERY_DATE, start_time=HOT_TIME, filter_type=1,
    )
    locations = resp["result"].get("locations") or []
    if not locations:
        print(f"  [warn] no locations returned for {county}")
        continue
    params = locations[0].get("parameters", {})
    heat_scores[county] = params.get("apparent_temperature_celsius")

print("Heat scores (apparent temperature, hot hour):", heat_scores)

Submitted -> activity_id=351115f8-bc61-4967-bf7f-a6dce8992742
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: completed
Done.
Submitted -> activity_id=7e59bdff-76ab-4069-a0e3-e87c51a96f4a
  status: processing
  status: completed
Done.
Submitted -> activity_id=7ce83d6a-c9e8-4a09-96b8-257e24332f8a
  status: processing
  status: processing
  status: processing
  status: processing
  status: completed
Done.
Submitted -> activity_id=a6e697c3-c0dd-47f1-885b-172bc76dd7aa
  status: processing
  status: completed
Done.
Submitted -> activity_id=d8131526-6dfa-4cf3-8b17-fa1549692cb4
  status: processing
  status: processing
  status: processing
  status: processing
  status: completed
Done.
Submitted -> activity_id=655845e6-e28a-45c4-a880-671a5958ca1b
  status: processing
  status: processing
  status: completed
Done.
Submitted -> activity_id=1afe0087-a325-4bb9-93f7-edca420637bf
  status: p

In [ ]:
import json

heat_scores = {'Bexar': [35.1], 'Brazoria': [34.8], 'Cameron': [34.3], 'Collin': [35.5],
                'Dallas': [36.5], 'Ector': [32.4], 'El Paso': [36.0], 'Fort Bend': [35.3],
                'Galveston': [33.2], 'Gregg': [37.5], 'Harris': [36.1], 'Hidalgo': [35.8],
                'Jefferson': [36.2], 'Liberty': [35.2], 'Lubbock': [31.2], 'Midland': [31.9],
                'Montgomery': [35.7], 'Nueces': [33.8], 'Polk': [35.3], 'Potter': [30.2],
                'San Jacinto': [34.8], 'Smith': [36.8], 'Tarrant': [37.2], 'Travis': [35.1],
                'Walker': [34.8], 'Waller': [33.3], 'Wharton': [34.6]}

with open("tx_county_heat_scores.json", "w") as f:
    json.dump(heat_scores, f, indent=2)

print(f"Saved heat scores for {len(heat_scores)} counties.")

Saved heat scores for 27 counties.


In [ ]:
"""
[TX-2, STAGE B cont.] Combine heat + outages + vulnerability into one
scoreboard per candidate county, then select PILOT_COUNTY as the top-ranked
county. Uses the already-cached heat_scores (tx_county_heat_scores.json),
NOT a re-run of the FortyGuard API cell.

VULNERABILITY COLUMN: P_EJINDEX doesn't exist in the EJScreen 2024 release
(confirmed via EJScreen_2024_Tract_Percentiles_Columns.xlsx - no EJINDEX
column at all). Using P_DEMOGIDX_5 (Percentile for Supplemental Demographic
Index) instead - it's the closer fit for "outages hit vulnerable people
harder" since, unlike P_DEMOGIDX_2, it folds in unemployment, less-than-HS
education, AND disability - disability in particular matters directly here
(medical equipment dependent on power).
"""
import json
import pandas as pd
import geopandas as gpd

VULN_COL = "P_DEMOGIDX_5"  # was P_EJINDEX - see note above

# ── 1. Heat: load from cache, don't re-hit the API ──
with open("tx_county_heat_scores.json") as f:
    heat_scores_cached = json.load(f)

def _scalar(v):
    return v[0] if isinstance(v, (list, tuple)) else v

heat_series = pd.Series({c: _scalar(v) for c, v in heat_scores_cached.items()}, name="heat")

# ── 2. Outages: already computed in Section 1.2.1 ──
outage_series = county_outage_totals.reindex(candidate_counties)

# ── 3. Vulnerability: population-weighted mean P_DEMOGIDX_5 per COUNTY ──
ejscreen = gpd.read_file("ejscreen_tx_clean.geojson")
ejscreen["CNTY_NORM"] = (
    ejscreen["CNTY_NAME"].str.strip().str.lower().str.replace(" county", "", regex=False)
)

def weighted_vuln(county_name):
    rows = ejscreen[ejscreen["CNTY_NORM"] == county_name.lower()]
    if rows.empty or rows["ACSTOTPOP"].sum() == 0:
        return float("nan")
    return (rows[VULN_COL] * rows["ACSTOTPOP"]).sum() / rows["ACSTOTPOP"].sum()

vuln_series = pd.Series({c: weighted_vuln(c) for c in candidate_counties}, name="vulnerability")

# ── 4. Combine ──
scoreboard = pd.DataFrame({
    "heat": heat_series,
    "outages": outage_series,
    "vulnerability": vuln_series,
}).reindex(candidate_counties)

missing = scoreboard[scoreboard.isna().any(axis=1)]
if not missing.empty:
    print("WARNING: excluded from ranking (missing data):")
    print(missing)
scoreboard = scoreboard.dropna()

def normalize_0_100(s):
    lo, hi = s.min(), s.max()
    if hi - lo == 0:
        return pd.Series(0.0, index=s.index)
    return (s - lo) / (hi - lo) * 100.0

# ⚠️ ASSUMPTION - equal weighting, not documented anywhere in the notebook.
STAGE_B_WEIGHTS = {"heat": 1/3, "outages": 1/3, "vulnerability": 1/3}

scoreboard["heat_score"] = normalize_0_100(scoreboard["heat"])
scoreboard["outage_score"] = normalize_0_100(scoreboard["outages"])
scoreboard["vulnerability_score"] = normalize_0_100(scoreboard["vulnerability"])

scoreboard["composite_score"] = (
    scoreboard["heat_score"] * STAGE_B_WEIGHTS["heat"]
    + scoreboard["outage_score"] * STAGE_B_WEIGHTS["outages"]
    + scoreboard["vulnerability_score"] * STAGE_B_WEIGHTS["vulnerability"]
)

scoreboard = scoreboard.sort_values("composite_score", ascending=False)
print(scoreboard.round(1))

PILOT_COUNTY = scoreboard.index[0]
print(f"\n>>> PILOT_COUNTY selected: {PILOT_COUNTY} "
      f"(composite score {scoreboard.iloc[0]['composite_score']:.1f})")

             heat    outages  vulnerability  heat_score  outage_score  \
Harris       36.1  560269098           56.8        80.8         100.0   
Polk         35.3    7268745           83.4        69.9           1.3   
Hidalgo      35.8    1311104           79.3        76.7           0.2   
Gregg        37.5    3931039           60.9       100.0           0.7   
El Paso      36.0     839522           71.2        79.5           0.1   
Liberty      35.2    9998937           75.3        68.5           1.7   
San Jacinto  34.8    7275108           78.9        63.0           1.3   
Jefferson    36.2    2535298           65.7        82.2           0.4   
Dallas       36.5    8653407           58.6        86.3           1.5   
Cameron      34.3     596052           76.2        56.2           0.1   
Smith        36.8    9409274           52.2        90.4           1.6   
Tarrant      37.2    3686613           46.2        95.9           0.6   
Wharton      34.6    3721228           65.8        

In [ ]:
"""
[TX-2, STAGE C, CACHED] Build the precise FortyGuard AOI for PILOT_COUNTY
from real EJScreen block-group boundaries - same contiguous-growth
strategy used for Fresno (start from the highest-vulnerability block
group, grow outward only through touching block groups, stop before the
Basic tier's 10 mi^2 cap). Block groups are smaller than census tracts,
so expect MORE units selected than the 1-2 tracts Fresno needed to fill
the same area cap.

FIXES:
- CNTY_NAME in this EJScreen release is "Harris County" (full name), while
  PILOT_COUNTY is just "Harris" - normalize both sides before comparing.
- P_EJINDEX doesn't exist in this release - using P_DEMOGIDX_5 instead
  (see Section 1.5's cleaning cell for the full explanation).

CACHING: skips the AOI-growth computation if {pilot}_aoi_precise.geojson
already exists.
"""
import os
import json as _json
import geopandas as gpd
from shapely.ops import unary_union
from shapely.geometry import MultiPolygon

BUFFER_EPS_DEG = 0.00001
INPUT_PATH = "ejscreen_tx_clean.geojson"
OUTPUT_PATH = f"{PILOT_COUNTY.lower().replace(' ', '_')}_aoi_precise.geojson"
force_refresh = False

BASIC_TIER_CAP_MI2 = 10.0
SAFETY_MARGIN_MI2 = 1.0
TARGET_AREA_MI2 = BASIC_TIER_CAP_MI2 - SAFETY_MARGIN_MI2
SQKM_PER_MI2 = 2.58999
VULN_COL = "P_DEMOGIDX_5"  # was P_EJINDEX - see Section 1.5 note


def load_pilot_blockgroups(path: str = INPUT_PATH) -> gpd.GeoDataFrame:
    gdf = gpd.read_file(path)
    gdf = gdf.set_crs("EPSG:4326") if gdf.crs is None else gdf.to_crs("EPSG:4326")
    normalized = gdf["CNTY_NAME"].str.strip().str.lower().str.replace(" county", "", regex=False)
    pilot = gdf[normalized == PILOT_COUNTY.strip().lower()].copy()
    if pilot.empty:
        raise RuntimeError(f"No block groups found for county '{PILOT_COUNTY}'")
    return pilot

  # ~1m at these latitudes - tiny relative to block-group scale


def build_precise_aoi(blockgroups: gpd.GeoDataFrame):
    ranked = blockgroups.sort_values(VULN_COL, ascending=False).to_crs("EPSG:3083")  # Texas Centric Albers Equal Area
    remaining = ranked.copy()
    selected_geoms, selected_ids = [], []
    running_area_mi2 = 0.0

    while not remaining.empty:
        if not selected_geoms:
            candidate_idx = remaining.index[0]
        else:
            current_union = unary_union(selected_geoms)
            touching = remaining[remaining.geometry.touches(current_union) | remaining.geometry.intersects(current_union)]
            if touching.empty:
                break
            candidate_idx = touching.index[0]

        candidate_geom = remaining.loc[candidate_idx, "geometry"]
        candidate_id = remaining.loc[candidate_idx, "ID"]
        candidate_union = unary_union(selected_geoms + [candidate_geom])
        candidate_area_mi2 = candidate_union.area / 1_000_000 / SQKM_PER_MI2

        if candidate_area_mi2 > TARGET_AREA_MI2:
            if not selected_geoms:
                selected_geoms.append(candidate_geom)
                selected_ids.append(candidate_id)
                running_area_mi2 = candidate_area_mi2
            break

        selected_geoms.append(candidate_geom)
        selected_ids.append(candidate_id)
        running_area_mi2 = candidate_area_mi2
        remaining = remaining.drop(index=candidate_idx)

    # ---- FIX: unary_union can produce a MultiPolygon (corner-only touching,
    # or micro-gaps from TIGER topology mismatches) - FortyGuard rejects
    # anything that isn't a single Polygon. Close micro-gaps with a tiny
    # buffer-out/buffer-in pass; fall back to convex hull if pieces are
    # still disjoint after that (documented, since it slightly overshoots
    # the selected area rather than silently dropping a piece). ----
    final_union_m = unary_union(selected_geoms)
    if isinstance(final_union_m, MultiPolygon):
        closed = final_union_m.buffer(BUFFER_EPS_DEG).buffer(-BUFFER_EPS_DEG)
        if isinstance(closed, MultiPolygon):
            print(f"WARNING: {len(closed.geoms)} disjoint pieces even after gap-closing "
                  f"(likely corner-only touching, not a topology gap) - "
                  f"falling back to convex hull. This will slightly overshoot the "
                  f"selected block groups' actual combined area.")
            final_union_m = final_union_m.convex_hull
        else:
            print("Closed micro-gaps between block groups via buffer pass - now a single Polygon.")
            final_union_m = closed

    final_union_wgs84 = gpd.GeoSeries([final_union_m], crs="EPSG:3083").to_crs("EPSG:4326").iloc[0]

    polygon_aoi = {
        "type": "FeatureCollection",
        "features": [{"type": "Feature", "properties": {}, "geometry": final_union_wgs84.__geo_interface__}],
    }
    return polygon_aoi, running_area_mi2, selected_ids


if os.path.exists(OUTPUT_PATH) and not force_refresh:
    print(f"{OUTPUT_PATH} already exists - loading cached copy.")
    with open(OUTPUT_PATH) as f:
        polygon_aoi = _json.load(f)
    print(f"Loaded existing {PILOT_COUNTY} AOI.")
else:
    pilot_bgs = load_pilot_blockgroups()
    print(f"{PILOT_COUNTY} County block groups available: {len(pilot_bgs)}")

    polygon_aoi, area_mi2, bg_ids = build_precise_aoi(pilot_bgs)
    print(f"Selected {len(bg_ids)} block group(s): {bg_ids}")
    print(f"Precise combined area: {area_mi2:.2f} mi\u00b2 (cap: {BASIC_TIER_CAP_MI2} mi\u00b2)")

    selected_rows = pilot_bgs[pilot_bgs["ID"].isin(bg_ids)]
    print(f"Total population: {selected_rows['ACSTOTPOP'].sum():,.0f}")
    print(f"Average {VULN_COL}: {selected_rows[VULN_COL].mean():.1f}")

    with open(OUTPUT_PATH, "w") as f:
        _json.dump(polygon_aoi, f)
    print(f"Saved to {OUTPUT_PATH}")

Harris County block groups available: 1115
Selected 7 block group(s): ['48201222501', '48201222601', '48201240101', '48201222800', '48201222401', '48201221800', '48201221900']
Precise combined area: 8.32 mi² (cap: 10.0 mi²)
Total population: 24,892
Average P_DEMOGIDX_5: 98.9
Saved to harris_aoi_precise.geojson


## 3. Heat Zone Grid & Spatial Joins

### 3.1 Building the canonical grid

Same approach as the California version: every dataset gets joined onto one fixed
grid of 1x1 km cells (`CELL_SIZE_DEG = 0.01`), clipped to the pilot AOI from Section
2. This grid is still the spatial "primary key" for the rest of the project -
nothing about this step changes with the state, only the AOI file it clips to.

In [ ]:
"""
[TX-3.1, CACHED] Canonical GridHeat AI "heat zone" grid for the Texas
pilot county, built directly from the AOI produced in Stage C above.
Identical logic to the California version - only the input/output
filenames change.

CACHING: skips grid construction if heat_zone_grid_tx.geojson already
exists - loads it instead. Delete the file or set force_refresh=True to
rebuild (needed if the AOI file changes).
"""
import os
import geopandas as gpd
from shapely.geometry import box

CELL_SIZE_DEG = 0.01  # ~1.1 km at this latitude
GRID_CRS = "EPSG:4326"
AOI_PATH = OUTPUT_PATH  # the {pilot}_aoi_precise.geojson path from Stage C
GRID_OUTPUT_PATH = "heat_zone_grid_tx.geojson"
force_refresh = False


def build_grid(minx, miny, maxx, maxy, cell_size_deg=CELL_SIZE_DEG):
    grid_cells = []
    x = minx
    while x < maxx:
        y = miny
        while y < maxy:
            grid_cells.append(box(x, y, x + cell_size_deg, y + cell_size_deg))
            y += cell_size_deg
        x += cell_size_deg
    grid = gpd.GeoDataFrame({"geometry": grid_cells}, crs=GRID_CRS)
    grid["zone_id"] = [f"Z{idx:05d}" for idx in range(len(grid))]
    return grid[["zone_id", "geometry"]]


def build_grid_from_aoi(aoi_path: str, cell_size_deg: float = CELL_SIZE_DEG, clip_to_aoi: bool = True) -> gpd.GeoDataFrame:
    aoi_gdf = gpd.read_file(aoi_path)
    aoi_gdf = aoi_gdf.set_crs(GRID_CRS) if aoi_gdf.crs is None else aoi_gdf.to_crs(GRID_CRS)
    minx, miny, maxx, maxy = aoi_gdf.total_bounds
    grid = build_grid(minx, miny, maxx, maxy, cell_size_deg)
    if clip_to_aoi:
        aoi_union = aoi_gdf.geometry.union_all()
        grid = grid[grid.geometry.intersects(aoi_union)].reset_index(drop=True)
    return grid


if os.path.exists(GRID_OUTPUT_PATH) and not force_refresh:
    print(f"{GRID_OUTPUT_PATH} already exists - loading cached copy "
          f"(set force_refresh=True above to rebuild).")
    grid = gpd.read_file(GRID_OUTPUT_PATH)
    print(f"Loaded grid cells: {len(grid)}")
else:
    grid = build_grid_from_aoi(AOI_PATH)
    print(f"Grid cells: {len(grid)}")
    grid.to_file(GRID_OUTPUT_PATH, driver="GeoJSON")
    print(f"Saved to {GRID_OUTPUT_PATH}")


Grid cells: 50
Saved to heat_zone_grid_tx.geojson


### 3.2 Joining every layer onto the grid

Same two join primitives as before (`join_lines` / `join_points` for line and
point geometries; `join_polygons_area_weighted` for polygons, still split into
`extensive_cols` vs `intensive_cols` for the same reason as the California
version - population is additive, an EJ index score is not).

**What's genuinely different for Texas, and why it matters:**

- **EAGLE-I outages have NO fine geometry** - it's a county-level aggregate
  (customers-out per county per 15-min reading), not per-event points/polygons
  like CPUC's PSPS data was. Since the whole AOI sits inside a single county by
  construction (Stage C built it from that county's block groups), there is
  nothing to spatially differentiate - every grid cell in this AOI gets the
  *same* county-level outage figure. This is a real resolution ceiling in the
  data, not a join bug, and it's documented here so it isn't mistaken for one
  later when every zone shows an identical outage number.
- **ERCOT demand is still not spatially joined** here, same as CAISO wasn't -
  it's weather-zone-level, not zone-level, and feeds the demand-forecasting
  model in Section 4 directly.

In [ ]:
# NOTE: relocated to run AFTER Section 3.1 (grid build) - originally sat right after the EIA-861 cells (Section 1.3b), before Section 2/3.1 had run, which caused a FileNotFoundError on heat_zone_grid_tx.geojson.
"""
[TX-1.3c, CACHED, FALLBACK] Infrastructure-density weighting - proxy for
redistributing the single ERCOT_Total series (Section 1.3 / EIA-930)
across the pilot county's heat_zone grid cells.

WHY THIS EXISTS: EIA-861 utility-level sales weighting (Section 1.3b)
does NOT work for ERCOT territory - Texas retail electricity is
DEREGULATED, meaning the local utility (e.g. CenterPoint Energy in
Harris County) only owns/operates the wires (a Transmission & Distribution
Utility / TDU) and is legally barred from selling electricity directly to
end customers. Actual retail sales are made by ~100+ separate "Retail
Electric Providers" (REPs, e.g. TXU Energy, Reliant, Gexa) who report
sales in EIA-861 with NO reliable per-county attribution. This was
confirmed empirically: CenterPoint Energy (utility_id 8901) has zero rows
in the Sales_Ult_Cust_2024 sheet despite being the dominant service-
territory utility for Harris County - not a data-quality bug, a structural
mismatch between EIA-861's sales schedule and how the ERCOT retail market
is organized.

APPROACH: instead of weighting by utility sales share, weight each grid
cell (zone_id) by its share of built electrical infrastructure - substation
count + transmission line count, both already fetched and cleaned in
Section 1.1 (tx_substations_clean.geojson, tx_transmission_lines_clean.geojson).
Rationale: zones with more substations/transmission line presence
generally serve denser load - this is still a PROXY, not a measurement,
same caveat class as the EIA-861 approach would have had, just built from
data that actually applies to a deregulated market.

CACHING: skips the computation if infra_weights_tx.csv already exists.
"""
import os
import geopandas as gpd
import pandas as pd

OUTPUT_PATH = "infra_weights_tx.csv"
force_refresh = False

GRID_PATH = "heat_zone_grid_tx.geojson"
SUBSTATIONS_PATH = "tx_substations_clean.geojson"
TRANSMISSION_PATH = "tx_transmission_lines_clean.geojson"

# Relative importance of each infrastructure type in the composite weight.
# Substations are a stronger proxy for load-serving capacity than a line
# merely passing through a cell, hence the heavier weight.
SUBSTATION_WEIGHT = 0.65
TRANSMISSION_WEIGHT = 0.35


def count_points_per_zone(grid: gpd.GeoDataFrame, points_gdf: gpd.GeoDataFrame) -> pd.Series:
    points_gdf = points_gdf.to_crs(grid.crs)
    joined = gpd.sjoin(points_gdf, grid, how="inner", predicate="within")
    return joined.groupby("zone_id").size()


def count_lines_per_zone(grid: gpd.GeoDataFrame, lines_gdf: gpd.GeoDataFrame) -> pd.Series:
    lines_gdf = lines_gdf.to_crs(grid.crs)
    joined = gpd.sjoin(grid, lines_gdf, how="inner", predicate="intersects")
    # a line can intersect a zone via multiple segments after the sjoin -
    # count distinct line objects per zone, not row duplicates
    id_col = next((c for c in lines_gdf.columns if c.upper() in ("OBJECTID", "OBJECTID_1", "ID")), None)
    if id_col is None:
        return joined.groupby("zone_id").size()
    return joined.groupby("zone_id")[id_col].nunique()


if os.path.exists(OUTPUT_PATH) and not force_refresh:
    print(f"{OUTPUT_PATH} already exists - loading cached copy "
          f"(set force_refresh=True above to recompute).")
    df_infra_weights = pd.read_csv(OUTPUT_PATH)
    print(f"Loaded shape: {df_infra_weights.shape}")
else:
    for p in (GRID_PATH, SUBSTATIONS_PATH, TRANSMISSION_PATH):
        if not os.path.exists(p):
            raise FileNotFoundError(
                f"Required input '{p}' not found - run the corresponding "
                f"cell in Section 1.1 / 3.1 first."
            )

    grid = gpd.read_file(GRID_PATH)
    substations = gpd.read_file(SUBSTATIONS_PATH)
    transmission = gpd.read_file(TRANSMISSION_PATH)

    print(f"Grid cells: {len(grid)}")
    print(f"Substations (statewide, pre-clip): {len(substations)}")
    print(f"Transmission lines (statewide, pre-clip): {len(transmission)}")

    sub_counts = count_points_per_zone(grid, substations)
    line_counts = count_lines_per_zone(grid, transmission)

    df_infra = pd.DataFrame({"zone_id": grid["zone_id"]}).set_index("zone_id")
    df_infra["n_substations"] = sub_counts.reindex(df_infra.index).fillna(0).astype(int)
    df_infra["n_transmission_lines"] = line_counts.reindex(df_infra.index).fillna(0).astype(int)

    n_zones_with_any_infra = ((df_infra["n_substations"] > 0) | (df_infra["n_transmission_lines"] > 0)).sum()
    print(f"\nZones with at least one substation or transmission line: "
          f"{n_zones_with_any_infra} / {len(df_infra)}")

    if n_zones_with_any_infra == 0:
        raise ValueError(
            "No zone in the pilot AOI intersects any substation or "
            "transmission line - infra-based weighting cannot be computed. "
            "Check that the grid and HIFLD layers share the same AOI/CRS."
        )

    # ---- Normalize each component to 0-1 within the AOI, then combine ----
    def normalize_share(series: pd.Series) -> pd.Series:
        total = series.sum()
        if total == 0:
            return pd.Series(0.0, index=series.index)
        return series / total

    sub_share = normalize_share(df_infra["n_substations"])
    line_share = normalize_share(df_infra["n_transmission_lines"])

    df_infra["infra_weight"] = SUBSTATION_WEIGHT * sub_share + TRANSMISSION_WEIGHT * line_share

    # Zones with literally zero infrastructure get zero weight by
    # construction (no substations/lines -> no plausible load share) -
    # this is a modeling choice, stated explicitly rather than silently
    # smoothed over with a floor value.
    zero_weight_zones = (df_infra["infra_weight"] == 0).sum()
    if zero_weight_zones > 0:
        print(f"NOTE: {zero_weight_zones} zone(s) have zero infrastructure "
              f"and will receive zero demand share under this proxy.")

    # Renormalize so weights sum to exactly 1.0 across the AOI (handles
    # any floating-point drift from the two-component blend above).
    weight_sum = df_infra["infra_weight"].sum()
    if weight_sum > 0:
        df_infra["infra_weight"] = df_infra["infra_weight"] / weight_sum

    df_infra_weights = df_infra.reset_index()[
        ["zone_id", "n_substations", "n_transmission_lines", "infra_weight"]
    ].sort_values("infra_weight", ascending=False).reset_index(drop=True)

    print(f"\nInfrastructure-based zone weights (top 10 by weight):")
    print(df_infra_weights.head(10).to_string(index=False))
    print(f"\nSum of infra_weight across all zones: {df_infra_weights['infra_weight'].sum():.6f}")

    df_infra_weights.to_csv(OUTPUT_PATH, index=False)
    print(f"\nSaved to {OUTPUT_PATH}")

Grid cells: 50
Substations (statewide, pre-clip): 4939
Transmission lines (statewide, pre-clip): 10233

Zones with at least one substation or transmission line: 15 / 50
NOTE: 35 zone(s) have zero infrastructure and will receive zero demand share under this proxy.

Infrastructure-based zone weights (top 10 by weight):
zone_id  n_substations  n_transmission_lines  infra_weight
 Z00038              1                    10      0.296212
 Z00036              1                     4      0.248485
 Z00035              1                     3      0.240530
 Z00046              0                     4      0.031818
 Z00030              0                     3      0.023864
 Z00022              0                     3      0.023864
 Z00014              0                     3      0.023864
 Z00006              0                     3      0.023864
 Z00037              0                     2      0.015909
 Z00032              0                     2      0.015909

Sum of infra_weight across all 

In [ ]:
"""
[TX-3.2, CACHED] Join every layer onto the canonical Texas heat_zone grid.
Same join_lines / join_points / join_polygons_area_weighted primitives as
the California version (unchanged logic - only which files get fed in
changes).

CACHING: if all three output CSVs already exist, this cell loads them
instead of recomputing. Delete any of them, or set force_refresh=True, to
recompute all.
"""
import os
import geopandas as gpd
import pandas as pd

GRID_CRS = "EPSG:4326"
OUTPUT_PATHS = ["lines_joined_tx.csv", "vulnerability_joined_tx.csv", "battery_joined_tx.csv"]
force_refresh = True


def join_lines(grid: gpd.GeoDataFrame, lines_gdf: gpd.GeoDataFrame, id_col: str = "OBJECTID") -> pd.DataFrame:
    lines_gdf = lines_gdf.to_crs(GRID_CRS)
    joined = gpd.sjoin(grid, lines_gdf, how="inner", predicate="intersects")
    return joined[["zone_id", id_col]].drop_duplicates().reset_index(drop=True)


def join_points(grid: gpd.GeoDataFrame, points_gdf: gpd.GeoDataFrame, id_col: str) -> pd.DataFrame:
    points_gdf = points_gdf.to_crs(GRID_CRS)
    joined = gpd.sjoin(points_gdf, grid, how="inner", predicate="within")
    return joined[["zone_id", id_col]].drop_duplicates().reset_index(drop=True)


def join_polygons_area_weighted(grid, polygons_gdf, id_col, extensive_cols=None, intensive_cols=None):
    """extensive_cols (counts, e.g. population): apportioned by the source
    polygon's area fraction. intensive_cols (scores/rates): area-weighted
    AVERAGE within each cell (not diluted by how much of the source
    polygon's total area landed there)."""
    polygons_gdf = polygons_gdf.to_crs(GRID_CRS)
    extensive_cols = extensive_cols or []
    intensive_cols = intensive_cols or []

    grid_m = grid.to_crs("EPSG:3083")  # Texas Centric Albers Equal Area
    polys_m = polygons_gdf.to_crs("EPSG:3083")
    overlay = gpd.overlay(grid_m, polys_m, how="intersection")
    overlay["piece_area"] = overlay.geometry.area

    results = []
    if extensive_cols:
        poly_total_area = polys_m.set_index(id_col).geometry.area
        overlay["poly_total_area"] = overlay[id_col].map(poly_total_area)
        overlay["area_fraction_of_source"] = overlay["piece_area"] / overlay["poly_total_area"]
        ext = overlay.copy()
        for col in extensive_cols:
            ext[col] = ext[col] * ext["area_fraction_of_source"]
        results.append(ext.groupby("zone_id")[extensive_cols].sum())

    if intensive_cols:
        cell_covered_area = overlay.groupby("zone_id")["piece_area"].transform("sum")
        overlay["area_fraction_of_cell"] = overlay["piece_area"] / cell_covered_area
        intens = overlay.copy()
        for col in intensive_cols:
            intens[col] = intens[col] * intens["area_fraction_of_cell"]
        results.append(intens.groupby("zone_id")[intensive_cols].sum())

    if not results:
        raise ValueError("Must supply at least one of extensive_cols or intensive_cols")
    return pd.concat(results, axis=1).reset_index()


if all(os.path.exists(p) for p in OUTPUT_PATHS) and not force_refresh:
    print("All joined outputs already exist - loading cached copies "
          "(set force_refresh=True above to recompute all joins).")
    lines_joined = pd.read_csv("lines_joined_tx.csv")
    vulnerability_joined = pd.read_csv("vulnerability_joined_tx.csv")
    battery_joined = pd.read_csv("battery_joined_tx.csv")
    print(f"Loaded: lines_joined ({len(lines_joined)}), vulnerability_joined "
          f"({len(vulnerability_joined)}), battery_joined ({len(battery_joined)})")
else:
    grid = gpd.read_file("heat_zone_grid_tx.geojson")

    # ---- 1. Transmission lines (HIFLD) ----
    lines_gdf = gpd.read_file("tx_transmission_lines_clean.geojson")
    lines_id_col = "OBJECTID" if "OBJECTID" in lines_gdf.columns else lines_gdf.columns[0]
    lines_joined = join_lines(grid, lines_gdf, id_col=lines_id_col)
    print(f"[1] Transmission lines -> {lines_joined['zone_id'].nunique()} / {len(grid)} zones")

    # ---- 2. EJScreen (vulnerability) ----
    ej_gdf = gpd.read_file("ejscreen_tx_clean.geojson")
    EJ_EXTENSIVE = ["ACSTOTPOP"]
    EJ_INTENSIVE = ["P_DEMOGIDX_5", "PEOPCOLORPCT", "OVER64PCT", "UNDER5PCT"]
    vulnerability_joined = join_polygons_area_weighted(
        grid, ej_gdf, id_col="ID",
        extensive_cols=[c for c in EJ_EXTENSIVE if c in ej_gdf.columns],
        intensive_cols=[c for c in EJ_INTENSIVE if c in ej_gdf.columns],
    )
    print(f"[2] EJScreen -> {len(vulnerability_joined)} / {len(grid)} zones matched")

    # ---- 3. Battery storage ----
    battery_gdf = gpd.read_file("battery_storage_tx_clean.geojson")
    battery_pairs = join_points(grid, battery_gdf, id_col="Plant Code")
    battery_joined = (
        battery_pairs
        .merge(battery_gdf[["Plant Code", "Plant Name", "Nameplate Capacity (MW)",
                             "Nameplate Energy Capacity (MWh)"]], on="Plant Code", how="left")
        .groupby("zone_id")
        .agg(n_battery_sites=("Plant Code", "nunique"),
             total_capacity_mw=("Nameplate Capacity (MW)", "sum"),
             total_energy_mwh=("Nameplate Energy Capacity (MWh)", "sum"))
        .reset_index()
    )
    print(f"[3] Battery storage -> {len(battery_joined)} / {len(grid)} zones have a battery site")

    # ---- 4. EAGLE-I outages: county-level, no fine geometry - broadcast the
    # pilot county's cumulative summer customers-out total to every zone in
    # this AOI (all zones fall inside the same county by construction) ----
    df_outages = pd.read_csv("eagle_i_tx_clean.csv")
    county_col = "county" if "county" in df_outages.columns else "county_name"
    customers_col = "customers_out" if "customers_out" in df_outages.columns else "sum"
    pilot_total_outage_customers = (
        df_outages[df_outages[county_col].str.strip() == PILOT_COUNTY][customers_col].sum()
    )
    outage_joined = grid[["zone_id"]].copy()
    outage_joined["total_customers_deenergized"] = pilot_total_outage_customers
    print(f"[4] EAGLE-I -> broadcast {pilot_total_outage_customers:,.0f} cumulative summer "
          f"customers-out ({PILOT_COUNTY} County total) to all {len(grid)} zones")

    # ---- 5. ERCOT demand: no geometry, intentionally not spatially joined ----
    ercot_df = pd.read_csv("ercot_demand_clean.csv")
    print(f"[5] ERCOT demand: {ercot_df.shape[0]} rows - feeds the demand model separately (Section 4), not this join.")

    lines_joined.to_csv("lines_joined_tx.csv", index=False)
    vulnerability_joined.to_csv("vulnerability_joined_tx.csv", index=False)
    battery_joined.to_csv("battery_joined_tx.csv", index=False)
    outage_joined.to_csv("outage_joined_tx.csv", index=False)
    print("\nSaved: lines_joined_tx.csv, vulnerability_joined_tx.csv, battery_joined_tx.csv, outage_joined_tx.csv")


[1] Transmission lines -> 15 / 50 zones
[2] EJScreen -> 50 / 50 zones matched
[3] Battery storage -> 0 / 50 zones have a battery site
[4] EAGLE-I -> broadcast 560,269,098 cumulative summer customers-out (Harris County total) to all 50 zones
[5] ERCOT demand: 2928 rows - feeds the demand model separately (Section 4), not this join.

Saved: lines_joined_tx.csv, vulnerability_joined_tx.csv, battery_joined_tx.csv, outage_joined_tx.csv


## 4. Demand Forecasting Model (Temperature → Electricity Demand)

**Goal:** learn how the pilot county's weather-zone peak demand responds to
temperature in the pilot AOI, so the risk engine's "demand stress" component
reflects a real predicted relationship rather than a placeholder.

**Data:** FortyGuard historical daily average temperature over the pilot AOI, for
the same days already pulled from ERCOT, merged with ERCOT weather-zone demand for
whichever zone the pilot county was gated into in Section 2 (Stage A). This is
still explicitly a **proxy relationship** - temperature in one small ~9 mi² AOI vs.
demand across an entire ERCOT weather zone - if anything a coarser proxy than the
California version (which compared against a utility service territory, smaller
than a weather zone), and is stated as such rather than presented as more precise
than it is.

**Note on the findings below:** the California version of this section documents
specific numbers found by actually running the pipeline on Fresno data (a
near-zero single-split R², a Pearson r ≈ 0.76, linear regression winning CV with
R²=0.316). Those numbers are Fresno-specific and won't carry over to Texas - the
cells below keep the same *diagnostic methodology* (check X variance, check raw
correlation, compare CV across model families) but leave the actual findings
un-filled-in until you run this against the real Texas pull. Update this markdown
with whatever the Texas run actually shows, the same way the Fresno numbers got
written down after the fact rather than assumed in advance.

In [ ]:
"""
[TX-4, CACHED] Demand Forecasting Model (Temperature -> Electricity Demand)
data preparation.

CHANGE FROM ORIGINAL: previously merged pilot-AOI temperature against the
ERCOT weather-zone column the pilot county was gated into (PILOT_ZONE).
Section 1.3 now sources demand from EIA-930, which only publishes a single
ERCOT-wide series (ERCOT_Total) - there is no PILOT_ZONE column to select
anymore. This cell merges pilot-AOI temperature against that single
system-wide series instead. Documented consequence: the model now learns
"pilot-AOI temperature -> ALL of ERCOT's demand", an even coarser proxy
than "AOI temp -> weather-zone demand" was - stated explicitly, not hidden.

CHANGE (phase 2): also pulls solar irradiance at the pilot AOI's centroid
via FortyGuard's `environmental_parameters` endpoint (point-based, not
polygon), one extra API call per training day. This feeds the multivariate
model built in Section 4.1b/4.2 (solar suppresses net grid demand during
daylight - relevant for ERCOT given Texas's large and growing solar fleet).
If this endpoint isn't reachable with your plan/key, solar_irradiance_wm2
comes back all-NaN and the multivariate model downstream just drops it
(same graceful-degradation pattern as the temp-only fallback).

Historical temperature: FortyGuard over the pilot AOI.
Demand: ercot_demand_clean.csv (ERCOT_Total column, from EIA-930).

CACHING: if tx_temp_demand_merged.csv already exists, this cell skips the
FortyGuard API calls entirely (each day = one quota-limited heatmap call,
plus one env_params call for solar) and just loads the existing merged
dataset. Set force_refresh=True (or delete the file) to resume/redo the
fetch.
"""
import os
import time
from datetime import datetime, timedelta

import pandas as pd
from shapely.geometry import shape

MERGED_OUTPUT_PATH = "tx_temp_demand_merged.csv"
force_refresh = True

_centroid = shape(polygon_aoi["features"][0]["geometry"]).centroid
CENTROID_LAT, CENTROID_LON = _centroid.y, _centroid.x


def fetch_daily_avg_temperature(client, polygon_aoi: dict, date: datetime) -> tuple[float, float]:
    """Returns (avg_temp_f, avg_temp_c) - the Celsius value is now also
    needed as the required `temperature=` anchor for environmental_parameters."""
    response = client.create_heatmap(
        polygon_aoi=polygon_aoi, start_date=date.strftime("%Y-%m-%d"),
        filter_type=3, granularity=100,
    )
    mean_c = response["result"]["stats_data"]["temperature_stats"]["mean"]
    return mean_c * 9 / 5 + 32, mean_c


def fetch_solar_irradiance(client, lat: float, lon: float, date: datetime, temperature_c: float, hour: int = 15):
    """FIX: environmental_parameters() requires `temperature` (Celsius) as
    input - reuses the mean_c already fetched by fetch_daily_avg_temperature
    for this AOI/day instead of paying for a second heatmap call."""
    try:
        resp = client.environmental_parameters(
            latitude=lat, longitude=lon, temperature=temperature_c,
            start_date=date.strftime("%Y-%m-%d"), start_time=f"{hour:02d}:00",
            filter_type=1,
        )
        locations = resp["result"].get("locations") or []
        if not locations:
            return None
        params = locations[0].get("parameters", {})
        for key in ("solar_irradiance_wm2", "solar_irradiance", "irradiance_wm2", "ghi_wm2", "ghi", "solar"):
            if key in params:
                return params[key]
        clear_sky = locations[0].get("solar_irradiance", {}).get("clear_sky", {})
        return clear_sky.get("ghi")
    except Exception as e:
        print(f"    [solar] unavailable for {date.date()} ({e})")
        return None

def build_historical_temperature(client, polygon_aoi, start, n_days, output_path="tx_daily_temperature.csv"):
    """Fetch n_days of temperature + solar irradiance starting from `start`,
    APPENDING to output_path if it already exists - so a second run on
    another day (once the daily FortyGuard quota resets) extends the
    dataset instead of re-fetching days you already paid for."""
    MIN_DELAY_SECONDS = 6

    existing = None
    existing_dates = set()
    if os.path.exists(output_path):
        existing = pd.read_csv(output_path, parse_dates=["date"])
        existing_dates = set(existing["date"].dt.date)
        print(f"Found {len(existing_dates)} existing day(s) already saved - will skip those.")

    rows = []
    for i in range(n_days):
        date = start + timedelta(days=i)
        if date.date() in existing_dates:
            print(f"Skipping {date.date()} (already fetched)")
            continue
        print(f"Fetching FortyGuard heatmap for {date.date()}... ({i + 1}/{n_days})")
        avg_temp, avg_temp_c = fetch_daily_avg_temperature(client, polygon_aoi, date)
        time.sleep(MIN_DELAY_SECONDS)
        solar_wm2 = fetch_solar_irradiance(client, CENTROID_LAT, CENTROID_LON, date, temperature_c=avg_temp_c)
        rows.append({"date": date.date(), "avg_temp_f": avg_temp, "solar_irradiance_wm2": solar_wm2})
        time.sleep(MIN_DELAY_SECONDS)

    new_df = pd.DataFrame(rows)
    if existing is not None:
        combined = pd.concat([existing, new_df], ignore_index=True) if not new_df.empty else existing
    else:
        combined = new_df
    combined["date"] = pd.to_datetime(combined["date"])  # normalize dtype every time, not just when new_df is non-empty
    combined = combined.drop_duplicates(subset="date").sort_values("date").reset_index(drop=True)
    combined.to_csv(output_path, index=False)
    return combined


def build_daily_demand(csv_path: str = "ercot_demand_clean.csv") -> pd.DataFrame:
    """Aggregate ERCOT_Total (EIA-930, system-wide) to daily mean/peak.
    No weather_zone parameter anymore - EIA-930 has one demand column."""
    df = pd.read_csv(csv_path, parse_dates=["Time"])
    demand_col = "ERCOT_Total" if "ERCOT_Total" in df.columns else \
        [c for c in df.columns if c not in ("Time", "Time_UTC")][0]
    df["date"] = df["Time"].dt.date
    return (
        df.groupby("date")[demand_col]
        .agg(demand_mean_mw="mean", demand_peak_mw="max")
        .reset_index()
    )


if os.path.exists(MERGED_OUTPUT_PATH) and not force_refresh:
    print(f"{MERGED_OUTPUT_PATH} already exists - loading cached copy "
          f"(set force_refresh=True above to resume/redo the fetch).")
    merged = pd.read_csv(MERGED_OUTPUT_PATH, parse_dates=["date"])
else:
    START_DATE = datetime(2025, 6, 1)
    N_DAYS = 120

    temp_df = build_historical_temperature(client, polygon_aoi, START_DATE, N_DAYS)
    demand_df = build_daily_demand()

    temp_df["date_only"] = pd.to_datetime(temp_df["date"]).dt.date
    merged = temp_df.merge(demand_df, left_on="date_only", right_on="date", how="inner", suffixes=("", "_demand"))
    merged = merged[["date", "avg_temp_f", "solar_irradiance_wm2", "demand_mean_mw", "demand_peak_mw"]]

    print(f"Merged rows: {len(merged)} / {N_DAYS} expected")
    merged.to_csv(MERGED_OUTPUT_PATH, index=False)
    print(f"Saved to {MERGED_OUTPUT_PATH}")

print(merged.describe())


Found 90 existing day(s) already saved - will skip those.
Fetching FortyGuard heatmap for 2025-06-01... (1/120)
Submitted -> activity_id=8665f2f0-2e94-477a-af42-376a83a1029e
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: completed
Done.
Submitted -> activity_id=05053805-4e66-457c-8ca7-e049997ae720
  status: processing
  status: processing
  status: completed
Done.
Fetching FortyGuard heatmap for 2025-06-02... (2/120)
Submitted -> activity_id=361ce7d0-2ab2-4f40-9979-471a6777b59a
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: completed
Done.
Submitted -> activity_id=6aa7ea78-34cf-46fa-a24f-7aa46f3af084
  status: processing
  status: completed
Done.

In [ ]:
usage = client.fetch_api_key_usage()
print(usage)

{'api_key': None, 'subscription_id': 'sub_o80e437oax', 'plan_details': {'plan_type': 'Hackathon', 'cycle_type': 'Hackathon', 'subscription_start_date': 'Aug 28, 2026', 'billing_period': 'Aug 28, 2026 – Oct 02, 2026', 'active': True, 'credits_reset_date': 'Oct 02, 2026'}, 'api_key_details': {'status': 'active', 'valid': True, 'expiry_date': '2026-10-02T10:30:48.970317Z', 'api_access_available': True}, 'credit_summary': {'total_available_credits': 2000000, 'cycle_credits_used': 1045060, 'cycle_remaining_credits': 954940, 'cycle_usage_percentage': 52.3, 'total_credits_used': 1045060, 'total_remaining_credits': 954940}, 'activity_breakdown': [{'name': 'Heatmap Generation', 'credits': 624560, 'count': 148, 'percentage': 31.23}, {'name': 'Environment Parameter Analysis', 'credits': 420500, 'count': 145, 'percentage': 21.02}, {'name': 'Unused Credits', 'credits': 954940, 'count': 0, 'percentage': 47.75}], 'total_credits_used': 1045060, 'billing_cycle': {'start_date': '2026-08-28T10:30:48.9705

### 4.1 Diagnosing the temperature-demand relationship

Before trusting whatever R² a single train/test split gives, rule out the same
three things checked for Fresno: too little variance in the input, a merge bug, or
comparing against the wrong target. Run the cell below and see which applies -
don't assume the Fresno finding (narrow within-month temperature range, but a real
r≈0.76 once weather-zone demand was used as the target) repeats here; Texas summer
temperature variance and weather-zone demand behavior may look completely
different.

In [ ]:
"""
[TX-4.1] Diagnose the pilot-AOI-temp -> ERCOT-system-demand relationship.

CHANGE FROM ORIGINAL: the original version cross-checked correlation
against each of ERCOT's 8 weather-zone columns as a sanity check. Since
Section 1.3 now sources demand from EIA-930 (a single ERCOT-wide
ERCOT_Total column, not per-zone), that cross-check no longer applies -
there's only one demand series to correlate against. The three core
diagnostic checks (X variance, raw correlation, eyeball table) are
unchanged.
"""
import pandas as pd

merged = pd.read_csv("tx_temp_demand_merged.csv", parse_dates=["date"])

print("=== Temperature (X) variance ===")
print(merged["avg_temp_f"].describe())
print(f"Range: {merged['avg_temp_f'].max() - merged['avg_temp_f'].min():.1f} F across the window.")

print("\n=== Demand (y) variance - ERCOT system-wide (EIA-930) ===")
print(merged["demand_peak_mw"].describe())

print("\n=== Raw correlation (Pearson r) ===")
corr = merged["avg_temp_f"].corr(merged["demand_peak_mw"])
print(f"corr(avg_temp_f, demand_peak_mw) = {corr:.3f}")

print("\n=== Full data table (eyeball check) ===")
print(merged[["date", "avg_temp_f", "demand_peak_mw"]].sort_values("date").to_string(index=False))

print("\nNOTE: this correlation is against ERCOT's ENTIRE system demand, not "
      "a weather-zone or county-specific figure (see Section 1.3 discussion) - "
      "expect a weaker/noisier relationship than a truly local demand series "
      "would show, since statewide demand mixes in weather patterns from "
      "outside the pilot AOI entirely.")


=== Temperature (X) variance ===
count    120.000000
mean      85.314718
std        2.655319
min       77.318062
25%       83.932361
50%       85.665879
75%       87.216975
max       89.379759
Name: avg_temp_f, dtype: float64
Range: 12.1 F across the window.

=== Demand (y) variance - ERCOT system-wide (EIA-930) ===
count      120.000000
mean     76817.116667
std       4016.072052
min      63517.000000
25%      75436.750000
50%      76824.000000
75%      79716.250000
max      83597.000000
Name: demand_peak_mw, dtype: float64

=== Raw correlation (Pearson r) ===
corr(avg_temp_f, demand_peak_mw) = 0.704

=== Full data table (eyeball check) ===
      date  avg_temp_f  demand_peak_mw
2025-06-01   83.976521           72334
2025-06-02   85.481176           76162
2025-06-03   83.897395           74753
2025-06-04   83.937057           73302
2025-06-05   85.769433           75902
2025-06-06   87.270957           75912
2025-06-07   87.806903           75710
2025-06-08   88.146898           76397

### 4.1b — Feature engineering (calendar, lag, holiday)

Same reasoning as the Fresno/California version's Section 8b — see the code cell below for the full decision log on what was added vs. deliberately left out.


In [ ]:
"""
[TX-4.1b] Feature engineering - calendar, lag, and holiday features.

Pure pandas transform of tx_temp_demand_merged.csv - no API calls, always
safe to re-run. Kept as its own cell/file (tx_demand_features.csv) rather
than overwriting the raw merge, so Section 4.1's diagnostic cell keeps
reading the untouched raw temp+demand series.

Added: previous-day temp, previous-day demand, 3-day rolling demand,
day-of-week / weekend / month, and a holiday flag. Same reasoning as the
Fresno version's Section 8b.

Deliberately NOT added, and why (same evaluation as Fresno):
  - Population: constant across this single system-wide series - no
    day-to-day variation for a regression to learn from.
  - Wind / precipitation: not exposed by any documented FortyGuard
    endpoint.
  - Hour of day: the target here is the *daily peak* (demand_peak_mw), one
    value per day - hour-of-day needs an hourly target redesign, not a
    feature add. (Also out of scope for this section - it belongs with the
    Risk Engine if it's ever added, as a per-zone timing signal, the way
    it was added on the California/Fresno side - not touched here.)
"""
import pandas as pd

FEATURES_OUTPUT_PATH = "tx_demand_features.csv"

merged = pd.read_csv("tx_temp_demand_merged.csv", parse_dates=["date"])
merged = merged.sort_values("date").reset_index(drop=True)

merged["dow"]                = merged["date"].dt.dayofweek
merged["is_weekend"]         = (merged["dow"] >= 5).astype(int)
merged["month"]              = merged["date"].dt.month
merged["prev_day_demand_mw"] = merged["demand_peak_mw"].shift(1)
merged["prev_temp_f"]        = merged["avg_temp_f"].shift(1)
merged["rolling_3d_demand_mw"] = (
    merged["demand_peak_mw"].shift(1).rolling(3, min_periods=1).mean()
)

try:
    import holidays as holidays_lib
    us_holidays = holidays_lib.US(years=sorted(merged["date"].dt.year.unique().tolist()))
except ImportError:
    print("NOTE: `holidays` package not installed - using a minimal hardcoded fallback "
          "(add it to Section 0's requirements.txt / pip install for full coverage).")
    years = merged["date"].dt.year.unique().tolist()
    us_holidays = {pd.Timestamp(f"{y}-07-04").date() for y in years} | \
                  {pd.Timestamp(f"{y}-01-01").date() for y in years} | \
                  {pd.Timestamp(f"{y}-12-25").date() for y in years}
merged["is_holiday"] = merged["date"].dt.date.apply(lambda d: int(d in us_holidays))

SOLAR_AVAILABLE = "solar_irradiance_wm2" in merged.columns and merged["solar_irradiance_wm2"].notna().any()
if not SOLAR_AVAILABLE:
    print("NOTE: no usable solar_irradiance_wm2 values in tx_temp_demand_merged.csv "
          "(check the [solar] warnings from the fetch cell above) - the multivariate "
          "model in Section 4.2 will skip this feature.")

merged.to_csv(FEATURES_OUTPUT_PATH, index=False)
print(f"Saved feature-engineered dataset to {FEATURES_OUTPUT_PATH} "
      f"({merged.shape[0]} rows, {merged.shape[1]} columns)")
print(merged[["date","dow","is_weekend","month","is_holiday",
              "prev_day_demand_mw","prev_temp_f","rolling_3d_demand_mw"]].tail(5).to_string(index=False))


Saved feature-engineered dataset to tx_demand_features.csv (120 rows, 12 columns)
      date  dow  is_weekend  month  is_holiday  prev_day_demand_mw  prev_temp_f  rolling_3d_demand_mw
2025-09-24    2           0      9           0             79113.0    85.362559          78126.000000
2025-09-25    3           0      9           0             77934.0    85.453055          78636.666667
2025-09-26    4           0      9           0             71868.0    83.211000          76305.000000
2025-09-27    5           1      9           0             72230.0    80.129498          74010.666667
2025-09-28    6           1      9           0             70961.0    78.296035          71686.333333


### 4.2 Model selection via cross-validation

Two comparisons: the univariate baseline (avg_temp_f only, kept as the
live-prediction fallback) and the phase-2 multivariate model (calendar +
lag + rolling + holiday [+ solar irradiance if fetchable]) built on
Section 4.1b's feature-engineered dataset. Both use Leave-One-Out CV
(<20 rows) or 5-Fold CV given the small sample - never a single
train/test split. Whichever family wins each comparison may or may not be
linear regression - that was a Fresno-specific result, not a property of
this notebook.


In [ ]:
"""
[TX-4.2] Model comparison via cross-validation.

Two comparisons, kept side by side:
  1. Univariate (avg_temp_f only) - identical logic to the Fresno version,
     kept as the safety-net fallback model for live prediction.
  2. Multivariate (calendar + lag + rolling + holiday [+ solar]) - the
     phase-2 model, evaluated the same way (LOO/K-fold CV, not a single
     split) given the small sample.

Whichever wins in each comparison may or may not be linear regression -
that was a Fresno-specific result, not a property of this notebook.
"""
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, LeaveOneOut, cross_val_score, cross_val_predict
from sklearn.metrics import mean_absolute_error

# ── 1. Univariate baseline (unchanged) ──
merged = pd.read_csv("tx_temp_demand_merged.csv", parse_dates=["date"])
X = merged[["avg_temp_f"]]
y = merged["demand_peak_mw"]

cv = LeaveOneOut() if len(merged) < 20 else KFold(n_splits=5, shuffle=True, random_state=42)
cv_label = "LOO-CV" if len(merged) < 20 else "5-Fold CV"
print(f"Univariate: using {cv_label} (n={len(merged)})\n")

univariate_models = {
    "linear": LinearRegression(),
    "poly2": make_pipeline(PolynomialFeatures(degree=2, include_bias=False), LinearRegression()),
    "gbr": GradientBoostingRegressor(random_state=42, n_estimators=20, max_depth=2),
}

results = []
for name, model in univariate_models.items():
    scores = cross_val_score(model, X, y, cv=cv, scoring="r2")
    preds = cross_val_predict(model, X, y, cv=cv)
    mae = mean_absolute_error(y, preds)
    results.append({"model": name, "cv_r2_mean": scores.mean(), "cv_r2_std": scores.std(), "pooled_mae_mw": mae})

results_df = pd.DataFrame(results).sort_values("cv_r2_mean", ascending=False).reset_index(drop=True)
print(results_df.to_string(index=False))

best_uni_name = results_df.iloc[0]["model"]
baseline_r2 = results_df[results_df["model"] == "linear"]["cv_r2_mean"].iloc[0]
print(f"\nBest univariate by CV R^2: {best_uni_name}")
print("\n>>> Fill in Section 4.2's markdown above with the actual winner and its "
      "CV R^2 once you've run this, the same way the Fresno version documents "
      "linear/0.316 as an after-the-fact finding, not an assumption.")

# ── 2. Multivariate (phase 2) ──
features_df = pd.read_csv("tx_demand_features.csv", parse_dates=["date"])

FEATURES_MULTI = ["avg_temp_f", "dow", "is_weekend", "month",
                   "prev_day_demand_mw", "prev_temp_f", "rolling_3d_demand_mw", "is_holiday"]
if "solar_irradiance_wm2" in features_df.columns and features_df["solar_irradiance_wm2"].notna().any():
    FEATURES_MULTI += ["solar_irradiance_wm2"]

model_data = features_df.dropna(subset=FEATURES_MULTI + ["demand_peak_mw"]).copy()
n_rows, n_feats = len(model_data), len(FEATURES_MULTI)
print(f"\n\nMultivariate: {n_rows} usable rows, {n_feats} features")
if n_rows < 10 * n_feats:
    print(f"⚠️ Only ~{n_rows / n_feats:.1f} rows per feature (rule of thumb wants ≥10). "
          f"Trust the CV R² below over any single-fit number, and prefer collecting "
          f"more historical days before adding further features.")

X_multi = model_data[FEATURES_MULTI]
y_multi = model_data["demand_peak_mw"]
cv_multi = LeaveOneOut() if n_rows < 20 else KFold(n_splits=5, shuffle=True, random_state=42)
cv_multi_label = "LOO-CV" if n_rows < 20 else "5-Fold CV"
print(f"Using {cv_multi_label} (n={n_rows})\n")

multivariate_models = {
    "linear_multi": LinearRegression(),
    "gbr_multi": GradientBoostingRegressor(random_state=42, n_estimators=20, max_depth=2),
}

multi_results = []
for name, model in multivariate_models.items():
    scores = cross_val_score(model, X_multi, y_multi, cv=cv_multi, scoring="r2")
    preds = cross_val_predict(model, X_multi, y_multi, cv=cv_multi)
    mae = mean_absolute_error(y_multi, preds)
    multi_results.append({"model": name, "cv_r2_mean": scores.mean(), "cv_r2_std": scores.std(), "pooled_mae_mw": mae})

multi_results_df = pd.DataFrame(multi_results).sort_values("cv_r2_mean", ascending=False).reset_index(drop=True)
print(multi_results_df.to_string(index=False))

best_multi_name = multi_results_df.iloc[0]["model"]
best_multi_r2 = multi_results_df.iloc[0]["cv_r2_mean"]
print(f"\nBest multivariate by CV R^2: {best_multi_name} (R²={best_multi_r2:.3f}) "
      f"vs. univariate baseline (linear, R²={baseline_r2:.3f})")
if best_multi_r2 <= max(baseline_r2, 0):
    print("⚠️ The multivariate model did NOT clearly beat the temp-only baseline on this "
          "sample size. Section 4-FINAL still saves it for comparison and prefers it at "
          "live-prediction time when its features are available, but treat its output "
          "with caution until more days of data accumulate and this gap closes.")


Univariate: using 5-Fold CV (n=120)

 model  cv_r2_mean  cv_r2_std  pooled_mae_mw
linear    0.442420   0.235793    2233.608518
 poly2    0.434886   0.245547    2264.590020
   gbr    0.373268   0.172693    2400.203765

Best univariate by CV R^2: linear

>>> Fill in Section 4.2's markdown above with the actual winner and its CV R^2 once you've run this, the same way the Fresno version documents linear/0.316 as an after-the-fact finding, not an assumption.


Multivariate: 117 usable rows, 9 features
Using 5-Fold CV (n=117)

       model  cv_r2_mean  cv_r2_std  pooled_mae_mw
linear_multi    0.703974   0.062808    1517.075648
   gbr_multi    0.585814   0.088010    1762.177001

Best multivariate by CV R^2: linear_multi (R²=0.704) vs. univariate baseline (linear, R²=0.442)


In [ ]:
"""
[TX-4 - FINAL, CACHED] Train and persist BOTH the univariate (fallback) and
multivariate (phase-2) demand models as a single bundle for use by the risk
engine and LangGraph workflow.

CHANGE FROM ORIGINAL: previously saved one bare sklearn model
(tx_demand_model.joblib = a fitted estimator). Now saves a dict bundle:
    {"linear": <fitted univariate model>,
     "multivariate": <fitted multivariate model>,
     "features": [<feature names, in order>]}
`demand_forecast_agent` in Section 7 loads this bundle and prefers
"multivariate" whenever all its live features are fetchable, falling back
to "linear" (avg_temp_f only) otherwise - see predict_demand_stress()
there for the fallback logic.

MODEL SELECTION (filled in after running Section 4.2 on 120-day data,
n=120 univariate / n=117 multivariate, 5-fold CV):
- Univariate winner: linear (CV R^2 = 0.442, +/- 0.236) - matches the
  default LinearRegression() below, no change needed.
- Multivariate winner: linear_multi (CV R^2 = 0.704, +/- 0.063) - CHANGED
  from GradientBoostingRegressor to LinearRegression to match. Note the
  much tighter std (0.063 vs. the earlier 31-day run's 0.206) now that
  n has grown from 28 to 117 rows.

CACHING: if tx_demand_model.joblib already exists, this cell loads it
instead of re-fitting. Set force_refresh=True (or delete the .joblib
file) to retrain.
"""
import os
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, LeaveOneOut, cross_val_score, cross_val_predict
from sklearn.metrics import mean_absolute_error
import joblib

MODEL_OUTPUT_PATH = "tx_demand_model.joblib"
force_refresh = True  # retrain - the 120-day data + model choice both changed

merged = pd.read_csv("tx_temp_demand_merged.csv", parse_dates=["date"])
print(f"Training univariate model on {len(merged)} days")

# Baseline used by the demand forecast agent to compute
# demand_stress_pct = (predicted - baseline) / baseline * 100.
baseline_demand_mw = merged["demand_peak_mw"].mean()

if os.path.exists(MODEL_OUTPUT_PATH) and not force_refresh:
    print(f"{MODEL_OUTPUT_PATH} already exists - loading cached bundle instead of retraining.")
    demand_bundle = joblib.load(MODEL_OUTPUT_PATH)
else:
    # ── Univariate (fallback) - winner: linear (see note above) ──
    X = merged[["avg_temp_f"]]
    y = merged["demand_peak_mw"]
    cv = LeaveOneOut() if len(merged) < 20 else KFold(n_splits=5, shuffle=True, random_state=42)

    linear_model = LinearRegression()
    scores = cross_val_score(linear_model, X, y, cv=cv, scoring="r2")
    preds = cross_val_predict(linear_model, X, y, cv=cv)
    print(f"[univariate] CV R^2 = {scores.mean():.3f} (+/- {scores.std():.3f}) | "
          f"pooled MAE = {mean_absolute_error(y, preds):.1f} MW")
    linear_model.fit(X, y)

    # ── Multivariate (phase 2) - winner: linear_multi (see note above) ──
    features_df = pd.read_csv("tx_demand_features.csv", parse_dates=["date"])
    FEATURES_MULTI = ["avg_temp_f", "dow", "is_weekend", "month",
                       "prev_day_demand_mw", "prev_temp_f", "rolling_3d_demand_mw", "is_holiday"]
    if "solar_irradiance_wm2" in features_df.columns and features_df["solar_irradiance_wm2"].notna().any():
        FEATURES_MULTI += ["solar_irradiance_wm2"]

    model_data = features_df.dropna(subset=FEATURES_MULTI + ["demand_peak_mw"]).copy()
    X_multi = model_data[FEATURES_MULTI]
    y_multi = model_data["demand_peak_mw"]
    cv_multi = LeaveOneOut() if len(model_data) < 20 else KFold(n_splits=5, shuffle=True, random_state=42)

    multi_model = LinearRegression()  # was GradientBoostingRegressor - linear_multi won CV (see note above)
    scores_multi = cross_val_score(multi_model, X_multi, y_multi, cv=cv_multi, scoring="r2")
    preds_multi = cross_val_predict(multi_model, X_multi, y_multi, cv=cv_multi)
    print(f"[multivariate] CV R^2 = {scores_multi.mean():.3f} (+/- {scores_multi.std():.3f}) | "
          f"pooled MAE = {mean_absolute_error(y_multi, preds_multi):.1f} MW "
          f"(n={len(model_data)}, features={FEATURES_MULTI})")
    multi_model.fit(X_multi, y_multi)

    demand_bundle = {"linear": linear_model, "multivariate": multi_model, "features": FEATURES_MULTI}
    joblib.dump(demand_bundle, MODEL_OUTPUT_PATH)
    print(f"Saved bundle (linear + multivariate + feature list) to {MODEL_OUTPUT_PATH}")

print(f"Baseline peak demand (ERCOT system-wide, sampled window): {baseline_demand_mw:.1f} MW")

Training univariate model on 120 days
[univariate] CV R^2 = 0.442 (+/- 0.236) | pooled MAE = 2233.6 MW
[multivariate] CV R^2 = 0.704 (+/- 0.063) | pooled MAE = 1517.1 MW (n=117, features=['avg_temp_f', 'dow', 'is_weekend', 'month', 'prev_day_demand_mw', 'prev_temp_f', 'rolling_3d_demand_mw', 'is_holiday', 'solar_irradiance_wm2'])
Saved bundle (linear + multivariate + feature list) to tx_demand_model.joblib
Baseline peak demand (ERCOT system-wide, sampled window): 76817.1 MW


## 5. Live Heat Integration

Identical logic to the California version - pulls the current (most recent
complete day's) heat over the pilot AOI and joins it onto the canonical
`zone_id` grid, producing the `live_heat` table that feeds the risk engine
(Section 6). Same caching-per-target-date behavior, same reason `live_heat`
stays a separate table rather than a column on `grid` itself. Only the
AOI/grid file paths change.

In [ ]:
"""
[TX-5, CACHED-PER-DAY] Pull CURRENT heat over the pilot AOI and join it
onto the canonical heat_zone grid - feeds `live_heat` into the risk engine
(Section 6), NOT a column on `grid` itself.

CACHING: cached PER TARGET DATE, not forever - if tx_live_heat_by_zone.csv
already exists AND its as_of_date matches today's target date, this cell
loads the cached file; otherwise it re-fetches. Set force_refresh=True to
always re-fetch.
"""
import json
import os
from datetime import date, timedelta

import geopandas as gpd
import pandas as pd
from shapely.geometry import shape

OUTPUT_PATH = "tx_live_heat_by_zone.csv"
force_refresh = False


def tiles_to_geodataframe(map_data: dict) -> gpd.GeoDataFrame:
    rows = []
    for i, feature in enumerate(map_data["features"]):
        props = feature["properties"]
        geom = shape(feature["geometry"])
        rows.append({**props, "tile_id": i, "geometry": geom})
    return gpd.GeoDataFrame(rows, geometry="geometry", crs="EPSG:4326")


def join_polygons_area_weighted(grid, polygons_gdf, id_col, extensive_cols=None, intensive_cols=None):
    """Same fixed primitive used in Section 3.2, reused here for tiles."""
    polygons_gdf = polygons_gdf.to_crs("EPSG:4326")
    extensive_cols = extensive_cols or []
    intensive_cols = intensive_cols or []

    grid_m = grid.to_crs("EPSG:3083")  # Texas Centric Albers Equal Area
    polys_m = polygons_gdf.to_crs("EPSG:3083")
    overlay = gpd.overlay(grid_m, polys_m, how="intersection")
    overlay["piece_area"] = overlay.geometry.area

    results = []
    if extensive_cols:
        poly_total_area = polys_m.set_index(id_col).geometry.area
        overlay["poly_total_area"] = overlay[id_col].map(poly_total_area)
        overlay["area_fraction_of_source"] = overlay["piece_area"] / overlay["poly_total_area"]
        ext = overlay.copy()
        for col in extensive_cols:
            ext[col] = ext[col] * ext["area_fraction_of_source"]
        results.append(ext.groupby("zone_id")[extensive_cols].sum())

    if intensive_cols:
        cell_covered_area = overlay.groupby("zone_id")["piece_area"].transform("sum")
        overlay["area_fraction_of_cell"] = overlay["piece_area"] / cell_covered_area
        intens = overlay.copy()
        for col in intensive_cols:
            intens[col] = intens[col] * intens["area_fraction_of_cell"]
        results.append(intens.groupby("zone_id")[intensive_cols].sum())

    if not results:
        raise ValueError("Must supply at least one of extensive_cols or intensive_cols")
    return pd.concat(results, axis=1).reset_index()


def fetch_live_heat_by_zone(client, polygon_aoi: dict, grid: gpd.GeoDataFrame,
                             target_date: str | None = None) -> pd.DataFrame:
    """FortyGuard heatmaps are for a completed day, not instantaneous -
    'live' here means 'most recent day with data' (defaults to yesterday)."""
    if target_date is None:
        target_date = (date.today() - timedelta(days=1)).strftime("%Y-%m-%d")

    response = client.create_heatmap(
        polygon_aoi=polygon_aoi, start_date=target_date, filter_type=3, granularity=100,
    )
    tiles_gdf = tiles_to_geodataframe(response["result"]["map_data"])

    heat_by_zone = join_polygons_area_weighted(
        grid, tiles_gdf, id_col="tile_id", intensive_cols=["average_temperature"],
    )
    heat_by_zone = heat_by_zone.rename(columns={"average_temperature": "heat_raw_c"})
    heat_by_zone["heat_raw_f"] = heat_by_zone["heat_raw_c"] * 9 / 5 + 32
    heat_by_zone["as_of_date"] = target_date
    return heat_by_zone


target_date = (date.today() - timedelta(days=1)).strftime("%Y-%m-%d")

cached_matches_today = False
if os.path.exists(OUTPUT_PATH) and not force_refresh:
    _cached = pd.read_csv(OUTPUT_PATH)
    if not _cached.empty and str(_cached["as_of_date"].iloc[0]) == target_date:
        cached_matches_today = True

if cached_matches_today:
    print(f"{OUTPUT_PATH} already has data for {target_date} - loading cached "
          f"copy instead of calling the FortyGuard API again "
          f"(set force_refresh=True above to force a re-fetch).")
    live_heat = _cached
    print(f"Loaded live heat for {len(live_heat)} zones (as of {target_date})")
else:
    from google.colab import userdata
    from fortyguard import FortyGuardClient
    fg_client = FortyGuardClient(api_key=userdata.get("FORTYGUARD_API_KEY"))
    with open(f"{PILOT_COUNTY.lower().replace(' ', '_')}_aoi_precise.geojson") as f:
        pilot_aoi = json.load(f)


    grid_gdf = gpd.read_file("heat_zone_grid_tx.geojson")

    print(fg_client.fetch_api_key_usage())

    live_heat = fetch_live_heat_by_zone(fg_client, pilot_aoi, grid_gdf, target_date=target_date)
    print(f"Live heat joined to {len(live_heat)} / {len(grid_gdf)} zones")
    live_heat.to_csv(OUTPUT_PATH, index=False)
    print(f"Saved to {OUTPUT_PATH}")


{'api_key': None, 'subscription_id': 'sub_o80e437oax', 'plan_details': {'plan_type': 'Hackathon', 'cycle_type': 'Hackathon', 'subscription_start_date': 'Aug 28, 2026', 'billing_period': 'Aug 28, 2026 – Oct 02, 2026', 'active': True, 'credits_reset_date': 'Oct 02, 2026'}, 'api_key_details': {'status': 'active', 'valid': True, 'expiry_date': '2026-10-02T10:30:48.970317Z', 'api_access_available': True}, 'credit_summary': {'total_available_credits': 2000000, 'cycle_credits_used': 1045060, 'cycle_remaining_credits': 954940, 'cycle_usage_percentage': 52.3, 'total_credits_used': 1045060, 'total_remaining_credits': 954940}, 'activity_breakdown': [{'name': 'Heatmap Generation', 'credits': 624560, 'count': 148, 'percentage': 31.23}, {'name': 'Environment Parameter Analysis', 'credits': 420500, 'count': 145, 'percentage': 21.02}, {'name': 'Unused Credits', 'credits': 954940, 'count': 0, 'percentage': 47.75}], 'total_credits_used': 1045060, 'billing_cycle': {'start_date': '2026-08-28T10:30:48.9705

## 6. Deterministic Grid Heat Risk Engine

Same five-component design as the California version - **heat** (30%), **demand
stress** (25%), **infrastructure density** (20%), **historical outage exposure**
(15%), **vulnerability** (10%) - min-max normalized to 0-100 before weighting, for
the same reason as before (raw units differ wildly - °F vs. line-count vs. an
EJScreen index - so weighting raw values directly would distort the intended
split).

**The two California bugs still apply here and stay fixed the same way:**

1. **Wrong heat source** - live temperature still has to come from the separate
   `live_heat` table (Section 5), not a column on `grid` itself.
2. **Demand score always zero** - `demand_stress_pct` is still a single
   weather-zone-level number, identical for every zone in this AOI by construction
   (all zones are inside the same county/weather-zone). Still scored against the
   real historical spread of daily stress % from Section 4's training window, not
   min-max'd across zones.

**One NEW wrinkle for Texas, not present in the California version:** the outage
component has the *same* zero-variance problem demand always had, for a different
reason - EAGLE-I's `total_customers_deenergized` is a single county-level number
broadcast to every zone (Section 3.2), not per-event geometry like PSPS was. So
`outage_score` is now scored the same way `demand_score` is - against a real
historical reference range (the spread of the county's cumulative summer outage
totals against other candidate counties from Section 2's Stage B ranking) - rather
than min-max'd across zones, which would silently give 0 here too. This is the
same category of bug as #2 above, just triggered by a different Texas-specific
data-resolution ceiling, so it's caught before it ships rather than after.

A zone with no live_heat reading is still filled with the mean of the zones that
do have one, same as before.

In [ ]:
"""
[TX-6] Risk engine - five-component design, with the demand-score fix
carried over from California, the outage-score fix for Texas's
county-level EAGLE-I ceiling, AND a new spatial-differentiation step for
demand.

CHANGE FROM ORIGINAL: demand_score and outage_score are still computed as
a SINGLE system/county-level number (EIA-930 has no zone breakdown;
EAGLE-I has no per-event geometry - see Section 1.3 / 3.2 notes) - that
part is unchanged. What IS new: instead of broadcasting that single
demand_score identically to every zone (which erases any spatial signal
at all), each zone's demand_score is now scaled by its relative share of
built infrastructure (infra_weights_tx.csv from Section 1.3c - substation
+ transmission-line density, itself a proxy). Zones with more
infrastructure get pulled above the system-wide average score; zones with
none get pulled toward zero. The AOI-wide AVERAGE across all zones is
mathematically unchanged (still the same system-wide demand_score) - only
the spread across zones changes. This is a proxy layered on a proxy:
stated explicitly in the printed output, not just this comment, so it's
never mistaken for a real per-zone demand measurement. outage_score is
NOT reweighted this way, since infra density has no established
relationship to *outage history* the way it plausibly does to *load* -
it stays a flat broadcast value, same as before.
"""
import joblib
import pandas as pd
import numpy as np

WEIGHTS = {"heat": 0.30, "demand": 0.25, "infrastructure": 0.20, "outage": 0.15, "vulnerability": 0.10}
assert abs(sum(WEIGHTS.values()) - 1.0) < 1e-9


def normalize_0_100(series: pd.Series) -> pd.Series:
    """For columns that genuinely vary across zones (heat, infra, vuln)."""
    lo, hi = series.min(), series.max()
    if hi - lo == 0:
        return pd.Series(0.0, index=series.index)
    return (series - lo) / (hi - lo) * 100.0


def score_against_reference_range(value: float, reference_range: tuple) -> float:
    """For components that are a single number broadcast to every zone by
    construction (demand_stress_pct pre-reweighting, and Texas's
    county-level outage total) - score against a real historical/
    comparative reference range instead of min-max across zones, which
    always gives 0 when every zone has the identical value."""
    lo, hi = reference_range
    if hi - lo == 0:
        return 0.0
    score = (value - lo) / (hi - lo) * 100.0
    return float(np.clip(score, 0, 100))


def compute_demand_stress_pct(demand_bundle, baseline_demand_mw: float, live_heat: pd.DataFrame) -> float:
    """NOTE: as of Section 4-FINAL, tx_demand_model.joblib holds a bundle
    dict ({"linear", "multivariate", "features"}), not a bare model. This
    standalone smoke-test path deliberately always uses demand_bundle["linear"]
    (temp-only) rather than the multivariate model - the multivariate
    path needs live features (previous-day demand, previous-day temp,
    solar) that this simple test harness doesn't fetch. The real live run
    uses the multivariate model when it can; see predict_demand_stress()
    in Section 7's LangGraph cell for that full fallback logic."""
    current_temp_f = live_heat["heat_raw_f"].mean()
    predicted_mw = demand_bundle["linear"].predict(pd.DataFrame({"avg_temp_f": [current_temp_f]}))[0]
    demand_stress_pct = (predicted_mw - baseline_demand_mw) / baseline_demand_mw * 100
    print(f"Today's mean AOI temperature: {current_temp_f:.1f}\u00b0F")
    print(f"Predicted ERCOT system-wide peak demand (linear/temp-only smoke test): "
          f"{predicted_mw:.1f} MW (baseline: {baseline_demand_mw:.1f} MW)")
    print(f"Demand stress: {demand_stress_pct:+.1f}%")
    return demand_stress_pct


def apply_infra_reweighting(risk: pd.DataFrame, base_score: float, infra_weights: pd.DataFrame) -> pd.Series:
    """Redistribute a single broadcast score across zones proportionally
    to infra_weight (which sums to 1.0 across the AOI's zones). The
    multiplier is infra_weight * n_zones, so its average across zones is
    exactly 1.0 - meaning the AOI-wide average of the resulting column
    stays equal to base_score; only the spread across zones changes.
    Zones missing from infra_weights (e.g. zero infrastructure) get a
    multiplier of 0."""
    n_zones = risk["zone_id"].nunique()
    w = risk[["zone_id"]].merge(infra_weights[["zone_id", "infra_weight"]], on="zone_id", how="left")
    w["infra_weight"] = w["infra_weight"].fillna(0.0)
    multiplier = w["infra_weight"].to_numpy() * n_zones
    return pd.Series(np.clip(base_score * multiplier, 0, 100), index=risk.index)


def build_risk_table(
    grid: pd.DataFrame,
    live_heat: pd.DataFrame,
    vulnerability_joined: pd.DataFrame,
    lines_joined: pd.DataFrame,
    demand_score: float,       # system-wide value, redistributed across zones via infra_weight below
    outage_score: float,       # pre-computed, same value for every zone by design (Texas-specific)
    infra_weights: pd.DataFrame = None,  # from Section 1.3c - enables demand spatial redistribution
) -> pd.DataFrame:
    heat = live_heat[["zone_id", "heat_raw_f"]].rename(columns={"heat_raw_f": "heat_raw"})
    infra = lines_joined.groupby("zone_id").size().rename("infra_raw").reset_index()

    vuln = vulnerability_joined[["zone_id", "P_DEMOGIDX_5"]].rename(columns={"P_DEMOGIDX_5": "vuln_raw"})
    all_zones = grid[["zone_id"]].drop_duplicates()
    risk = (
        all_zones
        .merge(heat, on="zone_id", how="left")
        .merge(infra, on="zone_id", how="left")
        .merge(vuln, on="zone_id", how="left")
    )
    risk["infra_raw"] = risk["infra_raw"].astype(float).fillna(0.0)

    n_missing_heat = risk["heat_raw"].isna().sum()
    if n_missing_heat > 0:
        print(f"WARNING: {n_missing_heat} zones missing a live_heat reading - filled with mean.")
        risk["heat_raw"] = risk["heat_raw"].fillna(risk["heat_raw"].mean())

    risk["heat_score"] = normalize_0_100(risk["heat_raw"])

    if infra_weights is not None and not infra_weights.empty:
        risk["demand_score"] = apply_infra_reweighting(risk, demand_score, infra_weights)
        print(f"Demand score redistributed across zones via infra_weight - "
              f"AOI-wide average preserved at {demand_score:.1f}, "
              f"actual per-zone range: [{risk['demand_score'].min():.1f}, {risk['demand_score'].max():.1f}]")
    else:
        print("NOTE: no infra_weights supplied - demand_score broadcast identically to every zone (no spatial signal).")
        risk["demand_score"] = demand_score

    risk["infra_score"] = normalize_0_100(risk["infra_raw"])
    risk["outage_score"] = outage_score      # same value for every zone - intentional (Texas-specific, not reweighted)
    risk["vuln_score"] = normalize_0_100(risk["vuln_raw"])

    risk["grid_heat_risk"] = (
        risk["heat_score"] * WEIGHTS["heat"]
        + risk["demand_score"] * WEIGHTS["demand"]
        + risk["infra_score"] * WEIGHTS["infrastructure"]
        + risk["outage_score"] * WEIGHTS["outage"]
        + risk["vuln_score"] * WEIGHTS["vulnerability"]
    )
    return risk.sort_values("grid_heat_risk", ascending=False).reset_index(drop=True)


if __name__ == "__main__":
    import os
    import geopandas as gpd

    grid_gdf = gpd.read_file("heat_zone_grid_tx.geojson")
    live_heat = pd.read_csv("tx_live_heat_by_zone.csv")
    vulnerability_joined = pd.read_csv("vulnerability_joined_tx.csv")
    lines_joined = pd.read_csv("lines_joined_tx.csv")
    infra_weights = pd.read_csv("infra_weights_tx.csv") if os.path.exists("infra_weights_tx.csv") else None

    demand_bundle = joblib.load("tx_demand_model.joblib")  # {"linear","multivariate","features"} - see Section 4-FINAL
    merged = pd.read_csv("tx_temp_demand_merged.csv")
    baseline_demand_mw = merged["demand_peak_mw"].mean()

    demand_stress_pct = compute_demand_stress_pct(demand_bundle, baseline_demand_mw, live_heat)
    merged["stress_pct"] = (merged["demand_peak_mw"] - baseline_demand_mw) / baseline_demand_mw * 100
    demand_historical_range = (merged["stress_pct"].min(), merged["stress_pct"].max())
    demand_score = score_against_reference_range(demand_stress_pct, demand_historical_range)
    print(f"Demand score (0-100 scale, system-wide before spatial redistribution): {demand_score:.1f}")

    # Outage reference range: the spread of cumulative summer customers-out
    # totals ACROSS the candidate counties considered in Section 2's Stage
    # B (not just this one county, which by itself has no spread to
    # compare against) - this is the real "low vs high" scale for a
    # single county-level number, the same role historical_stress_range
    # plays for demand.
    outage_reference_range = (scoreboard["outages"].min(), scoreboard["outages"].max())
    pilot_outage_total = scoreboard.loc[PILOT_COUNTY, "outages"]
    outage_score = score_against_reference_range(pilot_outage_total, outage_reference_range)
    print(f"Outage score (0-100 scale): {outage_score:.1f}")

    risk_table = build_risk_table(
        grid_gdf, live_heat, vulnerability_joined, lines_joined,
        demand_score, outage_score, infra_weights=infra_weights,
    )
    print(f"\nTop 10 highest-risk zones:")
    print(risk_table[["zone_id", "grid_heat_risk", "heat_score", "demand_score",
                       "infra_score", "outage_score", "vuln_score"]].head(10))

    risk_table.to_csv("tx_risk_table.csv", index=False)
    print("\nSaved to tx_risk_table.csv")


Today's mean AOI temperature: 90.3°F
Predicted ERCOT system-wide peak demand (linear/temp-only smoke test): 82098.9 MW (baseline: 76817.1 MW)
Demand stress: +6.9%
Demand score (0-100 scale, system-wide before spatial redistribution): 92.5
Outage score (0-100 scale): 100.0
Demand score redistributed across zones via infra_weight - AOI-wide average preserved at 92.5, actual per-zone range: [0.0, 100.0]

Top 10 highest-risk zones:
  zone_id  grid_heat_risk  heat_score  demand_score  infra_score  \
0  Z00046       81.939369   93.552968    100.000000         40.0   
1  Z00038       81.282251   49.300748    100.000000        100.0   
2  Z00006       80.461906   87.143788    100.000000         30.0   
3  Z00014       78.444198   76.632116    100.000000         30.0   
4  Z00022       70.643995   49.056396    100.000000         30.0   
5  Z00030       67.375938   47.321462    100.000000         30.0   
6  Z00036       66.430504   30.070939    100.000000         40.0   
7  Z00037       61.87966

## 7. LangGraph Agentic Workflow

Same graph shape as the California version: `HeatAgent -> DemandForecastAgent ->
GridAssetAgent -> OutageAgent -> VulnerabilityAgent -> RiskEngine`, then a
conditional edge to `ScenarioGenerator` or `Monitor`. Same design choices carried
over unchanged (`risk_threshold` as runtime state, the three cheap local-join
agents recomputing per run, `HeatAgent`'s cache-reuse fix, everything inlined
rather than imported from non-existent modules).

**What's different for Texas:** `OutageAgent` no longer does a real spatial join -
EAGLE-I has no per-event geometry (Section 3.2's finding), so it just re-reads the
same broadcast county-level total every other cell uses. It's kept as its own
graph node rather than folded into `RiskEngine` purely for structural
symmetry with the other four data-gathering agents - a real deployment might
reasonably simplify this once the geometry ceiling is a known, permanent
constraint rather than an open question.

In [ ]:
"""
[TX-7] GridHeat AI - LangGraph workflow, Texas pilot. Same inlining
strategy as the California version (avoids importing from
fortyguard_live_integration / risk_engine, which don't exist as real
files in Colab).

CHANGE FROM ORIGINAL: added an optional `infra_weights` field to state,
carrying infra_weights_tx.csv (Section 1.3c) through to RiskEngine so
demand_score can be spatially redistributed across zones the same way
Section 6's standalone version now does - see build_risk_table below and
the Section 6 markdown for the full rationale.
"""
from typing import TypedDict
from datetime import date, datetime, timedelta

import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import shape
import requests
from langgraph.graph import StateGraph, START, END


class GridHeatState(TypedDict, total=False):
    risk_threshold: float
    target_date: str | None
    fg_client: object
    pilot_aoi: dict
    grid: object
    demand_model: object
    baseline_demand_mw: float
    demand_historical_range: tuple
    outage_reference_range: tuple
    pilot_county: str
    infra_weights: pd.DataFrame  # from Section 1.3c - optional, enables demand spatial redistribution

    live_heat: pd.DataFrame
    demand_stress_pct: float
    demand_score: float
    demand_model_used: str  # "multivariate" or "linear (fallback)" - set by demand_forecast_agent
    outage_score: float
    lines_joined: pd.DataFrame
    vulnerability_joined: pd.DataFrame
    risk_table: pd.DataFrame
    critical_zones: pd.DataFrame
    status: str


# ============ Shared join primitives ============

def join_lines(grid, lines_gdf, id_col="OBJECTID"):
    lines_gdf = lines_gdf.to_crs(grid.crs)
    joined = gpd.sjoin(grid, lines_gdf, how="inner", predicate="intersects")
    return joined[["zone_id", id_col]].drop_duplicates().reset_index(drop=True)


def join_polygons_area_weighted(grid, polygons_gdf, id_col, extensive_cols=None, intensive_cols=None):
    polygons_gdf = polygons_gdf.to_crs(grid.crs)
    extensive_cols = extensive_cols or []
    intensive_cols = intensive_cols or []
    grid_m = grid.to_crs("EPSG:3083")  # Texas Centric Albers Equal Area
    polys_m = polygons_gdf.to_crs("EPSG:3083")
    overlay = gpd.overlay(grid_m, polys_m, how="intersection")
    overlay["piece_area"] = overlay.geometry.area

    results = []
    if extensive_cols:
        poly_total_area = polys_m.set_index(id_col).geometry.area
        overlay["poly_total_area"] = overlay[id_col].map(poly_total_area)
        overlay["area_fraction_of_source"] = overlay["piece_area"] / overlay["poly_total_area"]
        ext = overlay.copy()
        for col in extensive_cols:
            ext[col] = ext[col] * ext["area_fraction_of_source"]
        results.append(ext.groupby("zone_id")[extensive_cols].sum())
    if intensive_cols:
        cell_covered_area = overlay.groupby("zone_id")["piece_area"].transform("sum")
        overlay["area_fraction_of_cell"] = overlay["piece_area"] / cell_covered_area
        intens = overlay.copy()
        for col in intensive_cols:
            intens[col] = intens[col] * intens["area_fraction_of_cell"]
        results.append(intens.groupby("zone_id")[intensive_cols].sum())
    if not results:
        raise ValueError("Must supply at least one of extensive_cols or intensive_cols")
    return pd.concat(results, axis=1).reset_index()


# ============ Inlined from Section 5 ============

def tiles_to_geodataframe(map_data: dict) -> gpd.GeoDataFrame:
    rows = []
    for i, feature in enumerate(map_data["features"]):
        props = feature["properties"]
        geom = shape(feature["geometry"])
        rows.append({**props, "tile_id": i, "geometry": geom})
    return gpd.GeoDataFrame(rows, geometry="geometry", crs="EPSG:4326")


def fetch_live_heat_by_zone(client, polygon_aoi: dict, grid: gpd.GeoDataFrame, target_date: str | None = None) -> pd.DataFrame:
    if target_date is None:
        target_date = (date.today() - timedelta(days=1)).strftime("%Y-%m-%d")

    response = client.create_heatmap(polygon_aoi=polygon_aoi, start_date=target_date, filter_type=3, granularity=100)
    tiles_gdf = tiles_to_geodataframe(response["result"]["map_data"])

    heat_by_zone = join_polygons_area_weighted(grid, tiles_gdf, id_col="tile_id", intensive_cols=["average_temperature"])
    heat_by_zone = heat_by_zone.rename(columns={"average_temperature": "heat_raw_c"})
    heat_by_zone["heat_raw_f"] = heat_by_zone["heat_raw_c"] * 9 / 5 + 32
    heat_by_zone["as_of_date"] = target_date
    return heat_by_zone


# ============ Inlined from Section 6 ============

WEIGHTS = {"heat": 0.30, "demand": 0.25, "infrastructure": 0.20, "outage": 0.15, "vulnerability": 0.10}


def normalize_0_100(series: pd.Series) -> pd.Series:
    lo, hi = series.min(), series.max()
    if hi - lo == 0:
        return pd.Series(0.0, index=series.index)
    return (series - lo) / (hi - lo) * 100.0


def score_against_reference_range(value: float, reference_range: tuple) -> float:
    lo, hi = reference_range
    if hi - lo == 0:
        return 0.0
    return float(np.clip((value - lo) / (hi - lo) * 100.0, 0, 100))


def apply_infra_reweighting(risk: pd.DataFrame, base_score: float, infra_weights: pd.DataFrame) -> pd.Series:
    """See Section 6 markdown - redistributes a single broadcast score
    across zones proportionally to infra_weight, preserving the AOI-wide
    average. Zones missing from infra_weights get a multiplier of 0."""
    n_zones = risk["zone_id"].nunique()
    w = risk[["zone_id"]].merge(infra_weights[["zone_id", "infra_weight"]], on="zone_id", how="left")
    w["infra_weight"] = w["infra_weight"].fillna(0.0)
    multiplier = w["infra_weight"].to_numpy() * n_zones
    return pd.Series(np.clip(base_score * multiplier, 0, 100), index=risk.index)


def build_risk_table(grid, live_heat, vulnerability_joined, lines_joined, demand_score: float,
                      outage_score: float, infra_weights: pd.DataFrame = None) -> pd.DataFrame:
    heat = live_heat[["zone_id", "heat_raw_f"]].rename(columns={"heat_raw_f": "heat_raw"})
    infra = lines_joined.groupby("zone_id").size().rename("infra_raw").reset_index()

    vuln = vulnerability_joined[["zone_id", "P_DEMOGIDX_5"]].rename(columns={"P_DEMOGIDX_5": "vuln_raw"})
    all_zones = grid[["zone_id"]].drop_duplicates()
    risk = (
        all_zones.merge(heat, on="zone_id", how="left")
        .merge(infra, on="zone_id", how="left")
        .merge(vuln, on="zone_id", how="left")
    )
    risk["infra_raw"] = risk["infra_raw"].astype(float).fillna(0.0)

    if risk["heat_raw"].isna().any():
        risk["heat_raw"] = risk["heat_raw"].fillna(risk["heat_raw"].mean())

    risk["heat_score"] = normalize_0_100(risk["heat_raw"])

    if infra_weights is not None and not infra_weights.empty:
        risk["demand_score"] = apply_infra_reweighting(risk, demand_score, infra_weights)
    else:
        risk["demand_score"] = demand_score

    risk["infra_score"] = normalize_0_100(risk["infra_raw"])
    risk["outage_score"] = outage_score
    risk["vuln_score"] = normalize_0_100(risk["vuln_raw"])

    risk["grid_heat_risk"] = (
        risk["heat_score"] * WEIGHTS["heat"]
        + risk["demand_score"] * WEIGHTS["demand"]
        + risk["infra_score"] * WEIGHTS["infrastructure"]
        + risk["outage_score"] * WEIGHTS["outage"]
        + risk["vuln_score"] * WEIGHTS["vulnerability"]
    )
    return risk.sort_values("grid_heat_risk", ascending=False).reset_index(drop=True)


# ============ Agent nodes ============

def heat_agent(state: GridHeatState) -> GridHeatState:
    """[CACHED] Reuses tx_live_heat_by_zone.csv (Section 5's cache) if it
    already has today's target date, instead of always calling FortyGuard -
    same reasoning as the California version."""
    import os

    target_date = state.get("target_date")
    if target_date is None:
        target_date = (date.today() - timedelta(days=1)).strftime("%Y-%m-%d")

    cache_path = "tx_live_heat_by_zone.csv"
    if os.path.exists(cache_path) and not state.get("force_refresh_heat", False):
        cached = pd.read_csv(cache_path)
        if not cached.empty and str(cached["as_of_date"].iloc[0]) == target_date:
            return {"live_heat": cached}

    live_heat = fetch_live_heat_by_zone(
        client=state["fg_client"], polygon_aoi=state["pilot_aoi"],
        grid=state["grid"], target_date=target_date,
    )
    live_heat.to_csv(cache_path, index=False)
    return {"live_heat": live_heat}


def fetch_recent_ercot_demand(end_date, n_days: int = 3):
    """Fetch up to n_days of actual ERCOT system demand (EIA-930, ERCO
    respondent) ending the day before end_date. Returns a list of daily
    peak MW values, oldest first (may be shorter than n_days on partial
    failure - e.g. EIA-930's typical 1-2 day publication lag)."""
    try:
        from google.colab import userdata
        EIA_API_KEY = userdata.get("EIA_API_KEY")
        start = (end_date - timedelta(days=n_days)).strftime("%Y-%m-%d")
        end = (end_date - timedelta(days=1)).strftime("%Y-%m-%d")
        resp = requests.get(
            "https://api.eia.gov/v2/electricity/rto/region-data/data/",
            params={"api_key": EIA_API_KEY, "frequency": "hourly", "data[0]": "value",
                    "facets[respondent][]": "ERCO", "facets[type][]": "D",
                    "start": f"{start}T00", "end": f"{end}T23",
                    "sort[0][column]": "period", "sort[0][direction]": "asc", "length": 5000},
            timeout=30)
        resp.raise_for_status()
        data = resp.json()["response"]["data"]
        if not data:
            return []
        df = pd.DataFrame(data)
        df["period"] = pd.to_datetime(df["period"])
        df["value"] = pd.to_numeric(df["value"], errors="coerce")
        df["date"] = df["period"].dt.date
        return df.groupby("date")["value"].max().sort_index().tolist()
    except Exception as e:
        print(f"    [prev-day ERCOT demand] unavailable ({e})")
        return []


def fetch_live_solar_irradiance(fg_client, pilot_aoi, as_of_date, temperature_c: float, hour: int = 15):
    """Same fix as Section 4 - environmental_parameters() requires
    `temperature` (Celsius) as input."""
    try:
        centroid = shape(pilot_aoi["features"][0]["geometry"]).centroid
        resp = fg_client.environmental_parameters(
            latitude=centroid.y, longitude=centroid.x, temperature=temperature_c,
            start_date=as_of_date.strftime("%Y-%m-%d"), start_time=f"{hour:02d}:00",
            filter_type=1,
        )
        locations = resp["result"].get("locations") or []
        if not locations:
            return None
        params = locations[0].get("parameters", {})
        for key in ("solar_irradiance_wm2", "solar_irradiance", "irradiance_wm2", "ghi_wm2", "ghi", "solar"):
            if key in params:
                return params[key]
        clear_sky = locations[0].get("solar_irradiance", {}).get("clear_sky", {})
        return clear_sky.get("ghi")
    except Exception as e:
        print(f"    [solar] unavailable ({e})")
        return None

def predict_demand_stress(demand_bundle: dict, baseline_mw: float, current_temp_f: float,
                          fg_client, pilot_aoi: dict, as_of_date=None):
    """Returns (predicted_mw, demand_stress_pct, model_used). Prefers
    demand_bundle["multivariate"]; falls back to demand_bundle["linear"]
    (avg_temp_f only) whenever a required live feature isn't fetchable -
    EIA-930's publication lag, a FortyGuard quota miss, etc. NOTE: costs
    extra API calls (EIA-930 for rolling/previous-day demand, FortyGuard
    heatmap for previous-day temp, FortyGuard env_params for solar) - call
    once per pipeline run, not in a loop."""
    as_of_date = as_of_date or (datetime.now() - timedelta(days=1))
    features = demand_bundle.get("features")

    if demand_bundle.get("multivariate") is not None and features:
        row = {
            "avg_temp_f": current_temp_f,
            "dow": as_of_date.weekday(),
            "is_weekend": int(as_of_date.weekday() >= 5),
            "month": as_of_date.month,
        }
        recent = fetch_recent_ercot_demand(as_of_date, n_days=3)
        if recent:
            row["prev_day_demand_mw"] = recent[-1]
            row["rolling_3d_demand_mw"] = sum(recent) / len(recent)

        try:
            prev_resp = fg_client.create_heatmap(
                polygon_aoi=pilot_aoi, start_date=(as_of_date - timedelta(days=1)).strftime("%Y-%m-%d"),
                filter_type=3, granularity=100)
            row["prev_temp_f"] = prev_resp["result"]["stats_data"]["temperature_stats"]["mean"] * 9 / 5 + 32
        except Exception as e:
            print(f"    [prev-day temp] unavailable ({e})")

        try:
            import holidays as holidays_lib
            row["is_holiday"] = int(as_of_date.date() in holidays_lib.US(years=[as_of_date.year]))
        except ImportError:
            row["is_holiday"] = int((as_of_date.month, as_of_date.day) in {(7, 4), (1, 1), (12, 25)})


        if "solar_irradiance_wm2" in features:
            current_temp_c = (current_temp_f - 32) * 5 / 9  # current_temp_f already computed above
            row["solar_irradiance_wm2"] = fetch_live_solar_irradiance(fg_client, pilot_aoi, as_of_date, temperature_c=current_temp_c)
        if all(row.get(f) is not None for f in features):
            X_live = pd.DataFrame([row])[features]
            pred = demand_bundle["multivariate"].predict(X_live)[0]
            stress = (pred - baseline_mw) / baseline_mw * 100
            return pred, stress, "multivariate"
        missing = [f for f in features if row.get(f) is None]
        print(f"    [predict_demand_stress] missing live features {missing} - falling back to univariate model")

    pred = demand_bundle["linear"].predict(pd.DataFrame({"avg_temp_f": [current_temp_f]}))[0]
    stress = (pred - baseline_mw) / baseline_mw * 100
    return pred, stress, "linear (fallback)"


def demand_forecast_agent(state: GridHeatState) -> GridHeatState:
    """CHANGE FROM ORIGINAL: state["demand_model"] is now a bundle dict
    ({"linear", "multivariate", "features"} - see Section 4-FINAL), not a
    bare sklearn estimator. predict_demand_stress() picks whichever model
    it can actually feed with live data this run."""
    demand_bundle = state["demand_model"]
    baseline_mw = state["baseline_demand_mw"]
    current_temp_f = state["live_heat"]["heat_raw_f"].mean()
    predicted_mw, demand_stress_pct, model_used = predict_demand_stress(
        demand_bundle, baseline_mw, current_temp_f,
        fg_client=state["fg_client"], pilot_aoi=state["pilot_aoi"], as_of_date=None)
    demand_score = score_against_reference_range(demand_stress_pct, state["demand_historical_range"])
    return {"demand_stress_pct": demand_stress_pct, "demand_score": demand_score,
            "demand_model_used": model_used}


def grid_asset_agent(state: GridHeatState) -> GridHeatState:
    lines_gdf = gpd.read_file("tx_transmission_lines_clean.geojson")
    return {"lines_joined": join_lines(state["grid"], lines_gdf, id_col="OBJECTID")}


def outage_agent(state: GridHeatState) -> GridHeatState:
    """EAGLE-I has no per-event geometry (unlike PSPS) - re-reads the same
    county-level total every other cell uses and scores it against the
    cross-county reference range from Section 2's Stage B, rather than
    doing a spatial join that has nothing to differentiate."""
    outage_score = score_against_reference_range(
        pilot_total_outage_customers, state["outage_reference_range"]
    )
    return {"outage_score": outage_score}


def vulnerability_agent(state: GridHeatState) -> GridHeatState:
    ej_gdf = gpd.read_file("ejscreen_tx_clean.geojson")
    vulnerability_joined = join_polygons_area_weighted(
        state["grid"], ej_gdf, id_col="ID",
        extensive_cols=["ACSTOTPOP"],
        intensive_cols=["P_DEMOGIDX_5", "PEOPCOLORPCT", "OVER64PCT", "UNDER5PCT"],
    )
    return {"vulnerability_joined": vulnerability_joined}


def risk_engine_node(state: GridHeatState) -> GridHeatState:
    risk_table = build_risk_table(
        grid=state["grid"], live_heat=state["live_heat"],
        vulnerability_joined=state["vulnerability_joined"], lines_joined=state["lines_joined"],
        demand_score=state["demand_score"], outage_score=state["outage_score"],
        infra_weights=state.get("infra_weights"),
    )
    return {"risk_table": risk_table}


def route_by_risk(state: GridHeatState) -> str:
    threshold = state["risk_threshold"]
    critical = state["risk_table"][state["risk_table"]["grid_heat_risk"] >= threshold]
    return "monitor_only" if critical.empty else "escalated"


def scenario_generator_node(state: GridHeatState) -> GridHeatState:
    critical_zones = state["risk_table"][state["risk_table"]["grid_heat_risk"] >= state["risk_threshold"]].copy()
    return {"critical_zones": critical_zones, "status": "escalated"}


def monitor_node(state: GridHeatState) -> GridHeatState:
    return {"status": "monitor_only"}


def build_graph():
    graph = StateGraph(GridHeatState)
    graph.add_node("HeatAgent", heat_agent)
    graph.add_node("DemandForecastAgent", demand_forecast_agent)
    graph.add_node("GridAssetAgent", grid_asset_agent)
    graph.add_node("OutageAgent", outage_agent)
    graph.add_node("VulnerabilityAgent", vulnerability_agent)
    graph.add_node("RiskEngine", risk_engine_node)
    graph.add_node("ScenarioGenerator", scenario_generator_node)
    graph.add_node("Monitor", monitor_node)

    graph.add_edge(START, "HeatAgent")
    graph.add_edge("HeatAgent", "DemandForecastAgent")
    graph.add_edge("DemandForecastAgent", "GridAssetAgent")
    graph.add_edge("GridAssetAgent", "OutageAgent")
    graph.add_edge("OutageAgent", "VulnerabilityAgent")
    graph.add_edge("VulnerabilityAgent", "RiskEngine")
    graph.add_conditional_edges("RiskEngine", route_by_risk, {"escalated": "ScenarioGenerator", "monitor_only": "Monitor"})
    graph.add_edge("ScenarioGenerator", END)
    graph.add_edge("Monitor", END)
    return graph.compile()


Running the graph end-to-end wires in the real objects built above (the
FortyGuard client, the pilot AOI, the grid, the trained demand model, the
demand historical range, and the outage reference range from Section 2's
Stage B). The risk threshold below (40.0) is carried over as a starting
guess from the Fresno run - like Fresno, treat it as a placeholder until a
real run shows what Texas's actual scores top out at, then adjust the same
way Fresno's was lowered from 70 to 40 after seeing real numbers.

In [ ]:
import json
import joblib
from google.colab import userdata
from fortyguard import FortyGuardClient

client = FortyGuardClient(api_key=userdata.get("FORTYGUARD_API_KEY"))

AOI_PATH = f"{PILOT_COUNTY.lower().replace(' ', '_')}_aoi_precise.geojson"  # matches Stage C's OUTPUT_PATH (Cell 26)
with open(AOI_PATH) as f:
    pilot_aoi = json.load(f)

grid_gdf = gpd.read_file("heat_zone_grid_tx.geojson")
demand_model = joblib.load("tx_demand_model.joblib")  # now a bundle dict:
# {"linear": <model>, "multivariate": <model>, "features": [...]} - see Section 4-FINAL

merged = pd.read_csv("tx_temp_demand_merged.csv")
baseline_demand_mw = merged["demand_peak_mw"].mean()
merged["stress_pct"] = (merged["demand_peak_mw"] - baseline_demand_mw) / baseline_demand_mw * 100
demand_historical_range = (merged["stress_pct"].min(), merged["stress_pct"].max())

# From Section 2's Stage B scoreboard - the cross-county spread that gives
# a single county-level outage total something meaningful to be scored
# against (same role demand_historical_range plays for demand_stress_pct).
outage_reference_range = (scoreboard["outages"].min(), scoreboard["outages"].max())

# From Section 1.3c - optional; enables RiskEngine to spatially redistribute
# demand_score across zones instead of broadcasting one flat value everywhere.
import os
infra_weights = pd.read_csv("infra_weights_tx.csv") if os.path.exists("infra_weights_tx.csv") else None
if infra_weights is None:
    print("NOTE: infra_weights_tx.csv not found - demand_score will be broadcast "
          "identically to every zone in this run (no spatial signal).")

app = build_graph()
initial_state = {
    "risk_threshold": 40.0,  # placeholder - adjust once a real run shows actual scores, same as Fresno
    "target_date": None,
    "fg_client": client,
    "pilot_aoi": pilot_aoi,
    "grid": grid_gdf,
    "demand_model": demand_model,
    "baseline_demand_mw": baseline_demand_mw,
    "demand_historical_range": demand_historical_range,
    "outage_reference_range": outage_reference_range,
    "pilot_county": PILOT_COUNTY,
    "infra_weights": infra_weights,
}

result = app.invoke(initial_state)
print(f"Status: {result['status']}")
if result.get("demand_model_used"):
    print(f"Demand model used this run: {result['demand_model_used']}")
if result["status"] == "escalated":
    print(result["critical_zones"][["zone_id", "grid_heat_risk"]])


Submitted -> activity_id=b3f76587-a7b6-4802-b4b5-98b484866cc2
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: processing
  status: completed
Done.
Submitted -> activity_id=7d0e881b-b263-48f5-ab78-a8161168cdfa
  status: processing
  status: processing
  status: completed
Done.
Status: escalated
Demand model used this run: multivariate
   zone_id  grid_heat_risk
0   Z00046       81.939369
1   Z00038       81.282251
2   Z00006       80.461906
3   Z00014       78.444198
4   Z00022       70.643995
5   Z00030       67.375938
6   Z00036       66.430504
7   Z00037       63.363328
8   Z00035       58.537890
9   Z00044       56.060592
10  Z00032       55.207797
11  Z00033       55.060207
12  Z00045       54.409222
13  Z00052       52.940233
14  Z00034       51.477221
15  Z00003       50.026830
16  Z00005       49.976701
17  Z00002       49.776653
18  Z00004       48.5

## 8. Optimization / Scenario Layer — Resilience Decision Agent

Unchanged from the California version - as its own original comment noted, there
was no city-specific logic in this section to begin with. Same exact 0/1 knapsack
via dynamic programming (not greedy - greedy by value/cost ratio isn't guaranteed
optimal here), same `what_if_scenarios()` budget sweep.

In [ ]:
"""[SECTION 8] Optimization / Scenario layer - unchanged from the
California version (no city-specific logic here). Exact 0/1 knapsack via
DP (not greedy - greedy by value/cost ratio isn't guaranteed optimal for
0/1 knapsack)."""
from dataclasses import dataclass
import pandas as pd

ACTIONS = {
    "battery_dispatch":     {"cost": 20_000, "risk_reduction_pct": 30, "requires_battery": True},
    "crew_deployment":      {"cost": 8_000,  "risk_reduction_pct": 12, "requires_battery": False},
    "demand_response":      {"cost": 5_000,  "risk_reduction_pct": 18, "requires_battery": False},
    "emergency_monitoring": {"cost": 3_000,  "risk_reduction_pct": 5,  "requires_battery": False},
}


@dataclass
class ZoneActionOption:
    zone_id: str
    action: str
    cost: int
    value: float


def build_candidate_options(critical_zones: pd.DataFrame, battery_joined: pd.DataFrame) -> list:
    battery_zones = set(
        battery_joined.loc[battery_joined.get("n_battery_sites", 0) > 0, "zone_id"]
    ) if battery_joined is not None and not battery_joined.empty else set()

    options = []
    for _, row in critical_zones.iterrows():
        zone_id, zone_risk = row["zone_id"], row["grid_heat_risk"]
        for action_name, spec in ACTIONS.items():
            if spec["requires_battery"] and zone_id not in battery_zones:
                continue
            options.append(ZoneActionOption(
                zone_id=zone_id, action=action_name, cost=spec["cost"],
                value=spec["risk_reduction_pct"] * zone_risk / 100.0,
            ))
    return options


def optimize_plan(options: list, budget: int) -> dict:
    n = len(options)
    dp2 = [[0.0] * (budget + 1) for _ in range(n + 1)]
    for i, opt in enumerate(options, start=1):
        for b in range(budget + 1):
            dp2[i][b] = dp2[i - 1][b]
            if opt.cost <= b:
                candidate = dp2[i - 1][b - opt.cost] + opt.value
                if candidate > dp2[i][b]:
                    dp2[i][b] = candidate

    selected, b = [], budget
    for i in range(n, 0, -1):
        if dp2[i][b] != dp2[i - 1][b]:
            opt = options[i - 1]
            selected.append(opt)
            b -= opt.cost

    return {"selected": selected, "total_cost": sum(o.cost for o in selected),
            "total_value": dp2[n][budget], "budget": budget}


def summarize_plan(result: dict) -> pd.DataFrame:
    rows = [{"zone_id": o.zone_id, "action": o.action, "cost": o.cost, "value": round(o.value, 2)}
            for o in result["selected"]]
    return pd.DataFrame(rows).sort_values(["zone_id", "action"]).reset_index(drop=True)


def what_if_scenarios(critical_zones: pd.DataFrame, battery_joined: pd.DataFrame, budgets: list) -> pd.DataFrame:
    options = build_candidate_options(critical_zones, battery_joined)
    baseline_risk = critical_zones["grid_heat_risk"].sum()
    rows = [{"budget": 0, "risk_reduction_achieved": 0.0, "risk_reduction_pct_of_baseline": 0.0, "n_actions": 0}]
    for budget in budgets:
        result = optimize_plan(options, budget)
        rows.append({
            "budget": budget, "risk_reduction_achieved": round(result["total_value"], 2),
            "risk_reduction_pct_of_baseline": round(result["total_value"] / baseline_risk * 100, 1) if baseline_risk else 0.0,
            "n_actions": len(result["selected"]),
        })
    return pd.DataFrame(rows)


if __name__ == "__main__":
    critical_zones = result["critical_zones"]  # from Section 7's app.invoke() result
    battery_joined = pd.read_csv("battery_joined_tx.csv")

    options = build_candidate_options(critical_zones, battery_joined)
    print(f"Candidate (zone, action) pairs: {len(options)}")

    BUDGET = 50_000
    opt_result = optimize_plan(options, BUDGET)
    print(f"\nRecommended plan under ${BUDGET:,}:")
    print(summarize_plan(opt_result))
    print(f"\nTotal cost: ${opt_result['total_cost']:,} | Total risk-reduction value: {opt_result['total_value']:.2f}")

    print("\nWhat-if across budget levels:")
    print(what_if_scenarios(critical_zones, battery_joined, budgets=[10_000, 25_000, 50_000, 100_000]))


Candidate (zone, action) pairs: 75

Recommended plan under $50,000:
  zone_id           action  cost  value
0  Z00006  demand_response  5000  14.48
1  Z00014  demand_response  5000  14.12
2  Z00022  demand_response  5000  12.72
3  Z00030  demand_response  5000  12.13
4  Z00035  demand_response  5000  10.54
5  Z00036  demand_response  5000  11.96
6  Z00037  demand_response  5000  11.41
7  Z00038  demand_response  5000  14.63
8  Z00044  demand_response  5000  10.09
9  Z00046  demand_response  5000  14.75

Total cost: $50,000 | Total risk-reduction value: 126.82

What-if across budget levels:
   budget  risk_reduction_achieved  risk_reduction_pct_of_baseline  n_actions
0       0                     0.00                             0.0          0
1   10000                    29.38                             2.0          2
2   25000                    70.70                             4.9          5
3   50000                   126.82                             8.8         10
4  100000    

> ⚠️ **Modeling assumption, not operational fact.** The action costs and
> risk-reduction percentages in `ACTIONS` above are still **scenario
> assumptions used to demonstrate constrained optimization**, not
> utility-specific cost estimates or empirically calibrated intervention
> effects - same caveat as the California version, since these numbers were
> never state-specific to begin with. Everything upstream of this cell - heat,
> demand, transmission lines, EAGLE-I outage history, EJScreen vulnerability,
> battery locations - is real public Texas data. The optimizer's *method*
> (exact 0/1 knapsack) is real and correct; its *inputs* in `ACTIONS` remain
> illustrative placeholders.

*Section 9 (LLM Decision-Explanation Layer) carries over unchanged from the California version - it only ever consumed the generic `risk_table` / `critical_zones` / optimizer output already produced by Sections 6-8, with no state-specific logic of its own.*

## 9. LLM Decision-Explanation Layer

Everything through Section 8 is a **deterministic analytical pipeline**: fixed formulas
and an exact optimizer, with no LLM involved anywhere in the risk score or the
funding decision. This section adds the one piece that pipeline can't do on its
own — turning a JSON result into a plain-language explanation a non-technical
stakeholder can read — **without letting the LLM touch a single number**.

Uses **Groq's free-tier hosted inference** (`openai/gpt-oss-120b` via an
OpenAI-compatible chat API) rather than a paid API — a free key from
[console.groq.com/keys](https://console.groq.com/keys), stored as
`GROQ_API_KEY` in Colab Secrets, is enough to run this section and Section 10's
dashboard buttons with no billing required.

`openai/gpt-oss-120b` is a **reasoning model** — by default it spends part of
its token budget on hidden chain-of-thought before writing the actual answer.
`reasoning_effort="low"` keeps that internal reasoning minimal, and `max_tokens`
is set generously above what a short answer needs (500-600, not ~300) so the
final explanation still has room to be written even if some tokens go to
reasoning first — otherwise the reasoning alone can consume the whole budget
and the "explanation" comes back truncated to a couple of words or empty.

**What the LLM is, and isn't, allowed to do here:**

- It does **not** compute `grid_heat_risk`, any component score, or the
  optimizer's selected plan — those come entirely from Section 6 and Section 8.
- It **only** receives a small JSON "evidence packet" (the scores and real
  spatial assets already computed earlier) and is instructed to explain what's
  in that packet, not to introduce facts that aren't there.
- The system prompts explicitly forbid it from implying it performed the
  calculation or made the funding decision itself — it explains a decision
  that the optimizer already made mathematically.

### 9.1 Guardrails on the output

A system prompt is an instruction, not a guarantee — so every explanation is
checked programmatically **after** it comes back, before it ever reaches the
dashboard:

1. **Numeric grounding check.** Every number-like figure in the LLM's response
   (a score, a dollar amount, a count) is extracted and compared against the
   actual numbers in the evidence packet it was given. Anything that doesn't
   match a real evidence value within a small tolerance is treated as a
   hallucinated figure. Bare small integers (< 10) are exempted from this check
   — sentences like "the five components" trip this constantly and aren't the
   failure mode this guards against (a fabricated risk score or dollar amount
   is). Shorthand like "$50K" is expanded to 50,000 before comparison, so a
   real evidence figure written in shorthand isn't mistaken for a fabricated
   one — and the system prompts separately ask the model to avoid shorthand
   and new derived numbers (averages, percentages) it wasn't given in the
   first place, since those are a common source of plausible-looking but
   ungrounded figures.
2. **Self-attribution check.** A regex scan rejects phrasing like *"I
   calculated,"* *"I determined the risk,"* or *"I selected the action"* —
   the explanation layer explains a result, it doesn't get to claim it produced
   one.

If either check fails, the same prompt is retried once with the specific
failure reason appended as a correction; if it fails again, the pipeline
**falls back to a template-based sentence built directly from the evidence
packet** rather than showing an ungrounded explanation or breaking the
dashboard. Every guardrail rejection is printed to the console (visible in
Colab's output, and in LangSmith traces if Section 7.1's tracing is enabled) so a
pattern of repeated rejections is visible during development, not silent.

Two functions are exposed: `explain_zone()` (why one zone was flagged) and
`explain_plan()` (why the optimizer's specific action mix was chosen under the
budget) — both used by the dashboard's "Why?" buttons in Section 10.


In [ ]:
!pip install -q groq
"""
[NEW] LLM Decision-Explanation Layer, with guardrails.

Turns the deterministic risk_table + critical_zones + optimizer's recommended
plan (Cells 11 and 13) into a natural-language "why" explanation, using Groq's
free-tier hosted inference (Llama models, OpenAI-compatible chat API). This
layer never computes or changes a risk score or an optimization decision - it
only explains a result that already exists.

Every response is validated (see validate_explanation below) before being
returned: a self-attribution check (rejects phrasing implying the LLM computed
the score or made the funding decision) and a numeric-grounding check (rejects
any number-like figure that doesn't trace back to a real value in the
evidence packet). A failed check triggers one retry with the failure reason
appended, then falls back to a template built from the same evidence if it
fails again - never an ungrounded explanation, never a broken dashboard.
"""
import json
import re

from groq import Groq

# Free-tier Groq model. llama-3.3-70b-versatile was deprecated by Groq on
# 2026-08-16 - openai/gpt-oss-120b is Groq's official recommended
# replacement (see https://console.groq.com/docs/deprecations). Swap for
# "openai/gpt-oss-20b" for an even faster/cheaper option if rate limits
# become an issue.
EXPLANATION_MODEL = "openai/gpt-oss-120b"


def build_zone_evidence(zone_id: str, risk_table, lines_joined, battery_joined) -> dict:
    """Assemble the JSON 'evidence packet' for one zone - scores plus the real
    spatial assets already joined onto it in Cell 8. This dict is the ONLY
    input the LLM ever sees for a zone; it never has access to raw source
    data, the scoring formula, or the optimizer's code."""
    row = risk_table.loc[risk_table["zone_id"] == zone_id].iloc[0]

    n_lines = int((lines_joined["zone_id"] == zone_id).sum()) if lines_joined is not None else 0

    n_batteries = 0
    if battery_joined is not None and not battery_joined.empty:
        match = battery_joined.loc[battery_joined["zone_id"] == zone_id]
        if not match.empty:
            n_batteries = int(match["n_battery_sites"].iloc[0])

    return {
        "zone_id": zone_id,
        "grid_heat_risk": round(float(row["grid_heat_risk"]), 2),
        "heat_score": round(float(row["heat_score"]), 1),
        "demand_score": round(float(row["demand_score"]), 1),
        "infra_score": round(float(row["infra_score"]), 1),
        "outage_score": round(float(row["outage_score"]), 1),
        "vulnerability_score": round(float(row["vuln_score"]), 1),
        "nearby_transmission_lines": n_lines,
        "nearby_battery_sites": n_batteries,
    }


# ============ Guardrails ============

FORBIDDEN_SELF_ATTRIBUTION_PATTERNS = [
    r"\bi calculated\b", r"\bi computed\b", r"\bi determined the risk\b",
    r"\bi decided\b", r"\bmy calculation\b", r"\bi selected the\b",
    r"\bi chose the action\b", r"\bi optimized\b", r"\bi ran the optimization\b",
    r"\bi assigned the score\b", r"\bi ran the model\b", r"\bi allocated\b",
]


def _flatten_numeric_values(obj) -> list:
    """Recursively pull every int/float leaf out of a dict/list structure -
    handles both the flat zone-evidence dict and the nested plan payload
    (which has a list of {zone_id, action, cost, value} dicts inside it)."""
    values = []
    if isinstance(obj, dict):
        for v in obj.values():
            values.extend(_flatten_numeric_values(v))
    elif isinstance(obj, list):
        for v in obj:
            values.extend(_flatten_numeric_values(v))
    elif isinstance(obj, (int, float)) and not isinstance(obj, bool):
        values.append(float(obj))
    return values


def _extract_candidate_numbers(text: str) -> list:
    """Numbers worth grounding-checking: anything with a decimal point, a
    dollar sign, a K/M suffix, or a bare value >= 10. Bare small integers
    (2, 3, 4, 5...) are almost always sentence structure ("the five
    components", "3-5 sentences") rather than a claimed score or dollar
    figure, and checking those causes constant false positives without
    catching real hallucination. K/M suffixes ("50K", "1.2M") are expanded
    to their full value before comparison - otherwise "$50K" for a real
    $50,000 evidence figure gets parsed as the bare number 50 and wrongly
    flagged as ungrounded. The leading negative lookbehind blocks matches
    starting mid-token - without it, a zone ID like "Z00010" gets its
    digit run misread as the standalone number 10 (or 0010 -> 10), which
    is never in the evidence and would be wrongly flagged as hallucinated."""
    matches = re.findall(r"(?<![A-Za-z0-9])\$?\d[\d,]*\.?\d*\s?[kKmM]?\b", text)
    numbers = []
    for m in matches:
        m = m.strip()
        multiplier = 1
        if m and m[-1] in "kK":
            multiplier = 1_000
            m = m[:-1].strip()
        elif m and m[-1] in "mM":
            multiplier = 1_000_000
            m = m[:-1].strip()
        cleaned = m.replace("$", "").replace(",", "")
        try:
            value = float(cleaned) * multiplier
        except ValueError:
            continue
        if multiplier > 1 or "." in cleaned or m.startswith("$") or value >= 10:
            numbers.append(value)
    return numbers


def _is_grounded(value: float, evidence_numbers: list, rel_tol: float = 0.02, abs_tol: float = 0.6) -> bool:
    return any(abs(value - ev) <= max(abs_tol, abs(ev) * rel_tol) for ev in evidence_numbers)


def validate_explanation(text: str, evidence: dict) -> tuple:
    """Returns (is_valid, reason). Two checks: no self-attribution language
    implying the LLM computed a score or made the funding decision, and every
    number-like figure mentioned traces back to a real value in the evidence
    packet the LLM was actually given."""
    lowered = text.lower()
    for pattern in FORBIDDEN_SELF_ATTRIBUTION_PATTERNS:
        if re.search(pattern, lowered):
            return False, f"forbidden self-attribution phrase matched ('{pattern}')"

    evidence_numbers = _flatten_numeric_values(evidence)
    for value in _extract_candidate_numbers(text):
        if not _is_grounded(value, evidence_numbers):
            return False, f"ungrounded number in explanation: {value}"

    return True, "ok"


def _call_llm(client: "Groq", system_prompt: str, payload: dict, correction: str = None,
              model: str = EXPLANATION_MODEL, max_tokens: int = 600) -> str:
    """openai/gpt-oss-120b (the current EXPLANATION_MODEL) is a REASONING
    model on Groq - by default (reasoning_effort="medium") it spends part
    of max_tokens on a hidden chain-of-thought before writing the actual
    answer. With a low max_tokens budget (the original 300/350), the
    reasoning alone can consume the whole budget, leaving the final answer
    truncated to a few words or empty. reasoning_effort="low" keeps that
    internal reasoning minimal, and max_tokens is set generously above what
    a 2-5 sentence answer needs, so there's still room left for the actual
    explanation even if some reasoning tokens are spent."""
    messages = [{"role": "system", "content": system_prompt},
                {"role": "user", "content": json.dumps(payload)}]
    if correction:
        messages.append({"role": "user", "content": correction})
    response = client.chat.completions.create(
        model=model, max_tokens=max_tokens, reasoning_effort="low", messages=messages,
    )
    text = response.choices[0].message.content
    if not text or not text.strip():
        raise ValueError("Empty response from LLM (likely reasoning tokens consumed the "
                          "whole max_tokens budget before any answer text was written)")
    return text


ZONE_SYSTEM_PROMPT = """You are explaining the output of a deterministic grid-\
heat risk model to a non-technical stakeholder. You are given a JSON evidence \
packet for one zone (its risk score, its five component scores 0-100, and the \
real grid assets nearby). Write 2-4 plain-language sentences on why this zone \
was flagged, grounded ONLY in the numbers given. Do not invent data, do not \
state a precise cause-and-effect the numbers don't fully support, and do not \
imply you performed any calculation yourself - the scores were computed by a \
separate deterministic engine before you saw them. If a component score is 0 \
or notably low, say so plainly rather than guessing at a reason. Do not \
compute or state any new number that isn't already in the JSON (no averages, \
percentages, sums, or rounded-off shorthand like "50K") - reference the given \
figures directly, in full digit form."""


def explain_zone(client: "Groq", evidence: dict, model: str = EXPLANATION_MODEL, max_attempts: int = 2) -> str:
    correction = None
    for attempt in range(1, max_attempts + 1):
        try:
            text = _call_llm(client, ZONE_SYSTEM_PROMPT, evidence, correction, model, max_tokens=500)
        except Exception as e:
            print(f"[guardrail] zone explanation attempt {attempt} call failed: {e.__class__.__name__}: {e}")
            break
        is_valid, reason = validate_explanation(text, evidence)
        if is_valid:
            return text
        print(f"[guardrail] zone explanation attempt {attempt} rejected: {reason}")
        print(f"[guardrail] rejected text was: {text!r}")
        correction = (f"Your previous answer was rejected: {reason}. Rewrite it using ONLY "
                      f"the numbers already in the evidence JSON above (in full digit form, "
                      f"no rounding, no shorthand like '50K', no new averages or percentages), "
                      f"and do not claim you performed any calculation or made any decision yourself.")

    # Never let a dashboard render fail because the LLM call or guardrail did -
    # degrade to a template built from the same evidence packet.
    return (
        f"Zone {evidence['zone_id']} scored {evidence['grid_heat_risk']} on the "
        f"grid heat risk scale (heat {evidence['heat_score']}, demand "
        f"{evidence['demand_score']}, infrastructure {evidence['infra_score']}, "
        f"outage history {evidence['outage_score']}, vulnerability "
        f"{evidence['vulnerability_score']}), with {evidence['nearby_transmission_lines']} "
        f"transmission line(s) and {evidence['nearby_battery_sites']} battery site(s) nearby.\n"
        f"[LLM explanation unavailable after guardrail checks - showing evidence-based summary instead.]"
    )


PLAN_SYSTEM_PROMPT = """You are explaining a resource-allocation plan chosen \
by an exact 0/1 knapsack optimizer, not by you. You are given the optimizer's \
selected (zone, action) pairs, their MODELED costs and risk-reduction values, \
and the total budget. Explain in 3-5 plain-language sentences why this \
combination was selected, in terms of maximizing total modeled risk-reduction \
under the budget constraint. Explicitly state that action costs and risk-\
reduction percentages are modeling assumptions for this demonstration, not \
empirically validated operational figures. Do not claim any selection method \
other than exact optimization under the stated budget, and do not take credit \
for making the selection yourself - you are explaining a decision that was \
already made mathematically before you saw it. Do not compute or state any \
new number that isn't already in the JSON (no averages across actions, no \
percentages of the budget, no rounded-off shorthand like "50K") - reference \
the given dollar and value figures directly, in full digit form (e.g. "$50,000", \
not "$50K" or "50 grand")."""


def explain_plan(client: "Groq", plan_df, total_cost: float, total_value: float,
                  budget: int, model: str = EXPLANATION_MODEL, max_attempts: int = 2) -> str:
    payload = {
        "budget": budget,
        "total_cost": total_cost,
        "total_modeled_risk_reduction_value": round(total_value, 2),
        "selected_actions": plan_df.to_dict(orient="records"),
    }

    correction = None
    for attempt in range(1, max_attempts + 1):
        try:
            text = _call_llm(client, PLAN_SYSTEM_PROMPT, payload, correction, model, max_tokens=600)
        except Exception as e:
            print(f"[guardrail] plan explanation attempt {attempt} call failed: {e.__class__.__name__}: {e}")
            break
        is_valid, reason = validate_explanation(text, payload)
        if is_valid:
            return text
        print(f"[guardrail] plan explanation attempt {attempt} rejected: {reason}")
        print(f"[guardrail] rejected text was: {text!r}")
        correction = (f"Your previous answer was rejected: {reason}. Rewrite it using ONLY "
                      f"the numbers already in the JSON above (in full digit form, no rounding, "
                      f"no shorthand like '50K', no new averages or percentages), and do not "
                      f"claim you performed any calculation or made the selection yourself.")

    return (
        f"Under a ${budget:,} budget, the optimizer selected {len(plan_df)} "
        f"action(s) totaling ${total_cost:,}, for a modeled risk-reduction "
        f"value of {total_value:.2f}. Action costs and risk-reduction percentages "
        f"are modeling assumptions, not empirically validated figures.\n"
        f"[LLM explanation unavailable after guardrail checks - showing evidence-based summary instead.]"
    )


if __name__ == "__main__":
    from google.colab import userdata

    # Free API key from https://console.groq.com/keys - add it under Colab
    # Secrets (key icon, left sidebar) as GROQ_API_KEY before running this.
    groq_client = Groq(api_key=userdata.get("GROQ_API_KEY"))

    # Explain the single highest-risk critical zone from Cell 12's run.
    top_zone_id = critical_zones.sort_values("grid_heat_risk", ascending=False).iloc[0]["zone_id"]
    evidence = build_zone_evidence(top_zone_id, risk_table, lines_joined, battery_joined)
    print(f"--- Zone explanation: {top_zone_id} ---")
    print(explain_zone(groq_client, evidence))

    # Explain the recommended plan from Cell 13's optimizer. Cell 13's demo
    # block only prints summarize_plan(opt_result) - it doesn't save it to a
    # variable - so build plan_df here from the still-available opt_result
    # and summarize_plan() (both defined in Cell 13, already run earlier).
    plan_df = summarize_plan(opt_result)
    print("\n--- Plan explanation ---")
    print(explain_plan(groq_client, plan_df, opt_result["total_cost"],
                        opt_result["total_value"], BUDGET))


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 4.1 MB/s eta 0:00:00
--- Zone explanation: Z00046 ---
The model flagged zone Z00046 because its overall grid‑heat risk score is 81.94, driven by very high component scores for heat (93.6), demand (100.0) and outage (100.0). The infrastructure component is lower at 40.0, while vulnerability sits at 58.7. There are 4 nearby transmission lines and no nearby battery sites, which the engine captured directly in the report.

--- Plan explanation ---
The exact 0/1 knapsack optimizer chose the ten demand‑response actions because, given the $50,000 budget, this set uses the full $50,000 budget while delivering the highest possible total modeled risk‑reduction value of 126.82. Each selected action costs $5,000 and contributes a modeled risk‑reduction value ranging from 10.09 up to 14.75, and the optimizer identified that combining these particular ten actions yields the greatest sum of values without exceeding the budget. This decision r

## 10. Interactive Dashboard (Gradio)

Same four tabs as the California version. Only the loaded file names, CRS
(`EPSG:3083` instead of `EPSG:3310`), and the disclaimer's data-source list
change (EAGLE-I / EJScreen instead of PSPS / CalEnviroScreen) - the Gradio
layout, the map logic, and the on-demand "Why?" pattern are unchanged.

In [ ]:
!pip install -q plotly

In [ ]:
!pip install -q groq

In [ ]:
"""
[TX-10] GridHeat AI Dashboard - Gradio, 4 tabs (Risk Overview, Map View,
Zone Evidence with "Why?", and Critical Zones + Recommended Plan with "Why
this plan?"). LLM calls are on-demand only (per Section 9's guardrails) -
never used to compute a score or make the funding decision itself.
"""
import json

import gradio as gr
import geopandas as gpd
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import folium
import branca.colormap as cm
from groq import Groq

# ---- Load everything once at startup ----
risk_table = pd.read_csv("tx_risk_table.csv")
grid_gdf = gpd.read_file("heat_zone_grid_tx.geojson")
lines_gdf = gpd.read_file("tx_transmission_lines_clean.geojson")
lines_joined = pd.read_csv("lines_joined_tx.csv") if __import__("os").path.exists("lines_joined_tx.csv") else pd.DataFrame(columns=["zone_id"])
battery_joined = pd.read_csv("battery_joined_tx.csv") if __import__("os").path.exists("battery_joined_tx.csv") else pd.DataFrame()

RISK_THRESHOLD = 40.0  # same placeholder used in Section 7 - adjust once a real run shows actual scores
critical_zones = risk_table[risk_table["grid_heat_risk"] >= RISK_THRESHOLD].copy()

# ---- Groq client for the "Why?" buttons (Section 9) ----
try:
    from google.colab import userdata
    _groq_client = Groq(api_key=userdata.get("GROQ_API_KEY"))
except Exception:
    _groq_client = None

DISCLAIMER_MD = (
    "> \u26a0\ufe0f **Modeling assumption, not operational fact.** Action costs and "
    "risk-reduction percentages below are scenario assumptions used to "
    "demonstrate constrained optimization - not utility-specific cost "
    "estimates or empirically calibrated intervention effects. Heat, demand, "
    "transmission lines, EAGLE-I outage history, EJScreen vulnerability, and "
    "battery locations are real public Texas data; the dollar figures and % "
    "reductions for each action are illustrative placeholders. Demand and "
    "outage scores are also identical across every zone by construction - "
    "ERCOT publishes demand at weather-zone resolution and EAGLE-I at county "
    "resolution, both coarser than this AOI (see Sections 1.3, 3.2, and 6)."
)


# ============ Tab 1: Risk Overview ============

def get_risk_table():
    display_cols = ["zone_id", "grid_heat_risk", "heat_score", "demand_score",
                     "infra_score", "outage_score", "vuln_score"]
    return risk_table[display_cols].round(2)


# ============ Tab 2: Map View ============

def build_map_html():
    grid_wgs = grid_gdf.to_crs("EPSG:4326").merge(risk_table[["zone_id", "grid_heat_risk"]], on="zone_id", how="left")
    lines_wgs = lines_gdf.to_crs("EPSG:4326")

    center_point = grid_wgs.to_crs("EPSG:3083").geometry.union_all().centroid
    center_point_wgs = gpd.GeoSeries([center_point], crs="EPSG:3083").to_crs("EPSG:4326").iloc[0]
    center_lat, center_lon = center_point_wgs.y, center_point_wgs.x

    m = folium.Map(location=[center_lat, center_lon], zoom_start=13, tiles="cartodbpositron")

    colormap = cm.LinearColormap(
        colors=["#fee8c8", "#fdbb84", "#e34a33"],
        vmin=risk_table["grid_heat_risk"].min(), vmax=risk_table["grid_heat_risk"].max(),
        caption="Grid Heat Risk",
    )
    colormap.add_to(m)

    folium.GeoJson(
        grid_wgs,
        style_function=lambda feature: {
            "fillColor": colormap(feature["properties"]["grid_heat_risk"]),
            "color": "gray", "weight": 0.5, "fillOpacity": 0.7,
        },
        tooltip=folium.GeoJsonTooltip(fields=["zone_id", "grid_heat_risk"], aliases=["Zone", "Risk"]),
    ).add_to(m)

    folium.GeoJson(
        lines_wgs,
        style_function=lambda feature: {"color": "black", "weight": 2},
        tooltip=folium.GeoJsonTooltip(fields=["Name"]) if "Name" in lines_wgs.columns else None,
    ).add_to(m)

    return m._repr_html_()


# ============ Zone evidence + "Why?" (Section 9's explain_zone) ============

def build_zone_evidence(zone_id: str) -> dict:
    row = risk_table.loc[risk_table["zone_id"] == zone_id].iloc[0]
    n_lines = int((lines_joined["zone_id"] == zone_id).sum()) if not lines_joined.empty else 0
    n_batteries = 0
    if battery_joined is not None and not battery_joined.empty:
        match = battery_joined.loc[battery_joined["zone_id"] == zone_id]
        if not match.empty:
            n_batteries = int(match["n_battery_sites"].iloc[0])
    return {
        "zone_id": zone_id,
        "grid_heat_risk": round(float(row["grid_heat_risk"]), 2),
        "heat_score": round(float(row["heat_score"]), 1),
        "demand_score": round(float(row["demand_score"]), 1),
        "infra_score": round(float(row["infra_score"]), 1),
        "outage_score": round(float(row["outage_score"]), 1),
        "vulnerability_score": round(float(row["vuln_score"]), 1),
        "nearby_transmission_lines": n_lines,
        "nearby_battery_sites": n_batteries,
    }


def show_zone_evidence(zone_id: str):
    if not zone_id:
        return "Select a zone above.", ""
    ev = build_zone_evidence(zone_id)
    card = (
        f"### Zone {ev['zone_id']}\n\n"
        f"| Component | Score (0-100) |\n|---|---|\n"
        f"| \U0001F525 Heat | {ev['heat_score']} |\n"
        f"| \u26a1 Demand Stress | {ev['demand_score']} |\n"
        f"| \U0001F3D7 Infrastructure | {ev['infra_score']} |\n"
        f"| \u26a0\ufe0f Outage History | {ev['outage_score']} |\n"
        f"| \U0001F465 Vulnerability | {ev['vulnerability_score']} |\n\n"
        f"**Grid Heat Risk: {ev['grid_heat_risk']}**\n\n"
        f"Nearby assets: {ev['nearby_transmission_lines']} transmission line(s), "
        f"{ev['nearby_battery_sites']} battery site(s)."
    )
    return card, ""  # second value clears any stale explanation text


def why_this_zone(zone_id: str):
    if not zone_id:
        return "Select a zone first."
    if _groq_client is None:
        ev = build_zone_evidence(zone_id)
        return explain_zone_fallback(ev)
    from __main__ import explain_zone  # defined in Section 9 above
    ev = build_zone_evidence(zone_id)
    return explain_zone(_groq_client, ev)


def explain_zone_fallback(ev: dict) -> str:
    return (
        f"Zone {ev['zone_id']} scored {ev['grid_heat_risk']} on the grid heat risk "
        f"scale (heat {ev['heat_score']}, demand {ev['demand_score']}, infrastructure "
        f"{ev['infra_score']}, outage history {ev['outage_score']}, vulnerability "
        f"{ev['vulnerability_score']}), with {ev['nearby_transmission_lines']} transmission "
        f"line(s) and {ev['nearby_battery_sites']} battery site(s) nearby. "
        f"[LLM explanation unavailable - no Groq client configured.]"
    )


# ============ Tab 3: Critical Zones + Recommended Plan ============

ACTIONS = {
    "battery_dispatch":     {"cost": 20_000, "risk_reduction_pct": 30, "requires_battery": True},
    "crew_deployment":      {"cost": 8_000,  "risk_reduction_pct": 12, "requires_battery": False},
    "demand_response":      {"cost": 5_000,  "risk_reduction_pct": 18, "requires_battery": False},
    "emergency_monitoring": {"cost": 3_000,  "risk_reduction_pct": 5,  "requires_battery": False},
}


def build_candidate_options(zones_df, battery_df):
    battery_zones = set(
        battery_df.loc[battery_df.get("n_battery_sites", 0) > 0, "zone_id"]
    ) if battery_df is not None and not battery_df.empty else set()

    options = []
    for _, row in zones_df.iterrows():
        zone_id, zone_risk = row["zone_id"], row["grid_heat_risk"]
        for action_name, spec in ACTIONS.items():
            if spec["requires_battery"] and zone_id not in battery_zones:
                continue
            options.append({"zone_id": zone_id, "action": action_name, "cost": spec["cost"],
                             "value": spec["risk_reduction_pct"] * zone_risk / 100.0})
    return options


def optimize_plan(options, budget):
    n = len(options)
    dp = [[0.0] * (budget + 1) for _ in range(n + 1)]
    for i, opt in enumerate(options, start=1):
        for b in range(budget + 1):
            dp[i][b] = dp[i - 1][b]
            if opt["cost"] <= b:
                candidate = dp[i - 1][b - opt["cost"]] + opt["value"]
                if candidate > dp[i][b]:
                    dp[i][b] = candidate

    selected, b = [], budget
    for i in range(n, 0, -1):
        if dp[i][b] != dp[i - 1][b]:
            selected.append(options[i - 1])
            b -= options[i - 1]["cost"]
    return {"selected": selected, "total_cost": sum(o["cost"] for o in selected), "total_value": dp[n][budget]}


def update_plan(budget):
    options = build_candidate_options(critical_zones, battery_joined)
    result = optimize_plan(options, int(budget))

    plan_df = pd.DataFrame(result["selected"]).sort_values(["zone_id", "action"]).reset_index(drop=True)
    if plan_df.empty:
        plan_df = pd.DataFrame(columns=["zone_id", "action", "cost", "value"])
    plan_df["value"] = plan_df["value"].round(2)

    summary = (
        f"**Total cost:** ${result['total_cost']:,} / ${int(budget):,}  \n"
        f"**Risk-reduction value:** {result['total_value']:.2f}  \n\n"
        + DISCLAIMER_MD
    )
    return plan_df, summary, critical_zones[["zone_id", "grid_heat_risk"]].round(2), result


_last_plan_result = {"result": None, "budget": None}


def update_plan_and_cache(budget):
    plan_df, summary, crit_display, result = update_plan(budget)
    _last_plan_result["result"] = result
    _last_plan_result["budget"] = int(budget)
    return plan_df, summary, crit_display


def why_this_plan():
    result = _last_plan_result["result"]
    if result is None or not result["selected"]:
        return "Set a budget and let a plan compute first, or no actions are affordable/needed at this budget."
    plan_df = pd.DataFrame(result["selected"]).sort_values(["zone_id", "action"]).reset_index(drop=True)
    if _groq_client is None:
        return (
            f"Under a ${_last_plan_result['budget']:,} budget, the optimizer selected "
            f"{len(plan_df)} action(s) totaling ${result['total_cost']:,}, for a modeled "
            f"risk-reduction value of {result['total_value']:.2f}. Action costs and "
            f"risk-reduction percentages are modeling assumptions, not empirically "
            f"validated figures. [LLM explanation unavailable - no Groq client configured.]"
        )
    from __main__ import explain_plan  # defined in Section 9 above
    return explain_plan(_groq_client, plan_df, result["total_cost"], result["total_value"],
                         _last_plan_result["budget"])


# ============ Build the app ============

with gr.Blocks(title=f"GridHeat AI - {PILOT_COUNTY} County Pilot") as demo:
    gr.Markdown(f"# GridHeat AI \u2014 {PILOT_COUNTY} County Pilot (Texas)")
    gr.Markdown(f"{len(risk_table)} zones \u00b7 {len(critical_zones)} zones above risk threshold {RISK_THRESHOLD}")
    gr.Markdown(DISCLAIMER_MD)

    with gr.Tabs():
        with gr.Tab("Risk Overview"):
            gr.Dataframe(value=get_risk_table(), label="All zones, ranked by grid_heat_risk")

        with gr.Tab("Map View"):
            gr.HTML(value=build_map_html())

        with gr.Tab("Zone Evidence"):
            zone_dropdown = gr.Dropdown(
                choices=sorted(risk_table["zone_id"].tolist()), label="Select a zone"
            )
            evidence_card = gr.Markdown()
            why_zone_btn = gr.Button("Why?")
            zone_explanation = gr.Markdown()

            zone_dropdown.change(fn=show_zone_evidence, inputs=zone_dropdown,
                                  outputs=[evidence_card, zone_explanation])
            why_zone_btn.click(fn=why_this_zone, inputs=zone_dropdown, outputs=zone_explanation)

        with gr.Tab("Critical Zones + Recommended Plan"):
            budget_slider = gr.Slider(minimum=10_000, maximum=100_000, step=5_000, value=50_000, label="Budget ($)")
            with gr.Row():
                critical_display = gr.Dataframe(label="Critical zones", value=critical_zones[["zone_id", "grid_heat_risk"]].round(2))
                plan_display = gr.Dataframe(label="Recommended plan")
            plan_summary = gr.Markdown()
            why_plan_btn = gr.Button("Why this plan?")
            plan_explanation = gr.Markdown()

            budget_slider.change(fn=update_plan_and_cache, inputs=budget_slider,
                                  outputs=[plan_display, plan_summary, critical_display])
            demo.load(fn=lambda: update_plan_and_cache(50_000), outputs=[plan_display, plan_summary, critical_display])
            why_plan_btn.click(fn=why_this_plan, outputs=plan_explanation)

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ff3838dfb7eed40e24.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
